# Math-solver HARD-SUITE benchmark (SymPy / SmartSolver / StepSolver) - re-runnable, with a ranking

A second, harder benchmark on the same framework as the first notebook. It rebuilds isolated environments for the engine versions you choose, runs **89 harder test cases** (all references independently validated), judges them mathematically, scores derivations, probes engineering behaviour, and ends with a **ranking** that follows fixed rules, plus a head-to-head cross-check and a bootstrap that says whether the ranking is statistically trustworthy.

**What is harder**
- Tier 1 *hard* (weight 1) - e.g. quartic/cubic factoring, (x+y)^5, cyclic integration by parts, repeated L'Hospital, tan series, 3x3 systems, 4x4 determinants.
- Tier 2 *very hard* (weight 2) - extraneous-root traps (radicals, logs), nonlinear systems, nested-radical denesting, improper integrals, higher derivatives, complex eigenvalues, ODEs with repeated/complex characteristic roots.
- Tier 3 *adversarial* (weight 3) - the correct answer is "no solution / undefined / diverges / does not exist / not invertible / infinitely many". Fabricating an answer is a failure and is scored separately as **honesty**.

**Ranking to follow** (declared before any run): overall score = 35% difficulty-weighted correctness, 25% derivation quality, 15% coverage, 10% honesty on adversarial cases, 10% reliability, 3% performance, 2% integration. Engines within 2 points are a tie group, ordered by production eligibility, tier-3 correctness, tier-2 correctness, derivation score, latency. Hard gates (critical wrong answers, contradictory derivations, input executed as code, licence) are shown beside the rank and never averaged away. A bootstrap over cases (2000 resamples) reports P(rank) and whether any two engines are statistically separable (>= 90%).

**How to use**
1. Edit the configuration cell (engine versions / git refs). Defaults benchmark the *latest* code; the pinned values in its comment reproduce the original run. To test a fork, put its URL and commit in the StepSolver fields.
2. *Runtime -> Run all* (about 10-15 min; +5 min per engine if you enable their own test suites).
3. Read the leaderboard printed at the end; download the zip from the last cell.

**What is and is not automatic**
- Everything is computed by code from raw engine output; the reports contain no hand-written narrative.
- Derivation scores here are the **automated rubric only** (proxies: regex milestones, word counts, step-equivalence checks). The manual-review overlay mechanism exists (`data/manual_reviews.json`, fingerprinted to the exact output reviewed) but is empty for this suite. Human audit of derivations is recommended before relying on that 25%.
- Timings depend on the Colab machine; compare latencies only between runs on the same runtime type.
- If an engine's API changes (method renamed/removed) the adapter reports *unsupported* (SmartSolver) or an infrastructure error (import failure); it never substitutes another call. Edit the collapsed adapter cell to follow an API change.

In [1]:
#@title 1. Configuration { display-mode: "form" }
# ---- Engine versions ------------------------------------------------------------------------------------------
# Original (baseline) run:  SYMPY_SPEC="sympy==1.14.0"
#   SmartSolver REF f0d46a1089366f8e14f73de9ba811d8e4c1471b0   StepSolver REF bd8ea4ffdbe49277d649017cb6a31bf05b376bd8
SYMPY_SPEC = "sympy"                      #@param {type:"string"}   # "sympy" = latest release, or e.g. "sympy==1.14.0"
PIN_SYMPY_IN_CANDIDATES = True            #@param {type:"boolean"}  # give SmartSolver/StepSolver the same SymPy as the standalone engine (falls back to their own resolution if they conflict)
SMARTSOLVER_URL = "https://github.com/mngadilinda/SmartSolver"      #@param {type:"string"}
SMARTSOLVER_REF = "HEAD"                  #@param {type:"string"}   # branch, tag or commit; HEAD = latest default branch
STEPSOLVER_URL = "https://github.com/Jayfraggs/stepsolver.git"      #@param {type:"string"}
STEPSOLVER_REF = "main"                   #@param {type:"string"}

# ---- Judge (kept fixed so the same answers are judged the same way across engine versions) ---------------------
JUDGE_SYMPY_SPEC = "sympy==1.14.0"        #@param {type:"string"}

# ---- Protocol ---------------------------------------------------------------------------------------------------
REPEATS = 5                               #@param {type:"integer"}  # warm, cache-cleared repeats per answered case
TIMEOUT_S = 20                            #@param {type:"integer"}  # per-case timeout (worker is killed and restarted)
RUN_TWICE = False                          #@param {type:"boolean"}  # second full run -> reproducibility check
RUN_OWN_TEST_SUITES = False               #@param {type:"boolean"}  # also run each candidate's own pytest suite (slow; feeds the maintainability score)
COMPARE_WITH_BASELINE = True              #@param {type:"boolean"}
PYTHON_VERSION = "3.12"                   #@param {type:"string"}
BENCH_DIR = "/content/math_solver_hard_suite"   #@param {type:"string"}

In [2]:
#@title 2. Tooling (uv + git) { display-mode: "form" }
import base64, json, os, pathlib, platform, re, shutil, subprocess, sys, time

ROOT = pathlib.Path(BENCH_DIR)
ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)

def sh(cmd, check=True, timeout=None, cwd=None):
    r = subprocess.run(cmd, shell=isinstance(cmd, str), capture_output=True, text=True, timeout=timeout, cwd=cwd)
    if check and r.returncode:
        raise RuntimeError("command failed (%d): %s\n%s" % (r.returncode, cmd, (r.stdout + r.stderr)[-1500:]))
    return r

def run_stream(cmd, cwd=None, check=True):
    """Run a command and stream its output into the notebook."""
    p = subprocess.Popen(cmd, shell=isinstance(cmd, str), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=cwd)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    if check and p.returncode:
        raise RuntimeError("command failed (%d): %s" % (p.returncode, cmd))
    return p.returncode

def W(rel, text):
    p = ROOT / rel
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(text, encoding="utf-8")

if not shutil.which("uv"):
    sh([sys.executable, "-m", "pip", "install", "-q", "uv"])
UV = shutil.which("uv") or "uv"
print("uv:", sh([UV, "--version"]).stdout.strip(), "| git:", sh("git --version").stdout.strip(), "| notebook python:", platform.python_version())

uv: uv 0.13.5 (x86_64-unknown-linux-gnu) | git: git version 2.43.0 | notebook python: 3.13.18


## Benchmark source
The next four cells write the benchmark code to disk (collapsed by default; click *Show code* to read or edit them - e.g. to update an adapter after an API change). Nothing is executed here.

In [3]:
#@title 3a. Benchmark source: dataset generator + docs + data { display-mode: "form" }
W('tools/make_cases.py', r'''#!/usr/bin/env python3
"""Generate data/cases.json for the HARD suite (75 cases, ids M01..V03).

Tiers (difficulty weights used by the ranking):
  1 = hard         (weight 1)   multi-step algebra/calculus a strong student should handle
  2 = very hard    (weight 2)   traps, composite rules, improper/closed forms, nonlinear systems
  3 = adversarial  (weight 3)   the correct behaviour is to report 'no solution / undefined / diverges / does not exist / not invertible'
                                or an infinite solution family; fabricating an answer is a failure ('adversarial': true)
All comparison rules, tolerances, domains and form checks are declared HERE before any engine runs.  Nothing here imports an engine.
Expected values are validated independently by bench/reference_validation.py (Fraction / mpmath numerics, no SymPy solvers).
"""
import json
import pathlib
from fractions import Fraction as Fr

CATS = {
    "M": "Hard algebra: factoring, expansion, radicals, rational expressions",
    "N": "Equations with traps: absolute value, radicals, logs, exponentials, extraneous roots, complex roots",
    "O": "Nonlinear/3x3 systems and harder inequalities",
    "P": "Trigonometric exact values and identities",
    "Q": "Differentiation: composite, inverse-trig, higher order",
    "R": "Integration: by parts, inverse trig, improper, divergent",
    "S": "Limits (one-sided, DNE), series and sums",
    "T": "Linear algebra: 3x3, rank, eigenvalues, singular matrices",
    "U": "Ordinary differential equations",
    "V": "Complex-number arithmetic",
}
TOL = {"abs": 1e-9, "rel": 1e-9}
TIER2 = set("R12 R13 R14 R15 S13 S14 S16 T08 U06 U07 O09 M04 M05 M08 N04 N05 N07 N10 O01 O02 O03 O05 O07 P03 P05 Q01 Q03 Q05 R01 R02 R04 R06 R09 S03 S08 S09 S10 T02 T05 T07 U03 U04".split())
TIER3 = set("N02 N11 O04 O08 R10 R11 S06 S07 S11 T06".split())

MS = {
    "eval": [r"order of operations|evaluat|arithmetic|compute|simplif|multipl|add|subtract|power|root|absolute|exact value|identity"],
    "simplify": [r"combine|collect|like terms|distribut|cancel|simplif|factor|identity|rule|rationali|common denominator"],
    "expand": [r"expand|distribut|multiply|binomial|pascal"],
    "factor": [r"factor|common|difference of squares|substitut|grouping|rational root|product"],
    "solve": [r"isolate|inverse|both sides|substitut|factor|square|case|branch|exponent|logarithm|take the log|quadratic|zero.product|solve",
              r"verif|check|substitut|extraneous|reject|domain|valid|original"],
    "system": [r"substitut|eliminat|row|augmented|solve for|isolate",
               r"back.substitut|solution|verify|check"],
    "ineq": [r"critical|sign|test|interval|factor|case|abs|rewrite|boundary|solve|isolate"],
    "trig": [r"special angle|unit circle|exact value|identity|half.angle|sum|difference|pythag|evaluate|known value|quotient"],
    "diff": [r"rule|derivative|differentiat|chain|product|quotient|power|logarithmic|inner|outer"],
    "int": [r"rule|integral|antiderivative|substitut|by parts|standard|integrat|partial fraction|u\s*=|power.reduc|identity"],
    "improper": [r"improper|limit|antiderivative|converge|diverge|fundamental theorem|bound|discontinu|singular",
                 r"evaluate|F\(b\)|upper.*lower|limit as"],
    "limit": [r"l.?h.?pital|indeterminate|substitut|factor|cancel|conjugate|one.sided|left|right|squeeze|expon|series|dominant|standard limit|direct"],
    "series": [r"taylor|maclaurin|series|derivative|coefficient|expansion|term"],
    "sum": [r"series|converge|diverge|p.series|geometric|zeta|test|ratio|sum"],
    "matrix": [r"matrix|row|determinant|cofactor|adjugate|pivot|scale|elimination|formula|minor|expansion"],
    "eig": [r"characteristic|eigenvalue|determinant|lambda|polynomial|root"],
    "ode": [r"separat|characteristic|integrating factor|auxiliary|ansatz|integrate|homogeneous|particular|solve"],
    "complex": [r"conjugate|modulus|multiply|binomial|polar|de moivre|imaginary|i\^2|complex|evaluate|simplif"],
}

cases = []


def add(cid, op, display, spec_expected, problem, expected, compare, *, domain="real", dsens=False, restrictions=None, ms=None,
        tol=None, notes="", sample=None, adapter_inputs=None):
    tier = 3 if cid in TIER3 else 2 if cid in TIER2 else 1
    cases.append({
        "id": cid, "category": cid[0], "category_name": CATS[cid[0]], "operation": op, "display_input": display,
        "problem": problem, "domain": domain, "domain_sensitive": dsens, "required_restrictions": restrictions or [],
        "spec_expected": spec_expected, "expected": expected, "comparison": compare, "tolerance": tol or TOL,
        "reference_milestones": ms or [], "sample_points": sample, "notes": notes, "adapter_inputs": adapter_inputs or {},
        "tier": tier, "weight": tier, "adversarial": cid in TIER3,
    })


# ------------------------------------------------------------------------------------------------ M
add("M01", "factor", "Factor x^4 - 5*x^2 + 4", "(x - 2)(x - 1)(x + 1)(x + 2)", {"expr": "x^4 - 5*x^2 + 4"},
    {"expr": "(x - 2)*(x - 1)*(x + 1)*(x + 2)"}, {"method": "expr_equiv", "form": "factored_full"}, ms=MS["factor"])
add("M02", "factor", "Factor x^3 - 3*x^2 + 3*x - 1", "(x - 1)^3", {"expr": "x^3 - 3*x^2 + 3*x - 1"},
    {"expr": "(x - 1)^3"}, {"method": "expr_equiv", "form": "factored_full"}, ms=MS["factor"])
add("M03", "expand", "Expand (x + y)^5", "x^5 + 5x^4y + 10x^3y^2 + 10x^2y^3 + 5xy^4 + y^5", {"expr": "(x + y)^5"},
    {"expr": "x^5 + 5*x^4*y + 10*x^3*y^2 + 10*x^2*y^3 + 5*x*y^4 + y^5"}, {"method": "expr_equiv", "form": "expanded"}, ms=MS["expand"])
add("M04", "simplify", "Simplify (x^3 - 8)/(x^2 - 4)", "(x^2 + 2x + 4)/(x + 2), x != 2, x != -2", {"expr": "(x^3 - 8)/(x^2 - 4)"},
    {"expr": "(x^2 + 2*x + 4)/(x + 2)"}, {"method": "expr_equiv", "form": "reduced_fraction"}, dsens=True, restrictions=["x != 2", "x != -2"],
    ms=MS["simplify"] + [r"restrict|!=|≠|domain|undefined|cannot"])
add("M05", "simplify", "Simplify (1/x + 1/y)/(1/x - 1/y)", "(x + y)/(y - x), x != 0, y != 0", {"expr": "(1/x + 1/y)/(1/x - 1/y)"},
    {"expr": "(x + y)/(y - x)"}, {"method": "expr_equiv", "form": "reduced_fraction"}, dsens=True, restrictions=["x != 0", "y != 0"],
    ms=MS["simplify"] + [r"restrict|!=|≠|domain|undefined|cannot"])
add("M06", "simplify", "Simplify sqrt(50) + sqrt(18) - sqrt(8)", "6*sqrt(2)", {"expr": "sqrt(50) + sqrt(18) - sqrt(8)"},
    {"expr": "6*sqrt(2)"}, {"method": "expr_equiv", "form": None}, ms=MS["simplify"])
add("M07", "simplify", "Simplify (sqrt(3) + sqrt(2))*(sqrt(3) - sqrt(2))", "1", {"expr": "(sqrt(3) + sqrt(2))*(sqrt(3) - sqrt(2))"},
    {"expr": "1"}, {"method": "expr_equiv", "form": None}, ms=MS["simplify"])
add("M08", "simplify", "Simplify sqrt(7 + 4*sqrt(3))", "2 + sqrt(3)", {"expr": "sqrt(7 + 4*sqrt(3))"}, {"expr": "2 + sqrt(3)"},
    {"method": "expr_equiv", "form": None}, ms=MS["simplify"] + [r"denest|perfect square|\(a\s*\+\s*b\)"],
    notes="Nested-radical denesting; an unchanged sqrt(7 + 4*sqrt(3)) is value-equal but fails the simplification-form rule.")


# ------------------------------------------------------------------------------------------------ N
def N(cid, eq, values, spec, **kw):
    add(cid, "solve", eq, spec, {"equation": eq, "var": "x"},
        {"kind": "finite", "values": values} if values is not None else {"kind": "empty"}, {"method": "solution_set"},
        ms=kw.pop("ms", MS["solve"]), **kw)


N("N01", "abs(2*x - 3) = 5", ["4", "-1"], "x = 4, -1")
N("N02", "1/(x - 1) + 1/(x + 1) = 2/(x^2 - 1)", None, "No solution (x = 1 is extraneous)", dsens=True,
  notes="Clearing denominators gives x = 1, which makes the original undefined: the correct answer is the empty set.")
N("N03", "sqrt(2*x + 3) = x", ["3"], "x = 3 (x = -1 extraneous)", dsens=True)
N("N04", "sqrt(x + 5) + sqrt(x) = 5", ["4"], "x = 4", dsens=True)
N("N05", "log(x) + log(x - 3) = log(4)", ["4"], "x = 4 (x = -1 outside the domain)", dsens=True)
N("N06", "2^x = 7", ["log(7)/log(2)"], "x = log(7)/log(2)")
N("N07", "exp(2*x) - 5*exp(x) + 6 = 0", ["log(2)", "log(3)"], "x = ln 2, ln 3")
N("N08", "x^4 - 5*x^2 + 4 = 0", ["-2", "-1", "1", "2"], "x = -2, -1, 1, 2")
N("N09", "x^3 - 6*x^2 + 11*x - 6 = 0", ["1", "2", "3"], "x = 1, 2, 3")
N("N10", "x^4 + 4 = 0", ["1 + I", "1 - I", "-1 + I", "-1 - I"], "x = 1 +- i, -1 +- i (complex)", domain="complex", dsens=True,
  notes="Expected over C. A real-only answer (empty set) is recorded as real_domain_only, not correct.")
N("N11", "sqrt(x) = -2", None, "No real solution", dsens=True)


# ------------------------------------------------------------------------------------------------ O
def system(cid, eqs, vars_, exp, spec, linear, **kw):
    add(cid, "solve_system", "; ".join(eqs), spec, {"equations": eqs, "vars": vars_, "linear": linear, "method": None}, exp,
        {"method": "system_solution"}, ms=MS["system"], **kw)


system("O01", ["x^2 + y^2 = 25", "x + y = 7"], ["x", "y"], {"kind": "multi", "assignments": [{"x": "3", "y": "4"}, {"x": "4", "y": "3"}]},
       "(3, 4) and (4, 3)", False)
system("O02", ["x*y = 12", "x - y = 1"], ["x", "y"], {"kind": "multi", "assignments": [{"x": "4", "y": "3"}, {"x": "-3", "y": "-4"}]},
       "(4, 3) and (-3, -4)", False)
system("O03", ["x + y + z = 6", "2*y + 5*z = -4", "2*x + 5*y - z = 27"], ["x", "y", "z"],
       {"kind": "unique", "assignments": {"x": "5", "y": "3", "z": "-2"}}, "x = 5, y = 3, z = -2", True)
system("O04", ["x + y = 2", "2*x + 2*y = 4"], ["x", "y"], {"kind": "line", "note": "infinitely many solutions: one free parameter"},
       "Infinitely many solutions (x + y = 2)", True, dsens=True,
       notes="Correct = a parametric family satisfying both equations. A unique point or 'no solution' is wrong.")


def ineq(cid, s, intervals, spec, **kw):
    add(cid, "solve_inequality", s, spec, {"inequality": s, "var": "x"}, {"kind": "intervals", "intervals": intervals},
        {"method": "inequality_set"}, ms=MS["ineq"], **kw)


ineq("O05", "(x + 1)/(x - 2) >= 0", [["-oo", "-1", True, False], ["2", "oo", True, True]], "x <= -1 or x > 2", dsens=True)
ineq("O06", "abs(x - 1) < 3", [["-2", "4", True, True]], "-2 < x < 4")
ineq("O07", "x^3 - x > 0", [["-1", "0", True, True], ["1", "oo", True, True]], "-1 < x < 0 or x > 1")
ineq("O08", "x^2 + 1 < 0", [], "No real solution (empty set)", dsens=True)

# ------------------------------------------------------------------------------------------------ P
add("P01", "evaluate", "sin(pi/12)", "(sqrt(6) - sqrt(2))/4", {"expr": "sin(pi/12)"}, {"expr": "(sqrt(6) - sqrt(2))/4"}, {"method": "value", "form": "closed_form"}, ms=MS["trig"])
add("P02", "evaluate", "cos(7*pi/6)", "-sqrt(3)/2", {"expr": "cos(7*pi/6)"}, {"expr": "-sqrt(3)/2"}, {"method": "value", "form": "closed_form"}, ms=MS["trig"])
add("P03", "simplify", "Simplify (1 - cos(x)^2)/sin(x)^2", "1 (sin(x) != 0)", {"expr": "(1 - cos(x)^2)/sin(x)^2"}, {"expr": "1"},
    {"method": "expr_equiv", "form": None}, dsens=True, ms=MS["trig"])
add("P04", "simplify", "Simplify tan(x)*cos(x)", "sin(x)", {"expr": "tan(x)*cos(x)"}, {"expr": "sin(x)"}, {"method": "expr_equiv", "form": None}, ms=MS["trig"])
add("P05", "evaluate", "cos(pi/5)", "(1 + sqrt(5))/4", {"expr": "cos(pi/5)"}, {"expr": "(1 + sqrt(5))/4"}, {"method": "value", "form": "closed_form"}, ms=MS["trig"])


# ------------------------------------------------------------------------------------------------ Q
def Q(cid, expr, exp, spec, order=1, **kw):
    add(cid, "diff", ("Differentiate %s" % expr) + ("" if order == 1 else " (order %d)" % order), spec, {"expr": expr, "var": "x", "order": order},
        {"expr": exp}, {"method": "expr_equiv", "form": None}, ms=kw.pop("ms", MS["diff"]), **kw)


Q("Q01", "(x^2 + 1)^sin(x)", "(x^2 + 1)^sin(x)*(cos(x)*log(x^2 + 1) + 2*x*sin(x)/(x^2 + 1))", "(x^2+1)^sin(x) * (cos(x) ln(x^2+1) + 2x sin(x)/(x^2+1))",
  ms=MS["diff"] + [r"logarithmic|ln|log"])
Q("Q02", "atan(x^2)", "2*x/(x^4 + 1)", "2x/(1 + x^4)")
Q("Q03", "sqrt(1 + x^2)/x", "-1/(x^2*sqrt(1 + x^2))", "-1/(x^2 sqrt(1 + x^2)), x != 0", dsens=True)
Q("Q04", "x^4*exp(x)", "x^2*(x^2 + 8*x + 12)*exp(x)", "x^2 (x^2 + 8x + 12) e^x", order=2, ms=MS["diff"] + [r"second|twice|again|order 2|term 2"])
Q("Q05", "log(x + sqrt(x^2 + 1))", "1/sqrt(x^2 + 1)", "1/sqrt(x^2 + 1)")


# ------------------------------------------------------------------------------------------------ R
def Rint(cid, expr, exp, spec, real_on=None, sample=None, **kw):
    add(cid, "integrate", "Integrate " + expr, spec, {"expr": expr, "var": "x"}, {"expr": exp},
        {"method": "antiderivative", "require_real_on": real_on}, ms=kw.pop("ms", MS["int"]), sample=sample, **kw)


POS = {"x": ["1/2", "13/10", "27/10", "71/10"]}
Rint("R01", "x^2*exp(x)", "exp(x)*(x^2 - 2*x + 2)", "e^x (x^2 - 2x + 2) + C", ms=MS["int"] + [r"by parts|twice"])
Rint("R02", "log(x)", "x*log(x) - x", "x ln x - x + C (x > 0)", sample=POS, dsens=True, ms=MS["int"] + [r"by parts"])
Rint("R03", "1/(x^2 + 1)", "atan(x)", "arctan(x) + C")
Rint("R04", "1/sqrt(1 - x^2)", "asin(x)", "arcsin(x) + C (|x| < 1)", sample={"x": ["-3/4", "-1/3", "1/4", "4/5"]}, dsens=True)
Rint("R05", "sin(x)^2", "x/2 - sin(2*x)/4", "x/2 - sin(2x)/4 + C", ms=MS["int"] + [r"half.angle|power.reduc|double.angle|identity|cos\(2"])
Rint("R06", "1/(x^2 - 1)", "log(abs((x - 1)/(x + 1)))/2", "(1/2) ln|(x-1)/(x+1)| + C on each interval", real_on=["-3", "-1/2", "1/2", "3"], dsens=True,
     ms=MS["int"] + [r"partial fraction"], notes="Declared rule: F' = f AND F real-valued on all three intervals (needs |.| inside the log).")


def Rdef(cid, expr, lo, hi, exp, spec, method="value", **kw):
    add(cid, "integrate", "Integrate %s from %s to %s" % (expr, lo, hi), spec, {"expr": expr, "var": "x", "lower": lo, "upper": hi}, exp,
        {"method": method}, ms=kw.pop("ms", MS["improper"]), **kw)


Rdef("R07", "log(x)", "0", "1", {"expr": "-1"}, "-1 (improper at 0)", dsens=True)
Rdef("R08", "1/x^2", "1", "oo", {"expr": "1"}, "1")
Rdef("R09", "exp(-x^2)", "-oo", "oo", {"expr": "sqrt(pi)"}, "sqrt(pi)")
Rdef("R10", "1/x^2", "-1", "1", {"special": "divergent"}, "Diverges (naive FTC gives -2)", method="divergent", dsens=True,
     notes="Trap: F(1)-F(-1) = -2 is wrong; the integrand is positive and non-integrable at 0.")
Rdef("R11", "1/x", "-1", "1", {"special": "divergent"}, "Diverges (principal value 0 is not the improper integral)", method="divergent", dsens=True)


# ------------------------------------------------------------------------------------------------ S
def lim(cid, expr, pt, direction, exp, spec, method="value", **kw):
    add(cid, "limit", "Limit of %s as x -> %s%s" % (expr, pt, {"both": "", "+": " from the right", "-": " from the left"}[direction]), spec,
        {"expr": expr, "var": "x", "point": pt, "direction": direction}, exp, {"method": method}, ms=kw.pop("ms", MS["limit"]), **kw)


lim("S01", "(exp(x) - 1 - x)/x^2", "0", "both", {"expr": "1/2"}, "1/2")
lim("S02", "(1 + 1/x)^x", "oo", "both", {"expr": "exp(1)"}, "e")
lim("S03", "x^x", "0", "+", {"expr": "1"}, "1 (from the right)")
lim("S04", "1/x", "0", "+", {"expr": "oo"}, "+infinity (from the right)")
lim("S05", "abs(x)/x", "0", "-", {"expr": "-1"}, "-1 (from the left)")
lim("S06", "abs(x)/x", "0", "both", {"special": "dne"}, "Does not exist (left -1, right +1)", method="dne", dsens=True)
lim("S07", "sin(1/x)", "0", "both", {"special": "dne"}, "Does not exist (oscillates)", method="dne", dsens=True)
lim("S08", "sqrt(x^2 + x) - x", "oo", "both", {"expr": "1/2"}, "1/2")
add("S09", "series", "Maclaurin series of exp(x) up to x^4", "1 + x + x^2/2 + x^3/6 + x^4/24", {"expr": "exp(x)", "var": "x", "point": "0", "n": 5},
    {"expr": "1 + x + x^2/2 + x^3/6 + x^4/24"}, {"method": "series"}, ms=MS["series"])


def Sum(cid, expr, lo, hi, exp, spec, method="value", **kw):
    add(cid, "sum", "Sum of %s for k = %s to %s" % (expr, lo, hi), spec, {"expr": expr, "var": "k", "lower": lo, "upper": hi}, exp, {"method": method},
        ms=kw.pop("ms", MS["sum"]), **kw)


Sum("S10", "1/k^2", "1", "oo", {"expr": "pi^2/6"}, "pi^2/6")
Sum("S11", "1/k", "1", "oo", {"special": "divergent"}, "Diverges (harmonic series)", method="divergent")
Sum("S12", "(1/2)^k", "0", "oo", {"expr": "2"}, "2")

# ------------------------------------------------------------------------------------------------ T
M3 = [[2, 1, 3], [0, -1, 4], [1, 2, 5]]


def fr_matrix(rows):
    return [[Fr(v) for v in r] for r in rows]


def det3(m):
    return (m[0][0] * (m[1][1] * m[2][2] - m[1][2] * m[2][1]) - m[0][1] * (m[1][0] * m[2][2] - m[1][2] * m[2][0])
            + m[0][2] * (m[1][0] * m[2][1] - m[1][1] * m[2][0]))


def gj(rows, inverse=False):
    a = fr_matrix(rows)
    n = len(a)
    if inverse:
        a = [r + [Fr(int(i == j)) for j in range(n)] for i, r in enumerate(a)]
    r = 0
    for c in range(len(a[0])):
        p = next((i for i in range(r, n) if a[i][c] != 0), None)
        if p is None:
            continue
        a[r], a[p] = a[p], a[r]
        a[r] = [v / a[r][c] for v in a[r]]
        for i in range(n):
            if i != r and a[i][c] != 0:
                f = a[i][c]
                a[i] = [u - f * w for u, w in zip(a[i], a[r])]
        r += 1
    return [r_[n:] for r_ in a] if inverse else a


def fstr(f):
    return str(f.numerator) if f.denominator == 1 else "%d/%d" % (f.numerator, f.denominator)


assert det3(fr_matrix(M3)) == -19
add("T01", "det", "Determinant of [[2,1,3],[0,-1,4],[1,2,5]]", "-19", {"matrix": M3}, {"expr": "-19"}, {"method": "value"}, ms=MS["matrix"])
add("T02", "inverse", "Inverse of [[2,1,3],[0,-1,4],[1,2,5]]", "adj(A)/det(A)", {"matrix": M3}, {"matrix": [[fstr(v) for v in r] for r in gj(M3, True)]},
    {"method": "matrix"}, ms=MS["matrix"])
add("T03", "rank", "Rank of [[1,2,3],[2,4,6],[1,0,1]]", "2", {"matrix": [[1, 2, 3], [2, 4, 6], [1, 0, 1]]}, {"expr": "2"}, {"method": "value"}, ms=MS["matrix"])
add("T04", "eigenvalues", "Eigenvalues of [[2,1],[1,2]]", "3, 1", {"matrix": [[2, 1], [1, 2]]}, {"kind": "finite", "values": ["3", "1"]},
    {"method": "solution_set"}, ms=MS["eig"])
add("T05", "eigenvalues", "Eigenvalues of [[0,-1],[1,0]] (complex)", "i, -i", {"matrix": [[0, -1], [1, 0]]}, {"kind": "finite", "values": ["I", "-I"]},
    {"method": "solution_set"}, domain="complex", dsens=True, ms=MS["eig"])
add("T06", "inverse", "Inverse of the singular matrix [[1,2],[2,4]]", "Does not exist (det = 0)", {"matrix": [[1, 2], [2, 4]]}, {"special": "error"},
    {"method": "expect_error"}, dsens=True, ms=[r"determinant|singular|not invertible|zero"])
add("T07", "rref", "RREF of [[1,2,1,4],[2,4,0,2],[1,2,2,7]]", "reduced row echelon form", {"matrix": [[1, 2, 1, 4], [2, 4, 0, 2], [1, 2, 2, 7]]},
    {"matrix": [[fstr(v) for v in r] for r in gj([[1, 2, 1, 4], [2, 4, 0, 2], [1, 2, 2, 7]])]}, {"method": "matrix"}, ms=MS["matrix"])


# ------------------------------------------------------------------------------------------------ U
def ode(cid, disp, expr, order, fam, n, spec, ss, **kw):
    add(cid, "dsolve", disp, spec, {"ode": expr, "func": "y", "var": "x", "order": order}, {"family": fam, "n_constants": n},
        {"method": "ode_family"}, ms=MS["ode"], adapter_inputs={"stepsolver": ss}, **kw)


ode("U01", "Solve y' + y = x", "Derivative(y(x), x) + y(x) - x", 1, "x - 1 + C*exp(-x)", 1, "y = x - 1 + C e^(-x)", "dsolve(diff(y(x), x) + y(x) = x, y(x))")
ode("U02", "Solve y'' - 3y' + 2y = 0", "Derivative(y(x), (x, 2)) - 3*Derivative(y(x), x) + 2*y(x)", 2, "C1*exp(x) + C2*exp(2*x)", 2,
    "y = C1 e^x + C2 e^(2x)", "dsolve(diff(y(x), x, 2) - 3*diff(y(x), x) + 2*y(x) = 0, y(x))")
ode("U03", "Solve y'' - 2y' + y = 0", "Derivative(y(x), (x, 2)) - 2*Derivative(y(x), x) + y(x)", 2, "(C1 + C2*x)*exp(x)", 2,
    "y = (C1 + C2 x) e^x", "dsolve(diff(y(x), x, 2) - 2*diff(y(x), x) + y(x) = 0, y(x))")
ode("U04", "Solve y'' + 2y' + 5y = 0", "Derivative(y(x), (x, 2)) + 2*Derivative(y(x), x) + 5*y(x)", 2, "exp(-x)*(C1*cos(2*x) + C2*sin(2*x))", 2,
    "y = e^(-x) (C1 cos 2x + C2 sin 2x)", "dsolve(diff(y(x), x, 2) + 2*diff(y(x), x) + 5*y(x) = 0, y(x))")
ode("U05", "Solve y' = x*y", "Derivative(y(x), x) - x*y(x)", 1, "C*exp(x^2/2)", 1, "y = C e^(x^2/2)", "dsolve(diff(y(x), x) = x*y(x), y(x))")

# ------------------------------------------------------------------------------------------------ V
add("V01", "evaluate", "(1 + I)^8", "16", {"expr": "(1 + I)^8"}, {"expr": "16"}, {"method": "value", "form": "closed_form"}, domain="complex", ms=MS["complex"])
add("V02", "evaluate", "abs(3 + 4*I)", "5", {"expr": "abs(3 + 4*I)"}, {"expr": "5"}, {"method": "value", "form": "closed_form"}, domain="complex", ms=MS["complex"])
add("V03", "evaluate", "(2 + 3*I)/(1 - I)", "(-1 + 5i)/2", {"expr": "(2 + 3*I)/(1 - I)"}, {"expr": "(-1 + 5*I)/2"}, {"method": "value", "form": "closed_form"}, domain="complex",
    ms=MS["complex"])


# ------------------------------------------------------------------------------------------------ extra stress cases
INT1 = {"x": ["13/10", "2", "27/10", "7"]}
Rint("R12", "exp(x)*sin(x)", "exp(x)*(sin(x) - cos(x))/2", "e^x (sin x - cos x)/2 + C", ms=MS["int"] + [r"by parts|twice|cyclic|solve for the integral"])
Rint("R13", "1/(x*log(x))", "log(log(x))", "ln|ln x| + C (x > 1)", sample=INT1, dsens=True, ms=MS["int"] + [r"substitut|u\s*=|ln"])
Rint("R14", "x/(x^2 + x + 1)", "log(x^2 + x + 1)/2 - atan((2*x + 1)/sqrt(3))/sqrt(3)", "(1/2) ln(x^2+x+1) - (1/sqrt 3) arctan((2x+1)/sqrt 3) + C",
     ms=MS["int"] + [r"complete the square|split|substitut|derivative of the denominator"])
Rint("R15", "sqrt(1 - x^2)", "(x*sqrt(1 - x^2) + asin(x))/2", "(x sqrt(1-x^2) + arcsin x)/2 + C", sample={"x": ["-3/4", "-1/3", "1/4", "4/5"]}, dsens=True,
     ms=MS["int"] + [r"trig|sub|sin|by parts"])
lim("S13", "(tan(x) - x)/x^3", "0", "both", {"expr": "1/3"}, "1/3")
lim("S14", "cos(x)^(1/x^2)", "0", "both", {"expr": "exp(-1/2)"}, "e^(-1/2)", ms=MS["limit"] + [r"logarithm|ln|log"])
lim("S15", "x^(1/x)", "oo", "both", {"expr": "1"}, "1", ms=MS["limit"] + [r"logarithm|ln|log"])
add("S16", "series", "Maclaurin series of tan(x) up to x^5", "x + x^3/3 + 2x^5/15", {"expr": "tan(x)", "var": "x", "point": "0", "n": 6},
    {"expr": "x + x^3/3 + 2*x^5/15"}, {"method": "series"}, ms=MS["series"])
add("S17", "series", "Maclaurin series of log(1 + x) up to x^4", "x - x^2/2 + x^3/3 - x^4/4", {"expr": "log(1 + x)", "var": "x", "point": "0", "n": 5},
    {"expr": "x - x^2/2 + x^3/3 - x^4/4"}, {"method": "series"}, ms=MS["series"])
add("T08", "eigenvalues", "Eigenvalues of [[2,0,0],[0,3,4],[0,4,9]]", "2, 1, 11", {"matrix": [[2, 0, 0], [0, 3, 4], [0, 4, 9]]},
    {"kind": "finite", "values": ["2", "1", "11"]}, {"method": "solution_set"}, ms=MS["eig"])
M4 = [[1, 2, 0, 1], [0, 1, 3, 2], [2, 0, 1, 1], [1, 1, 1, 0]]


def det_elim(m):
    m = fr_matrix(m)
    n, d = len(m), Fr(1)
    for c in range(n):
        p = next((i for i in range(c, n) if m[i][c] != 0), None)
        if p is None:
            return Fr(0)
        if p != c:
            m[c], m[p] = m[p], m[c]
            d = -d
        d *= m[c][c]
        for i in range(c + 1, n):
            f_ = m[i][c] / m[c][c]
            m[i] = [u - f_ * w for u, w in zip(m[i], m[c])]
    return d


add("T09", "det", "Determinant of the 4x4 matrix [[1,2,0,1],[0,1,3,2],[2,0,1,1],[1,1,1,0]]", fstr(det_elim(M4)), {"matrix": M4}, {"expr": fstr(det_elim(M4))},
    {"method": "value"}, ms=MS["matrix"])
ode("U06", "Solve y'' + y = x", "Derivative(y(x), (x, 2)) + y(x) - x", 2, "x + C1*cos(x) + C2*sin(x)", 2, "y = x + C1 cos x + C2 sin x",
    "dsolve(diff(y(x), x, 2) + y(x) = x, y(x))")
ode("U07", "Solve y' = y^2", "Derivative(y(x), x) - y(x)^2", 1, "-1/(x + C)", 1, "y = -1/(x + C)", "dsolve(diff(y(x), x) = y(x)^2, y(x))")
ineq("O09", "(x - 1)*(x + 2)^2*(x - 3) > 0", [["-oo", "-2", True, True], ["-2", "1", True, True], ["3", "oo", True, True]], "x < 1 (x != -2) or x > 3", dsens=True)

assert len({c["id"] for c in cases}) == len(cases)
out = pathlib.Path(__file__).resolve().parent.parent / "data" / "cases.json"
out.write_text(json.dumps({"schema_version": 2, "suite": "hard", "cold_start_case": "N01", "count": len(cases), "categories": CATS, "cases": cases}, indent=1, ensure_ascii=False) + "\n")
tiers = {t: sum(1 for c in cases if c["tier"] == t) for t in (1, 2, 3)}
print("wrote", out, "cases:", len(cases), "tiers:", tiers, "weight total:", sum(c["weight"] for c in cases))
''')

W('docs/METHODOLOGY.md', r'''# Methodology - HARD suite

## Isolation and protocol
Same as the first benchmark: one virtualenv and one worker process per engine (`envs/sympy|smartsolver|stepsolver`), launched from an empty temp directory; a separate pinned `judge` environment runs the judge, reference validation and scoring. Per-case timeout (default 20 s, worker killed and restarted), cold-start trials, warm cache-cleared repeats, raw replies preserved verbatim in `results/raw`.

## Dataset (tools/make_cases.py)
89 cases, ids M01..V03 (categories M-V), each with: tier and weight, adversarial flag, domain (real|complex), declared comparison method, tolerance, required restrictions, form rule. Expected answers are validated independently by `bench/reference_validation.py` with Fraction/cmath/mpmath arithmetic (numerical differentiation, quadrature, root finding, exhaustive rational scans, truth tables, ODE residuals, Wronskians, divergence evidence) - no SymPy symbolic solvers.

## Adapters (what each engine is asked)
SymPy (bare API): simplify, expand, factor, solveset over the declared domain, solve for systems, solve_univariate_inequality, diff(order), limit(dir), integrate (incl. improper), series, summation, Matrix.rank/eigenvals/det/inv/rref, dsolve. SmartSolver: one documented public method per operation; operations with no method are unsupported; nonlinear systems are unsupported (the method is documented for linear systems); two-sided limits are composed from its two native one-sided calls (flagged). StepSolver: its documented ASCII operations; ODE syntax overrides; the imaginary unit I is written i; one-sided limits use its `left`/`right` argument.

## Judge (bench/judge.py)
Symbolic/numeric equivalence (never string equality), exact set logic for solution sets, interval-set equality for inequalities, parametric-family verification for dependent systems, multi-solution set matching for nonlinear systems, ODE residual + Wronskian checks, antiderivative F'=f plus real-valuedness on every interval where required, series comparison after removing the trailing O(...) marker (documented normalisation), negative-result handling: divergence (infinite/nan/undefined accepted, a finite value rejected), does-not-exist (AccumBounds/nan/undefined/error accepted, an infinite or finite value rejected), expected-error (singular matrix). A correct negative result stated in an engine's "unsolved" message or an error message is accepted and flagged for review.

## Rules added while building the suite (disclosed; applied to every engine)
(1) `closed_form` form rule for evaluation cases (no unevaluated functions or powers/quotients of sums) - added after an equivalent-but-different radical form was wrongly rejected by the generic operation-count rule and an unevaluated `sin(pi/12)` would have passed it. (2) SmartSolver's own statement "infinitely many solutions" for a dependent system is reported as such (an earlier adapter version mapped its empty solution dict to "no solution"). (3) Real-domain wording match broadened ("keep the real solutions") so an engine that states its real-domain restriction is labelled real_domain_only rather than a false "no solution". (4) StepSolver's boolean inequality results (`False`/`True`) are interpreted as empty/all reals. (5) 14 stress cases (R12-R15, S13-S17, T08-T09, U06-U07, O09) were added after a first pass showed the original 75 were solved too easily; no case was removed or changed after seeing engine output.

## Limitations
89 cases are one sample; derivation scores are automated proxies only here; timings are hardware-dependent; one engine's failure on a case may reflect its documented scope (reported as unsupported/unsolved, not as wrong); adapters follow the public API names that existed when written; offline operation was assessed only statically.
''')

W('docs/SCORING_RULES.md', r'''# Scoring and ranking rules - HARD suite (declared before any engine ran)

## Difficulty tiers
Tier 1 hard (weight 1), tier 2 very hard (weight 2), tier 3 adversarial (weight 3). Tier 3 cases are those whose correct behaviour is a negative or non-unique result (no solution, undefined, diverges, does not exist, not invertible, infinitely many solutions); fabricating a result is a failure.

## Overall score
`Overall = 0.35*Correctness + 0.25*Derivations + 0.15*Coverage + 0.10*Honesty + 0.10*Reliability + 0.03*Performance + 0.02*Integration`

| Component | Rule |
|---|---|
| Correctness | difficulty-weighted share of strictly correct cases. Strict = correct value AND required form AND (if required) the domain restriction reported. Unsupported/errors/invalid all count as not correct and are reported separately. |
| Derivations | mean final rubric score /10 over derivations that exist and whose answer was produced, x10 (as in the first benchmark). Availability is reported; an availability-penalised variant (unavailable = 0 over all cases) is in the sensitivity table. Automated rubric only in this suite (see below). |
| Coverage | cases where the engine produced a result in the expected form / all cases (unweighted). |
| Honesty | 100 x (1 - confident-wrong adversarial answers / adversarial cases). Only `invalid_answer` on a tier-3 case is dishonest; declaring "unsupported/unsolved", signalling the negative result in an error, or having no API is not fabrication (coverage and correctness already penalise those). |
| Reliability | mean of: repeatability (identical outputs across cache-cleared repeats), engineering-probe mean, 1 - crashes/attempted, strict-correct rate on domain-sensitive cases, untrusted-input safety as shipped. |
| Performance | latency score 100 at <=10 ms, 0 at >=2000 ms, log-linear; 0.4 median + 0.4 p95 + 0.2 x (1 - failure rate); on the cases ALL engines answered. |
| Integration | 14-point evidence-computed rubric (dependencies, install, adapter overrides, test count, licence, maintenance, 2 fixed assumptions) x 100/14. |

## Ranking (bench/rank.py)
1. Primary: overall score. Engines within 2.0 points of the neighbour above form a tie group, ordered by (a) production eligibility, (b) tier-3 correctness, (c) tier-2 correctness, (d) mean derivation score, (e) lower warm latency.
2. Cross-check: head-to-head. For each pair, each case where exactly one engine is strictly correct is a win for it, weighted by tier; larger weighted margin wins the pair; Copeland score = pairs won. If Copeland disagrees with the primary order, the order is reported as unsettled (the components pull in different directions).
3. Stability: bootstrap over cases (2000 resamples, fixed seed). Correctness, coverage, honesty and derivation mean are recomputed per resample; reliability, performance and integration are held fixed. Two engines are "statistically separable" only if P(A outranks B) >= 0.90.

## Hard gates (never averaged away)
G1 critical correctness: a confidently wrong answer (false "no solution", wrong value/set, fabricated result). FAIL if >= 1. Real-domain-only answers to complex-domain cases are major defects, not critical, when the engine's steps state the real-domain restriction.
G2 invalid or self-contradicting derivation: FAIL if >= 2 cases (automated invalid-step detection, `direct_evaluation` override, untied workings), CONDITIONAL if 1.
G3 benign canary `__import__('math').factorial(4)` evaluated by the engine's own string interface: FAIL, except CONDITIONAL for SymPy (programmatic API + allow-list adapter demonstrated).
G4 licence: PASS only if a permissive licence is detected; otherwise CONDITIONAL (legal review).

## Derivation rubric
Same 10-point rubric and automated implementation as the first benchmark (`bench/derivations.py`): correctness 0-3, completeness 0-2, explanation 0-2, consistency 0-1, domain 0-1, presentation 0-1. No manual overlay is applied in this suite.
''')

W('data/manual_reviews.json', r'''{}
''')

W('data/baseline.json', r'''{"label": "original hard-suite run: SymPy 1.14.0, SmartSolver 0.4.1 @ f0d46a1, StepSolver 0.1.0 @ bd8ea4f (2026-10-26, 1-core sandbox)", "engines": {"sympy-direct": {"version": "1.14.0", "commit": null}, "smartsolver": {"version": "0.4.1", "commit": "f0d46a1089366f8e14f73de9ba811d8e4c1471b0"}, "stepsolver": {"version": "0.1.0", "commit": "bd8ea4ffdbe49277d649017cb6a31bf05b376bd8"}}, "overall": {"sympy-direct": 78.20602825514183, "smartsolver": 60.6944518965739, "stepsolver": 81.70064011866363}, "components": {"sympy-direct": {"correctness": 93.37748344370861, "derivations": 40, "coverage": 97.75280898876404, "honesty": 100.0, "reliability": 66.46330680813439, "integration": 85.71428571428571, "performance": 83.34571021433536}, "smartsolver": {"correctness": 51.65562913907285, "derivations": 59.55555555555556, "coverage": 50.561797752808985, "honesty": 100.0, "reliability": 63.07692307692307, "integration": 85.71428571428571, "performance": 70.66150413700491}, "stepsolver": {"correctness": 88.74172185430463, "derivations": 59.24050632911392, "coverage": 91.01123595505618, "honesty": 100.0, "reliability": 89.38461538461539, "integration": 78.57142857142857, "performance": 55.64451280766645}}, "cases": {"sympy-direct|M01": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|M02": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|M03": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|M04": {"status": "invalid_answer", "strict_correct": false, "derivation_final": null}, "sympy-direct|M05": {"status": "valid", "strict_correct": false, "derivation_final": null}, "sympy-direct|M06": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|M07": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|M08": {"status": "invalid_answer", "strict_correct": false, "derivation_final": null}, "sympy-direct|N01": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|N02": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|N03": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|N04": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|N05": {"status": "invalid_answer", "strict_correct": false, "derivation_final": null}, "sympy-direct|N06": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|N07": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|N08": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|N09": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|N10": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|N11": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|O01": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|O02": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|O03": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|O04": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|O05": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|O06": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|O07": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|O08": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|P01": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|P02": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|P03": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|P04": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|P05": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|Q01": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|Q02": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|Q03": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|Q04": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|Q05": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|R01": {"status": "valid", "strict_correct": true, "derivation_final": 5}, "sympy-direct|R02": {"status": "valid", "strict_correct": true, "derivation_final": 4}, "sympy-direct|R03": {"status": "valid", "strict_correct": true, "derivation_final": 5}, "sympy-direct|R04": {"status": "valid", "strict_correct": true, "derivation_final": 4}, "sympy-direct|R05": {"status": "valid", "strict_correct": true, "derivation_final": 5}, "sympy-direct|R06": {"status": "invalid_answer", "strict_correct": false, "derivation_final": 2}, "sympy-direct|R07": {"status": "valid", "strict_correct": true, "derivation_final": 3}, "sympy-direct|R08": {"status": "valid", "strict_correct": true, "derivation_final": 4}, "sympy-direct|R09": {"status": "valid", "strict_correct": true, "derivation_final": 4}, "sympy-direct|R10": {"status": "valid", "strict_correct": true, "derivation_final": 3}, "sympy-direct|R11": {"status": "valid", "strict_correct": true, "derivation_final": 3}, "sympy-direct|S01": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|S02": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|S03": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|S04": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|S05": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|S06": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|S07": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|S08": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|S09": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|S10": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|S11": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|S12": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|T01": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|T02": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|T03": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|T04": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|T05": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|T06": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|T07": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|U01": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|U02": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|U03": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|U04": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|U05": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|V01": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|V02": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|V03": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|R12": {"status": "valid", "strict_correct": true, "derivation_final": 5}, "sympy-direct|R13": {"status": "valid", "strict_correct": true, "derivation_final": 4}, "sympy-direct|R14": {"status": "valid", "strict_correct": true, "derivation_final": 5}, "sympy-direct|R15": {"status": "valid", "strict_correct": true, "derivation_final": 4}, "sympy-direct|S13": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|S14": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|S15": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|S16": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|S17": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|T08": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|T09": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|U06": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|U07": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|O09": {"status": "valid", "strict_correct": true, "derivation_final": null}, "smartsolver|M01": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|M02": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|M03": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|M04": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|M05": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|M06": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|M07": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|M08": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|N01": {"status": "invalid_answer", "strict_correct": false, "derivation_final": 1}, "smartsolver|N02": {"status": "valid", "strict_correct": true, "derivation_final": 2}, "smartsolver|N03": {"status": "valid", "strict_correct": true, "derivation_final": 2}, "smartsolver|N04": {"status": "valid", "strict_correct": true, "derivation_final": 2}, "smartsolver|N05": {"status": "valid", "strict_correct": true, "derivation_final": 8}, "smartsolver|N06": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "smartsolver|N07": {"status": "valid", "strict_correct": true, "derivation_final": 8}, "smartsolver|N08": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "smartsolver|N09": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "smartsolver|N10": {"status": "valid", "strict_correct": true, "derivation_final": 8}, "smartsolver|N11": {"status": "valid", "strict_correct": true, "derivation_final": 2}, "smartsolver|O01": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|O02": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|O03": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "smartsolver|O04": {"status": "valid", "strict_correct": true, "derivation_final": 7}, "smartsolver|O05": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|O06": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|O07": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|O08": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|P01": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|P02": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|P03": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|P04": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|P05": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|Q01": {"status": "valid", "strict_correct": true, "derivation_final": 8}, "smartsolver|Q02": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "smartsolver|Q03": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "smartsolver|Q04": {"status": "valid", "strict_correct": true, "derivation_final": 8}, "smartsolver|Q05": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "smartsolver|R01": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "smartsolver|R02": {"status": "valid", "strict_correct": true, "derivation_final": 7}, "smartsolver|R03": {"status": "valid", "strict_correct": true, "derivation_final": 8}, "smartsolver|R04": {"status": "valid", "strict_correct": true, "derivation_final": 2}, "smartsolver|R05": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "smartsolver|R06": {"status": "invalid_answer", "strict_correct": false, "derivation_final": 4}, "smartsolver|R07": {"status": "valid", "strict_correct": true, "derivation_final": 6}, "smartsolver|R08": {"status": "valid", "strict_correct": true, "derivation_final": 8}, "smartsolver|R09": {"status": "valid", "strict_correct": true, "derivation_final": 6}, "smartsolver|R10": {"status": "valid", "strict_correct": true, "derivation_final": 6}, "smartsolver|R11": {"status": "valid", "strict_correct": true, "derivation_final": 7}, "smartsolver|S01": {"status": "valid", "strict_correct": true, "derivation_final": 3}, "smartsolver|S02": {"status": "valid", "strict_correct": true, "derivation_final": 3}, "smartsolver|S03": {"status": "valid", "strict_correct": true, "derivation_final": 8}, "smartsolver|S04": {"status": "valid", "strict_correct": true, "derivation_final": 3}, "smartsolver|S05": {"status": "valid", "strict_correct": true, "derivation_final": 3}, "smartsolver|S06": {"status": "valid", "strict_correct": true, "derivation_final": 2}, "smartsolver|S07": {"status": "valid", "strict_correct": true, "derivation_final": 2}, "smartsolver|S08": {"status": "valid", "strict_correct": true, "derivation_final": 3}, "smartsolver|S09": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|S10": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|S11": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|S12": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|T01": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|T02": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|T03": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|T04": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|T05": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|T06": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|T07": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "smartsolver|U01": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|U02": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|U03": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|U04": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|U05": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|V01": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|V02": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|V03": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|R12": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "smartsolver|R13": {"status": "valid", "strict_correct": true, "derivation_final": 8}, "smartsolver|R14": {"status": "valid", "strict_correct": true, "derivation_final": 8}, "smartsolver|R15": {"status": "valid", "strict_correct": true, "derivation_final": 2}, "smartsolver|S13": {"status": "valid", "strict_correct": true, "derivation_final": 3}, "smartsolver|S14": {"status": "valid", "strict_correct": true, "derivation_final": 3}, "smartsolver|S15": {"status": "valid", "strict_correct": true, "derivation_final": 8}, "smartsolver|S16": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|S17": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|T08": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|T09": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|U06": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|U07": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|O09": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "stepsolver|M01": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "stepsolver|M02": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "stepsolver|M03": {"status": "valid", "strict_correct": true, "derivation_final": 10}, "stepsolver|M04": {"status": "invalid_answer", "strict_correct": false, "derivation_final": 4}, "stepsolver|M05": {"status": "valid", "strict_correct": true, "derivation_final": 10}, "stepsolver|M06": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "stepsolver|M07": {"status": "valid", "strict_correct": true, "derivation_final": 10}, "stepsolver|M08": {"status": "invalid_answer", "strict_correct": false, "derivation_final": 4}, "stepsolver|N01": {"status": "valid", "strict_correct": true, "derivation_final": 3}, "stepsolver|N02": {"status": "valid", "strict_correct": true, "derivation_final": 10}, "stepsolver|N03": {"status": "valid", "strict_correct": true, "derivation_final": 2}, "stepsolver|N04": {"status": "valid", "strict_correct": true, "derivation_final": 2}, "stepsolver|N05": {"status": "valid", "strict_correct": true, "derivation_final": 2}, "stepsolver|N06": {"status": "valid", "strict_correct": true, "derivation_final": 3}, "stepsolver|N07": {"status": "valid", "strict_correct": true, "derivation_final": 3}, "stepsolver|N08": {"status": "valid", "strict_correct": true, "derivation_final": 10}, "stepsolver|N09": {"status": "valid", "strict_correct": true, "derivation_final": 10}, "stepsolver|N10": {"status": "invalid_answer", "strict_correct": false, "derivation_final": 5}, "stepsolver|N11": {"status": "valid", "strict_correct": true, "derivation_final": 2}, "stepsolver|O01": {"status": "valid", "strict_correct": true, "derivation_final": 3}, "stepsolver|O02": {"status": "valid", "strict_correct": true, "derivation_final": 3}, "stepsolver|O03": {"status": "valid", "strict_correct": true, "derivation_final": 8}, "stepsolver|O04": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "stepsolver|O05": {"status": "valid", "strict_correct": true, "derivation_final": 2}, "stepsolver|O06": {"status": "valid", "strict_correct": true, "derivation_final": 3}, "stepsolver|O07": {"status": "valid", "strict_correct": true, "derivation_final": 3}, "stepsolver|O08": {"status": "valid", "strict_correct": true, "derivation_final": 2}, "stepsolver|P01": {"status": "valid", "strict_correct": true, "derivation_final": 7}, "stepsolver|P02": {"status": "valid", "strict_correct": true, "derivation_final": 7}, "stepsolver|P03": {"status": "valid", "strict_correct": true, "derivation_final": 6}, "stepsolver|P04": {"status": "valid", "strict_correct": true, "derivation_final": 7}, "stepsolver|P05": {"status": "valid", "strict_correct": true, "derivation_final": 7}, "stepsolver|Q01": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "stepsolver|Q02": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "stepsolver|Q03": {"status": "valid", "strict_correct": true, "derivation_final": 10}, "stepsolver|Q04": {"status": "valid", "strict_correct": true, "derivation_final": 10}, "stepsolver|Q05": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "stepsolver|R01": {"status": "valid", "strict_correct": true, "derivation_final": 10}, "stepsolver|R02": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "stepsolver|R03": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "stepsolver|R04": {"status": "valid", "strict_correct": true, "derivation_final": 8}, "stepsolver|R05": {"status": "valid", "strict_correct": true, "derivation_final": 10}, "stepsolver|R06": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "stepsolver|R07": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "stepsolver|R08": {"status": "valid", "strict_correct": true, "derivation_final": 10}, "stepsolver|R09": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "stepsolver|R10": {"status": "valid", "strict_correct": true, "derivation_final": null}, "stepsolver|R11": {"status": "valid", "strict_correct": true, "derivation_final": null}, "stepsolver|S01": {"status": "valid", "strict_correct": true, "derivation_final": 10}, "stepsolver|S02": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "stepsolver|S03": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "stepsolver|S04": {"status": "valid", "strict_correct": true, "derivation_final": 8}, "stepsolver|S05": {"status": "valid", "strict_correct": true, "derivation_final": 3}, "stepsolver|S06": {"status": "valid", "strict_correct": true, "derivation_final": null}, "stepsolver|S07": {"status": "valid", "strict_correct": true, "derivation_final": 2}, "stepsolver|S08": {"status": "valid", "strict_correct": true, "derivation_final": 3}, "stepsolver|S09": {"status": "valid", "strict_correct": true, "derivation_final": 3}, "stepsolver|S10": {"status": "valid", "strict_correct": true, "derivation_final": 10}, "stepsolver|S11": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "stepsolver|S12": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "stepsolver|T01": {"status": "valid", "strict_correct": true, "derivation_final": 3}, "stepsolver|T02": {"status": "valid", "strict_correct": true, "derivation_final": 3}, "stepsolver|T03": {"status": "valid", "strict_correct": true, "derivation_final": 3}, "stepsolver|T04": {"status": "valid", "strict_correct": true, "derivation_final": 3}, "stepsolver|T05": {"status": "valid", "strict_correct": true, "derivation_final": 2}, "stepsolver|T06": {"status": "valid", "strict_correct": true, "derivation_final": null}, "stepsolver|T07": {"status": "valid", "strict_correct": true, "derivation_final": 3}, "stepsolver|U01": {"status": "valid", "strict_correct": true, "derivation_final": 3}, "stepsolver|U02": {"status": "valid", "strict_correct": true, "derivation_final": 3}, "stepsolver|U03": {"status": "valid", "strict_correct": true, "derivation_final": 3}, "stepsolver|U04": {"status": "valid", "strict_correct": true, "derivation_final": 3}, "stepsolver|U05": {"status": "valid", "strict_correct": true, "derivation_final": 3}, "stepsolver|V01": {"status": "valid", "strict_correct": true, "derivation_final": 10}, "stepsolver|V02": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "stepsolver|V03": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "stepsolver|R12": {"status": "valid", "strict_correct": true, "derivation_final": 10}, "stepsolver|R13": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "stepsolver|R14": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "stepsolver|R15": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "stepsolver|S13": {"status": "valid", "strict_correct": true, "derivation_final": 10}, "stepsolver|S14": {"status": "valid", "strict_correct": true, "derivation_final": 3}, "stepsolver|S15": {"status": "valid", "strict_correct": true, "derivation_final": 3}, "stepsolver|S16": {"status": "valid", "strict_correct": true, "derivation_final": 3}, "stepsolver|S17": {"status": "valid", "strict_correct": true, "derivation_final": 3}, "stepsolver|T08": {"status": "valid", "strict_correct": true, "derivation_final": 3}, "stepsolver|T09": {"status": "valid", "strict_correct": true, "derivation_final": 3}, "stepsolver|U06": {"status": "valid", "strict_correct": true, "derivation_final": 3}, "stepsolver|U07": {"status": "valid", "strict_correct": true, "derivation_final": 3}, "stepsolver|O09": {"status": "valid", "strict_correct": true, "derivation_final": 2}}}''')
print('wrote', 5, 'files')

wrote 5 files


In [4]:
#@title 3b. Benchmark source: worker protocol + adapters (SymPy, SmartSolver, StepSolver) { display-mode: "form" }
W('bench/__init__.py', r'''''')

W('bench/workers/__init__.py', r'''''')

W('bench/workers/common.py', r'''"""Shared worker protocol (stdlib only). One JSON request per line on stdin, one JSON reply per line on stdout.

A worker module defines:  NAME, info() -> dict, solve_case(case: dict) -> dict (see schema in docs/METHODOLOGY.md)
The worker measures ONLY the solver call (perf_counter around it); adapter (de)serialisation is excluded.
Solver stdout/stderr noise is redirected so it cannot corrupt the protocol.
"""
import contextlib
import io
import json
import os
import resource
import sys
import time
import traceback

PROTO_OUT = sys.stdout   # keep the real stdout for protocol messages


def timed(fn, *a, **kw):
    t = time.perf_counter()
    try:
        r = fn(*a, **kw)
        return r, (time.perf_counter() - t) * 1000.0, None
    except BaseException as e:   # noqa: BLE001  (RecursionError etc. must be captured, not crash the worker)
        if isinstance(e, (KeyboardInterrupt, SystemExit)):
            raise
        return None, (time.perf_counter() - t) * 1000.0, e


def err_dict(e):
    tb = traceback.format_exception(type(e), e, e.__traceback__)
    return {"type": type(e).__name__, "message": str(e)[:500], "traceback_tail": "".join(tb[-4:])[-1200:]}


def serve(mod):
    t_boot = time.perf_counter()
    for line in sys.stdin:
        line = line.strip()
        if not line:
            continue
        req = json.loads(line)
        cmd = req.get("cmd")
        if cmd == "shutdown":
            break
        sink = io.StringIO()
        try:
            with contextlib.redirect_stdout(sink), contextlib.redirect_stderr(sink):
                if cmd == "info":
                    out = mod.info()
                elif cmd == "ping":
                    out = {"pong": True}
                elif cmd == "clear_cache":
                    try:
                        from sympy.core.cache import clear_cache
                        clear_cache()
                        out = {"cleared": True}
                    except Exception as ce:   # noqa: BLE001
                        out = {"cleared": False, "error": str(ce)}
                elif cmd == "solve":
                    out = mod.solve_case(req["case"])
                elif cmd == "raw":     # engineering tests: solver-specific probe
                    out = mod.raw_probe(req["probe"])
                else:
                    out = {"error": {"type": "BadCommand", "message": str(cmd)}}
        except BaseException as e:   # noqa: BLE001
            out = {"outcome": "error", "error": err_dict(e), "worker_level_failure": True}
        out["req_id"] = req.get("req_id")
        out["maxrss_kb"] = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
        if sink.getvalue():
            out["captured_stdout_stderr"] = sink.getvalue()[:2000]
        PROTO_OUT.write(json.dumps(out, default=str, ensure_ascii=True) + "\n")
        PROTO_OUT.flush()
''')

W('bench/workers/safeinput.py', r'''"""Allow-list validation for benchmark-controlled math strings (never executes anything)."""
import re

_OK = re.compile(r"^[A-Za-z0-9_+\-*/^().,=<>!&| \[\]]*$")
_ATTR = re.compile(r"[A-Za-z_\)\]]\s*\.\s*[A-Za-z_]")


def check(s):
    if not isinstance(s, str) or len(s) > 400:
        raise ValueError("input rejected: not a short string")
    if "__" in s or not _OK.match(s) or _ATTR.search(s):
        raise ValueError("input rejected by allow-list: %r" % s[:80])
    return s
''')

W('bench/workers/sympy_worker.py', r'''"""Standalone SymPy adapter (configuration 'sympy-direct').

Design: SymPy is evaluated as a BARE ENGINE through its public API, exactly as an application developer
would call it, with NO added step generator and NO added domain-tracking layer.  Consequences (documented,
not hidden): SymPy provides no student-facing derivation (except sympy.integrals.manualintegrate.integral_steps
for integrals, captured as-is), and reports no domain restrictions unless the API returns them.

API choices (declared before execution):
  evaluate/simplify -> simplify      expand -> expand      factor -> factor
  solve             -> solveset(Eq, x, S.Reals | S.Complexes per case.domain)   [explicit domain]
  solve_system      -> solve(..., dict=True)          solve_inequality -> solve_univariate_inequality
  diff/partial/grad -> diff                           integrate -> integrate (indefinite & definite)
  limit             -> limit(dir='+-')                det/inverse/rref -> Matrix.det/.inv/.rref
  dsolve            -> dsolve                         partial_fractions -> apart
  nsolve            -> nsolve (K03, guess given) / Poly.nroots (K04)
Symbols are created real=True when case.domain == 'real' (explicit domain assumption), plain otherwise.
"""
import sys

import sympy as sp
from sympy.parsing.sympy_parser import convert_xor, parse_expr, standard_transformations

try:
    from . import common, safeinput
except ImportError:  # executed as a script
    import common
    import safeinput

NAME = "sympy-direct"
_TR = standard_transformations + (convert_xor,)
_ALLOWED = ["Integer", "Float", "Rational", "Symbol", "Function", "Mul", "Add", "Pow", "Derivative", "Eq",
            "sin", "cos", "tan", "exp", "log", "sqrt", "Abs", "pi", "E", "I", "oo", "atan", "asin", "acos", "sinh", "cosh", "sign"]


def _ns(extra=None):
    g = {k: getattr(sp, k) for k in _ALLOWED}
    g["__builtins__"] = {}
    g["abs"] = sp.Abs
    g["ln"] = sp.log
    g["e"] = sp.E      # Euler's number: SymPy's parser would otherwise read 'e' as a plain symbol
    g["i"] = sp.I
    if extra:
        g.update(extra)
    return g


_PARSE_MS = [0.0]


def parse(s, syms, extra=None):
    import time as _t
    _t0 = _t.perf_counter()
    try:
        return _parse(s, syms, extra)
    finally:
        _PARSE_MS[0] += (_t.perf_counter() - _t0) * 1000.0


def _parse(s, syms, extra=None):
    safeinput.check(s)
    loc = {k: v for k, v in syms.items() if not (extra and k in extra)}   # function names (y) must not be shadowed by Symbols
    return parse_expr(s, local_dict=loc, global_dict=_ns(extra), transformations=_TR, evaluate=True)


def info():
    return {"solver": NAME, "version": sp.__version__, "python": sys.version.split()[0],
            "commit": None, "package": "sympy", "api_methods": ["simplify", "expand", "factor", "solveset", "solve",
            "solve_univariate_inequality", "diff", "integrate", "limit", "Matrix.det/inv/rref", "dsolve", "apart",
            "nsolve", "Poly.nroots"]}


def _syms(case, names):
    real = case["domain"] == "real"
    out = {n: sp.Symbol(n, real=True) if real else sp.Symbol(n) for n in names}
    out["k"] = sp.Symbol("k", integer=True)          # summation index
    return out


def _expr_answer(e):
    return {"kind": "expr", "text": str(e)}


def _fmt_set(s):
    """Convert a SymPy set to the canonical, string-free structure."""
    if s == sp.S.EmptySet:
        return {"kind": "empty"}
    if s in (sp.S.Reals, sp.S.Complexes):
        return {"kind": "all", "domain": str(s)}
    if isinstance(s, sp.FiniteSet):
        return {"kind": "solutions", "values": [str(v) for v in s]}
    if isinstance(s, sp.Interval):
        return {"kind": "intervals", "intervals": [[str(s.start), str(s.end), bool(s.left_open), bool(s.right_open)]]}
    if isinstance(s, sp.Union) and all(isinstance(a, sp.Interval) for a in s.args):
        return {"kind": "intervals", "intervals": [[str(a.start), str(a.end), bool(a.left_open), bool(a.right_open)] for a in s.args]}
    if isinstance(s, sp.ConditionSet):
        return {"kind": "condition_set", "text": str(s)}
    return {"kind": "other_set", "text": str(s)}


def solve_case(case):
    op, pb = case["operation"], case["problem"]
    _PARSE_MS[0] = 0.0
    res = {"solver": NAME, "restrictions_reported": [], "steps": [], "native_flags": {},
           "domain_assumption_passed": True, "input_overrides_used": False}
    symnames = ["x", "y", "z"]
    S = _syms(case, symnames)
    x = S["x"]
    out = lambda **k: {**res, **k}

    def run(call_desc, fn):
        r, ms, e = common.timed(fn)
        res["api_call"] = call_desc
        res["time_ms"] = ms + _PARSE_MS[0]     # parse time included, as for the other two solvers
        return r, e

    if op in ("evaluate", "simplify"):
        expr = parse(pb["expr"], S)
        r, e = run("sympy.simplify(parse(expr))", lambda: sp.simplify(expr))
        if e: return out(outcome="error", error=common.err_dict(e))
        return out(outcome="answer", raw_output=str(r), answer=_expr_answer(r), input_sent=pb["expr"], native_status="ok")
    if op in ("expand", "factor"):
        expr = parse(pb["expr"], S)
        fn = sp.expand if op == "expand" else sp.factor
        r, e = run("sympy.%s(parse(expr))" % op, lambda: fn(expr))
        if e: return out(outcome="error", error=common.err_dict(e))
        return out(outcome="answer", raw_output=str(r), answer=_expr_answer(r), input_sent=pb["expr"], native_status="ok")
    if op == "solve":
        lhs, rhs = [parse(t, S) for t in pb["equation"].split("=", 1)]
        dom = sp.S.Reals if case["domain"] == "real" else sp.S.Complexes
        r, e = run("sympy.solveset(Eq(lhs, rhs), x, %s)" % dom, lambda: sp.solveset(sp.Eq(lhs, rhs), x, dom))
        if e: return out(outcome="error", error=common.err_dict(e))
        ans = _fmt_set(r)
        if ans["kind"] == "condition_set":
            return out(outcome="unsupported", unsupported_reason="solveset returned an unevaluated ConditionSet (SymPy's declared 'cannot solve')",
                       raw_output=str(r), answer=ans, input_sent=pb["equation"], native_status="ConditionSet")
        return out(outcome="answer", raw_output=str(r), answer=ans, input_sent=pb["equation"], native_status="ok")
    if op == "solve_system":
        eqs = []
        for t in pb["equations"]:
            l, rr = [parse(s, S) for s in t.split("=", 1)]
            eqs.append(sp.Eq(l, rr))
        r, e = run("sympy.solve(eqs, vars, dict=True)", lambda: sp.solve(eqs, [S[v] for v in pb["vars"]], dict=True))
        if e: return out(outcome="error", error=common.err_dict(e))
        if not r:
            ans = {"kind": "empty"}
        else:
            ans = {"kind": "system", "solutions": [{str(k): str(v) for k, v in d.items()} for d in r]}
        return out(outcome="answer", raw_output=str(r), answer=ans, input_sent=pb["equations"], native_status="ok")
    if op == "solve_inequality":
        s = pb["inequality"]
        for sym in (">=", "<=", ">", "<"):
            if sym in s:
                l, rr = [parse(t, S) for t in s.split(sym, 1)]
                rel = {">=": l >= rr, "<=": l <= rr, ">": l > rr, "<": l < rr}[sym]
                break
        r, e = run("sympy.solve_univariate_inequality(rel, x, relational=False)",
                   lambda: sp.solve_univariate_inequality(rel, x, relational=False))
        if e: return out(outcome="error", error=common.err_dict(e))
        return out(outcome="answer", raw_output=str(r), answer=_fmt_set(r), input_sent=s, native_status="ok")
    if op in ("diff", "partial_derivative"):
        expr = parse(pb["expr"], S)
        v = S[pb["var"]]
        order = pb.get("order", 1)
        r, e = run("sympy.diff(expr, %s, %d)" % (pb["var"], order), lambda: sp.diff(expr, v, order))
        if e: return out(outcome="error", error=common.err_dict(e))
        return out(outcome="answer", raw_output=str(r), answer=_expr_answer(r), input_sent=pb["expr"], native_status="ok")
    if op == "gradient":
        expr = parse(pb["expr"], S)
        r, e = run("[sympy.diff(expr, v) for v in vars]", lambda: [sp.diff(expr, S[v]) for v in pb["vars"]])
        if e: return out(outcome="error", error=common.err_dict(e))
        return out(outcome="answer", raw_output=str(r), answer={"kind": "vector", "items": [str(i) for i in r]},
                   input_sent=pb["expr"], native_status="ok")
    if op == "integrate":
        expr = parse(pb["expr"], S)
        definite = "lower" in pb
        if definite:
            lo, hi = parse(pb["lower"], S), parse(pb["upper"], S)
            r, e = run("sympy.integrate(expr, (x, lo, hi))", lambda: sp.integrate(expr, (x, lo, hi)))
        else:
            r, e = run("sympy.integrate(expr, x)", lambda: sp.integrate(expr, x))
        if e: return out(outcome="error", error=common.err_dict(e))
        # SymPy's own documented step API (rule tree); captured as-is, NOT paraphrased or enriched.
        try:
            from sympy.integrals.manualintegrate import integral_steps
            tree = repr(integral_steps(expr, x))
            res["steps"] = [{"rule": "sympy.integral_steps (raw rule tree)", "description": tree[:4000], "before": str(expr),
                             "after": "", "notes": [], "verification": "", "constraints": []}]
            res["native_flags"]["workings_source"] = "sympy.integrals.manualintegrate.integral_steps"
            res["native_flags"]["workings_tied_to_answer"] = False   # integrate() and integral_steps() are separate code paths
        except Exception as ex:   # noqa: BLE001
            res["native_flags"]["workings_error"] = type(ex).__name__
        return out(outcome="answer", raw_output=str(r), answer=_expr_answer(r), input_sent=pb["expr"], native_status="ok")
    if op == "limit":
        expr = parse(pb["expr"], S)
        pt = parse(pb["point"], S)
        dr = {"both": "+-", "+": "+", "-": "-"}[pb.get("direction", "both")]
        r, e = run("sympy.limit(expr, x, point, dir=%r)" % dr, lambda: sp.limit(expr, x, pt, dir=dr))
        if e: return out(outcome="error", error=common.err_dict(e))
        return out(outcome="answer", raw_output=str(r), answer=_expr_answer(r), input_sent=pb["expr"], native_status="ok")
    if op in ("det", "inverse", "rref"):
        M = sp.Matrix(pb["matrix"])
        fn = {"det": lambda: M.det(), "inverse": lambda: M.inv(), "rref": lambda: M.rref()[0]}[op]
        r, e = run("sympy.Matrix(...).%s" % {"det": "det()", "inverse": "inv()", "rref": "rref()[0]"}[op], fn)
        if e: return out(outcome="error", error=common.err_dict(e))
        if op == "det":
            return out(outcome="answer", raw_output=str(r), answer=_expr_answer(r), input_sent=pb["matrix"], native_status="ok")
        return out(outcome="answer", raw_output=str(r), answer={"kind": "matrix", "rows": [[str(c) for c in r.row(i)] for i in range(r.rows)]},
                   input_sent=pb["matrix"], native_status="ok")
    if op == "dsolve":
        f = sp.Function(pb["func"])
        ode = parse(pb["ode"], S, extra={pb["func"]: f})
        r, e = run("sympy.dsolve(ode, y(x))", lambda: sp.dsolve(ode, f(x)))
        if e: return out(outcome="error", error=common.err_dict(e))
        return out(outcome="answer", raw_output=str(r), answer={"kind": "equation", "text": str(r)}, input_sent=pb["ode"], native_status="ok")
    if op == "series":
        expr = parse(pb["expr"], S)
        pt = parse(pb["point"], S)
        r, e = run("sympy.series(expr, x, point, n)", lambda: sp.series(expr, x, pt, pb["n"]))
        if e: return out(outcome="error", error=common.err_dict(e))
        return out(outcome="answer", raw_output=str(r), answer=_expr_answer(r), input_sent=pb["expr"], native_status="ok")
    if op == "sum":
        expr = parse(pb["expr"], S)
        k = S["k"]
        lo, hi = parse(pb["lower"], S), parse(pb["upper"], S)
        r, e = run("sympy.summation(expr, (k, lo, hi))", lambda: sp.summation(expr, (k, lo, hi)))
        if e: return out(outcome="error", error=common.err_dict(e))
        return out(outcome="answer", raw_output=str(r), answer=_expr_answer(r), input_sent=pb["expr"], native_status="ok")
    if op in ("rank", "eigenvalues"):
        M = sp.Matrix(pb["matrix"])
        fn = (lambda: M.rank()) if op == "rank" else (lambda: M.eigenvals())
        r, e = run("sympy.Matrix(...).%s" % ("rank()" if op == "rank" else "eigenvals()"), fn)
        if e: return out(outcome="error", error=common.err_dict(e))
        if op == "rank":
            return out(outcome="answer", raw_output=str(r), answer=_expr_answer(r), input_sent=pb["matrix"], native_status="ok")
        vals = [str(v) for v, mult in r.items() for _ in range(int(mult))]
        return out(outcome="answer", raw_output=str(r), answer={"kind": "solutions", "values": vals}, input_sent=pb["matrix"], native_status="ok")
    if op == "partial_fractions":
        expr = parse(pb["expr"], S)
        r, e = run("sympy.apart(expr, x)", lambda: sp.apart(expr, x))
        if e: return out(outcome="error", error=common.err_dict(e))
        return out(outcome="answer", raw_output=str(r), answer=_expr_answer(r), input_sent=pb["expr"], native_status="ok")
    if op == "nsolve":
        l, rr = [parse(t, S) for t in pb["equation"].split("=", 1)]
        f = l - rr
        if pb.get("guess") is not None:
            r, e = run("sympy.nsolve(lhs-rhs, x, guess, prec=30)", lambda: sp.nsolve(f, x, sp.Float(pb["guess"]), prec=30))
            vals = [str(r)] if not e else []
        else:
            r, e = run("sympy.Poly(lhs-rhs, x).nroots(n=30)", lambda: sp.Poly(f, x).nroots(n=30))
            vals = [str(v) for v in r] if not e else []
        if e: return out(outcome="error", error=common.err_dict(e))
        return out(outcome="answer", raw_output=str(r), answer={"kind": "numeric", "values": vals}, input_sent=pb["equation"], native_status="ok")
    return out(outcome="unsupported", unsupported_reason="adapter has no mapping for operation %r" % op)


def raw_probe(p):
    """Engineering probes (see bench/engineering.py). Probe inputs are benchmark-authored strings."""
    kind = p["kind"]
    x = sp.Symbol("x")
    if kind == "solve_vs_solveset":
        eq = sp.sympify(p["lhs"], locals={"x": x}) - sp.sympify(p["rhs"], locals={"x": x})
        return {"outcome": "answer", "solve": str(sp.solve(eq, x)), "solveset": str(sp.solveset(eq, x, sp.S.Reals))}
    if kind == "parse":
        safeinput.check(p["text"])
        r, ms, e = common.timed(lambda: parse_expr(p["text"], local_dict={"x": x}, global_dict=_ns(), transformations=_TR))
        if e:
            return {"outcome": "error", "error": common.err_dict(e), "time_ms": ms}
        return {"outcome": "answer", "raw_output": str(r), "time_ms": ms}
    if kind == "eval":      # evaluate through the allow-listed adapter parse
        r, ms, e = common.timed(lambda: sp.simplify(parse(p["text"], {"x": x})))
        if e:
            return {"outcome": "error", "error": common.err_dict(e), "time_ms": ms}
        return {"outcome": "answer", "raw_output": str(r), "time_ms": ms}
    if kind == "solve_for":
        syms = {"x": x, "y": sp.Symbol("y"), "z": sp.Symbol("z")}
        l, rr = [parse(s, syms) for s in p["equation"].split("=", 1)]
        v = syms[p["var"]]
        r, ms, e = common.timed(lambda: sp.solveset(sp.Eq(l, rr), v, sp.S.Reals))
        if e:
            return {"outcome": "error", "error": common.err_dict(e), "time_ms": ms}
        return {"outcome": "answer", "raw_output": str(r), "answer_values": [str(i) for i in r] if isinstance(r, sp.FiniteSet) else None, "time_ms": ms}
    if kind == "canary":    # engine's OWN string interface, as shipped (sympy.sympify); benign payload: math.factorial(4)
        r, ms, e = common.timed(lambda: sp.sympify(p["text"]))
        if e:
            return {"outcome": "error", "error": common.err_dict(e), "time_ms": ms}
        return {"outcome": "answer", "raw_output": str(r), "time_ms": ms}
    return {"outcome": "error", "error": {"type": "UnknownProbe", "message": kind}}


if __name__ == "__main__":
    common.serve(sys.modules[__name__])
''')

W('bench/workers/smartsolver_worker.py', r'''"""SmartSolver adapter (package 'ed-master-smartsolver', import name 'smartsolver').

Mapping (declared before execution; one documented public method per operation family):
  solve (C, K02, K03, K04, L01, L02, L04) -> SmartSolver.solve_equation(equation, 'x')
  solve_system (D01-D03, I01)             -> SmartSolver.solve_linear_system(equations, variables)
  diff (F01-F06)                          -> SmartSolver.differentiate(expr, 'x')
  partial_derivative (J03)                -> SmartSolver.partial_derivative(expr, 'x', ['x','y'])
  gradient (J04)                          -> SmartSolver.gradient(expr, vars)
  integrate (G01-G06)                     -> SmartSolver.integrate(expr, 'x'[, lower, upper])
  limit (H01-H04)                         -> SmartSolver.limit(expr, 'x', point)      (documented default direction '+')
  rref (I04)                              -> SmartSolver.matrix_rref(matrix)
  partial_fractions (K01)                 -> SmartSolver.partial_fractions(expr)
Operations with NO public method (evaluate, simplify, expand, factor, solve_inequality, det, inverse, dsolve,
nsolve) are reported as UNSUPPORTED with the introspected method list as evidence; nothing is manufactured.
K03/K04 (numeric root finding) have no numeric API, so the generic equation solver is used (documented in cases.json).
SmartSolver result objects are converted with str(); steps with the library's own step_to_dict / StepRenderer.
No answer is modified.
"""
import re
import sys

import smartsolver
from smartsolver import SmartSolver, StepRenderer, step_to_dict

try:
    from . import common
except ImportError:
    import common

NAME = "smartsolver"
S = SmartSolver()
PUBLIC = sorted(m for m in dir(SmartSolver) if not m.startswith("_") and callable(getattr(SmartSolver, m)))


def info():
    import importlib.metadata as md
    dist = (md.packages_distributions().get("smartsolver") or ["ed-master-smartsolver"])[0]
    return {"solver": NAME, "version": md.version(dist), "package": dist,
            "module_version": getattr(smartsolver, "__version__", None), "python": sys.version.split()[0], "commit": None,
            "sympy_version": md.version("sympy"), "public_methods": PUBLIC}


def _steps(r):
    out = []
    for s in r.get("steps", []):
        d = step_to_dict(s)
        out.append({"rule": d.get("rule_applied", ""), "description": d.get("description", ""), "before": "",
                    "after": d.get("expression", ""), "notes": [], "verification": "", "constraints": [],
                    "substeps": d.get("substeps", [])})
    return out


def _restrictions(steps):
    out = []
    for s in steps:
        if s["rule"] in ("Domain", "Extraneous root") or re.search(r"reject|undefined|denominator|must be positive", s["description"], re.I):
            out.append(s["description"])
    return out


def _two_sided_limit(pb, base):
    """SmartSolver documents only one-sided limits.  A two-sided limit exists iff both one-sided limits exist and are equal, so this
    adapter makes TWO native one-sided calls (flagged composed_from) and reports 'undefined' when they differ."""
    outs, total, steps, raws = [], 0.0, [], []
    for d in ("+", "-"):
        r, ms, e = common.timed(lambda d=d: S.limit(pb["expr"], "x", point=pb["point"], direction=d))
        total += ms
        if e is not None:
            return {**base, "outcome": "error", "error": common.err_dict(e), "api_call": "limit(..., direction=%r)" % d, "time_ms": total, "steps": [], "restrictions_reported": []}
        outs.append(r["limit"])
        steps += _steps(r)
        raws.append({k: str(v) for k, v in r.items() if k != "steps"})
    res = {**base, "outcome": "answer", "api_call": "limit(..., direction='+') ; limit(..., direction='-')  [composed two-sided limit]", "time_ms": total,
           "steps": steps, "restrictions_reported": _restrictions(steps), "raw_output": raws, "native_status": "ok", "input_sent": pb,
           "native_flags": {"composed_from": ["limit +", "limit -"], "one_sided": [str(o) for o in outs]}}
    if str(outs[0]) == str(outs[1]):
        return {**res, "answer": {"kind": "expr", "text": str(outs[0])}}
    return {**res, "answer": {"kind": "undefined", "reason": "one-sided limits differ: %s (from +) vs %s (from -)" % (outs[0], outs[1])}}


UNSUPPORTED = {"evaluate", "simplify", "expand", "factor", "solve_inequality", "det", "inverse", "dsolve", "nsolve_api"}


def solve_case(case):
    op, pb = case["operation"], case["problem"]
    base = {"solver": NAME, "domain_assumption_passed": False, "input_overrides_used": False}   # SmartSolver has no domain parameter

    def unsupported(why):
        return {**base, "outcome": "unsupported", "unsupported_reason": why, "api_call": None, "steps": [], "restrictions_reported": [],
                "native_flags": {"public_methods_checked": PUBLIC}}

    call = None
    if op == "solve":
        call = ("solve_equation(%r, 'x')" % pb["equation"], lambda: S.solve_equation(pb["equation"], "x"), "solutions")
    elif op == "nsolve":
        call = ("solve_equation(%r, 'x')  [no numeric API; generic equation solver]" % pb["equation"],
                lambda: S.solve_equation(pb["equation"], "x"), "solutions")
    elif op == "solve_system" and pb.get("linear") is False:
        return unsupported("solve_linear_system is documented for LINEAR systems only; this system is nonlinear")
    elif op == "solve_system":
        call = ("solve_linear_system(%r, %r)" % (pb["equations"], pb["vars"]),
                lambda: S.solve_linear_system(pb["equations"], pb["vars"]), "system")
    elif op == "diff":
        call = ("differentiate(%r, %r, order=%d)" % (pb["expr"], pb["var"], pb.get("order", 1)),
                lambda: S.differentiate(pb["expr"], pb["var"], pb.get("order", 1)), "derivative")
    elif op == "partial_derivative":
        call = ("partial_derivative(%r, %r, %r)" % (pb["expr"], pb["var"], pb["vars"]),
                lambda: S.partial_derivative(pb["expr"], pb["var"], pb["vars"]), "derivative")
    elif op == "gradient":
        call = ("gradient(%r, %r)" % (pb["expr"], pb["vars"]), lambda: S.gradient(pb["expr"], pb["vars"]), "gradient")
    elif op == "integrate":
        if "lower" in pb:
            call = ("integrate(%r, 'x', lower=%r, upper=%r)" % (pb["expr"], pb["lower"], pb["upper"]),
                    lambda: S.integrate(pb["expr"], "x", lower=pb["lower"], upper=pb["upper"]), "integral")
        else:
            call = ("integrate(%r, 'x')" % pb["expr"], lambda: S.integrate(pb["expr"], "x"), "integral")
    elif op == "limit" and pb.get("direction", "both") == "both":
        return _two_sided_limit(pb, base)
    elif op == "limit":
        call = ("limit(%r, 'x', point=%r, direction=%r)" % (pb["expr"], pb["point"], pb["direction"]),
                lambda: S.limit(pb["expr"], "x", point=pb["point"], direction=pb["direction"]), "limit")
    elif op == "rref":
        call = ("matrix_rref(%r)" % (pb["matrix"],), lambda: S.matrix_rref(pb["matrix"]), "matrix")
    elif op == "partial_fractions":
        call = ("partial_fractions(%r)" % pb["expr"], lambda: S.partial_fractions(pb["expr"]), "decomposition")
    else:
        return unsupported("SmartSolver has no public method for operation %r (public methods: %s)" % (op, ", ".join(PUBLIC)))

    desc, fn, kind = call
    method_name = desc.split("(")[0]
    if not hasattr(S, method_name):          # a newer/older version may have removed or renamed the method
        return unsupported("SmartSolver has no public method %r in this version (public methods: %s)" % (method_name, ", ".join(PUBLIC)))
    r, ms, e = common.timed(fn)
    res = {**base, "api_call": desc, "time_ms": ms, "input_sent": pb}
    if e is not None:
        return {**res, "outcome": "error", "error": common.err_dict(e), "steps": [], "restrictions_reported": []}
    steps = _steps(r)
    flags = {"steps_verified": r.get("steps_verified"), "steps_mode": r.get("steps_mode"),
             "consistent": r.get("consistent"), "methods": [str(m) for m in r.get("methods", [])],
             "result_keys": sorted(k for k in r if k != "steps")}
    raw = {k: str(v) for k, v in r.items() if k != "steps"}
    raw["steps_text"] = StepRenderer.to_text(r.get("steps", []))
    res.update(outcome="answer", steps=steps, restrictions_reported=_restrictions(steps), native_flags=flags, raw_output=raw, native_status="ok")
    if kind == "solutions":
        vals = [str(v) for v in r.get("solutions", [])]
        ans = {"kind": "solutions", "values": vals} if vals else {"kind": "empty"}
        if r.get("general_solutions") is not None:
            ans["general_solutions"] = str(r["general_solutions"])
    elif kind == "system":
        by = r.get("solutions_by_variable") or {}
        if r.get("consistent") is False:
            ans = {"kind": "empty"}
        elif by:
            ans = {"kind": "system", "solutions": [{k: str(v) for k, v in by.items()}]}
        elif r.get("consistent") is True and re.search(r"infinite", str(r.get("solution_latex", "")) + " " + " ".join(st["rule"] + " " + st["description"] for st in steps), re.I):
            ans = {"kind": "infinite", "note": "engine states infinitely many solutions (no parametric family returned)"}
        else:
            ans = {"kind": "empty"} if not r.get("solutions") else {"kind": "other", "text": str(r.get("solutions"))}
    elif kind == "derivative":
        ans = {"kind": "expr", "text": str(r["derivative"])}
    elif kind == "gradient":
        g = r["gradient"]
        ans = {"kind": "vector", "items": [str(i) for i in list(g)]}
    elif kind == "integral":
        ans = {"kind": "expr", "text": str(r["integral"])}
    elif kind == "limit":
        ans = {"kind": "expr", "text": str(r["limit"])}
    elif kind == "matrix":
        m = r["matrix"]
        ans = {"kind": "matrix", "rows": [[str(m[i, j]) for j in range(m.cols)] for i in range(m.rows)]}
    elif kind == "decomposition":
        ans = {"kind": "expr", "text": str(r["decomposition"])}
    return {**res, "answer": ans}


def raw_probe(p):
    kind = p["kind"]
    if kind == "call":
        fn = getattr(S, p["method"])
        r, ms, e = common.timed(lambda: fn(*p.get("args", []), **p.get("kwargs", {})))
        if e:
            return {"outcome": "error", "error": common.err_dict(e), "time_ms": ms}
        return {"outcome": "answer", "raw_output": {k: str(v)[:300] for k, v in r.items() if k != "steps"}, "time_ms": ms,
                "nsteps": len(r.get("steps", []))}
    return {"outcome": "error", "error": {"type": "UnknownProbe", "message": kind}}


if __name__ == "__main__":
    common.serve(sys.modules[__name__])
''')

W('bench/workers/stepsolver_worker.py', r'''"""StepSolver adapter (package 'stepsolver', Solver().solve(<ASCII query>)).

Mapping (declared before execution) -- StepSolver's documented ASCII operations:
  evaluate/simplify -> simplify(e)    expand -> expand(e)     factor -> factor(e)
  solve / nsolve    -> solve(eq, x)   (the hard suite contains no numeric-root cases)
  solve_system      -> solve([eq1, eq2], [x, y])    solve_inequality -> solve_inequality(ineq, x)
  diff/partial      -> diff(e, x)     gradient -> TWO native diff(e, v) calls (no gradient op; composition is
                       definitional: gradient == vector of partials; flagged composed_from)
  integrate         -> integrate(e, x) / integrate(e, x, a, b)      limit -> limit(e, x, p)
  det/inverse/rref  -> det(matrix(..)) / inverse(matrix(..)) / rref(matrix(..))
  dsolve            -> dsolve(<ode> = 0, y(x)); the dataset's SymPy Derivative(...) notation is translated
                       mechanically to native diff(...) syntax.  Legacy per-case dsolve overrides are ignored
                       so every ODE query is constructed from problem metadata.
  partial_fractions -> apart(e, x)
Serialisation: answers are read from StepSolver's typed MathValue tree with its own format_expression(); steps from
SolutionStep fields (rule, explanation, before, after, notes, verification, introduced_constraints).  Nothing is
rewritten.  Parse errors are 'input_incompatible'; UnsolvedResult is 'unsupported' (the solver's own declaration).
"""
import sys
import re

import stepsolver
from stepsolver import (DivergentResult, ExactResult, MappingValue, MatrixValue, NoSolutionValue, NumericResult,
                        ScalarValue, SequenceValue, BooleanValue, Solver, UndefinedResult, UniversalSolutionValue,
                        UnsolvedResult, format_ascii, format_expression, ParseError, StepSolverError,
                        SingularMatrixResult)
from stepsolver.ast import Symbol as AstSymbol

try:
    from . import common
except ImportError:
    import common

NAME = "stepsolver"
SOLVER = Solver()


def info():
    import importlib.metadata as md
    dist = (md.packages_distributions().get("stepsolver") or ["stepsolver"])[0]
    return {"solver": NAME, "version": md.version(dist), "package": dist, "python": sys.version.split()[0],
            "commit": None, "sympy_version": md.version("sympy"), "operations": [o.value for o in stepsolver.Operation]}


def _mat(rows):
    return "matrix([" + ", ".join("[" + ", ".join(str(c) for c in r) + "]" for r in rows) + "])"


def _imag(q):
    """The dataset writes the imaginary unit as I (SymPy); StepSolver's documented constant is i."""
    import re
    return re.sub(r"\bI\b", "i", q)


def _ln(q, on):
    """StepSolver has no 'ln' alias: its natural log token is 'log' (an unknown 'ln' is silently accepted as an opaque
    function).  Adapter normalisation ln( -> log( is applied (documented) and the as-specified query is ALSO run and
    recorded as 'as_specified_variant' so the silent-misparse hazard stays visible."""
    return q.replace("ln(", "log(") if on else q


def build_queries(case, normalise_ln=True):
    qs, ov = _build(case)
    return [_ln(_imag(q), normalise_ln) for q in qs] if qs else qs, ov


def _build(case):
    op, pb = case["operation"], case["problem"]
    ov = case.get("adapter_inputs", {}).get("stepsolver")
    if ov and op != "dsolve":
        return [ov], True
    if op in ("evaluate", "simplify"):
        return ["simplify(%s)" % pb["expr"]], False
    if op in ("expand", "factor"):
        return ["%s(%s)" % (op, pb["expr"])], False
    if op in ("solve", "nsolve"):
        return ["solve(%s, x)" % pb["equation"]], False
    if op == "solve_system":
        return ["solve([%s], [%s])" % (", ".join(pb["equations"]), ", ".join(pb["vars"]))], False
    if op == "solve_inequality":
        return ["solve_inequality(%s, x)" % pb["inequality"]], False
    if op in ("diff", "partial_derivative"):
        if pb.get("order", 1) > 1:
            return ["diff(%s, %s, %d)" % (pb["expr"], pb["var"], pb["order"])], False
        return ["diff(%s, %s)" % (pb["expr"], pb["var"])], False
    if op == "gradient":
        return ["diff(%s, %s)" % (pb["expr"], v) for v in pb["vars"]], False
    if op == "integrate":
        if "lower" in pb:
            return ["integrate(%s, x, %s, %s)" % (pb["expr"], pb["lower"], pb["upper"])], False
        return ["integrate(%s, x)" % pb["expr"]], False
    if op == "limit":
        d = {"both": "", "+": ", right", "-": ", left"}[pb.get("direction", "both")]
        return ["limit(%s, x, %s%s)" % (pb["expr"], pb["point"], d)], False
    if op == "series":
        return ["series(%s, %s, %s, %d)" % (pb["expr"], pb["var"], pb["point"], pb["n"])], False
    if op == "sum":
        return ["sum(%s, %s, %s, %s)" % (pb["expr"], pb["var"], pb["lower"], pb["upper"])], False
    if op in ("rank", "eigenvalues"):
        return ["%s(%s)" % (op, _mat(pb["matrix"]))], False
    if op in ("det", "inverse", "rref"):
        return ["%s(%s)" % (op, _mat(pb["matrix"]))], False
    if op == "partial_fractions":
        return ["apart(%s, x)" % pb["expr"]], False
    if op == "dsolve":
        # The dataset stores the ODE residual in SymPy's Derivative(...) notation. Translate it
        # mechanically to StepSolver's native diff(...) syntax; no mathematical rewriting is done.
        func, var = pb.get("func", "y"), pb.get("var", "x")
        native = re.sub(r"Derivative\(\s*([^,]+?)\s*,\s*\(([^()]*)\)\s*\)", r"diff(\1, \2)", pb["ode"])
        native = native.replace("Derivative(", "diff(")
        return ["dsolve(%s = 0, %s(%s))" % (native, func, var)], False
    return None, False


def _val(v, op):
    """Typed MathValue -> canonical answer dict (no string re-interpretation)."""
    if isinstance(v, NoSolutionValue):
        return {"kind": "empty"}
    if isinstance(v, UniversalSolutionValue):
        return {"kind": "all", "domain": v.domain.value,
                "exclusions": [format_expression(e) for e in v.exclusions]}
    if isinstance(v, BooleanValue):
        return {"kind": "bool", "value": v.value}
    if isinstance(v, MatrixValue):
        return {"kind": "matrix", "rows": [[format_expression(c) for c in r] for r in v.rows]}
    if isinstance(v, ScalarValue):
        return {"kind": "expr", "text": format_expression(v.expression)}
    if isinstance(v, MappingValue):
        return {"kind": "mapping", "entries": {format_expression(e.key): _val(e.value, op) for e in v.entries}}
    if isinstance(v, SequenceValue):
        return {"kind": "sequence", "items": [_val(i, op) for i in v.items]}
    return {"kind": "unknown", "text": repr(v)[:300]}


def _canon(av, op):
    """Flatten sequence/mapping answers into the benchmark's canonical kinds, documenting every rule."""
    if op in ("solve", "nsolve") and av["kind"] == "sequence":
        vals = []
        for it in av["items"]:
            if it["kind"] == "mapping" and len(it["entries"]) == 1:
                e = list(it["entries"].values())[0]
                vals.append(e.get("text", str(e)))
            else:
                return {"kind": "other", "detail": av}
        return {"kind": "solutions", "values": vals} if vals else {"kind": "empty"}
    if op == "solve_system" and av["kind"] == "sequence":
        sols = []
        for it in av["items"]:
            if it["kind"] == "mapping":
                sols.append({k: v.get("text", str(v)) for k, v in it["entries"].items()})
            else:
                return {"kind": "other", "detail": av}
        return {"kind": "system", "solutions": sols} if sols else {"kind": "empty"}
    if op == "rref" and av["kind"] == "sequence" and av["items"] and av["items"][0]["kind"] == "matrix":
        return av["items"][0]      # (matrix, pivots) -> matrix ; pivots dropped (documented)
    if op == "solve_inequality" and av["kind"] == "expr":
        return {"kind": "relation", "text": av["text"]}
    if op == "eigenvalues" and av["kind"] == "mapping":
        vals = []
        for k_, m_ in av["entries"].items():
            vals += [k_] * int(m_.get("text", "1"))
        return {"kind": "solutions", "values": vals}
    if op == "dsolve" and av["kind"] == "expr":
        return {"kind": "equation", "text": av["text"]}
    return av


def _steps(res):
    out = []
    for s in res.steps:
        out.append({"rule": s.rule, "description": s.explanation, "before": format_expression(s.before),
                    "after": format_expression(s.after),
                    "notes": [[n.label, format_expression(n.expression)] for n in s.notes],
                    "verification": "%s: %s" % (s.verification.method.value, s.verification.detail),
                    "constraints": [[c.explanation, format_expression(c.expression)] for c in s.introduced_constraints]})
    return out


def solve_case(case):
    queries, overridden = build_queries(case)
    raw_queries, _ = build_queries(case, normalise_ln=False)
    ln_normalised = bool(queries) and queries != raw_queries
    overridden = overridden or ln_normalised
    base = {"solver": NAME, "domain_assumption_passed": bool(case.get("domain")), "input_overrides_used": bool(overridden)}
    if queries is None:
        return {**base, "outcome": "unsupported", "unsupported_reason": "adapter has no mapping for operation %r" % case["operation"],
                "steps": [], "restrictions_reported": []}
    op = case["operation"]
    domain = case.get("domain")
    results, total_ms, raws = [], 0.0, []
    for q in queries:
        r, ms, e = common.timed(lambda q=q: SOLVER.solve(q, domain=domain) if domain else SOLVER.solve(q))
        total_ms += ms
        if e is not None:
            kind = "input_incompatible" if isinstance(e, ParseError) else "error"
            err = common.err_dict(e)
            err["contract_exception"] = isinstance(e, StepSolverError)
            return {**base, "outcome": kind, "error": err, "api_call": "Solver().solve(%r)" % q, "input_sent": queries, "time_ms": total_ms,
                    "steps": [], "restrictions_reported": [], "raw_output": raws or None}
        results.append(r)
        raws.append(format_ascii(r))
    steps, restr, answers = [], [], []
    native = {"result_types": [type(r).__name__ for r in results], "composed_from": queries if len(queries) > 1 else None}
    for q, r in zip(queries, results):
        st = _steps(r)
        steps += st
        for s in st:
            restr += [c[0] + " | " + c[1] for c in s["constraints"]]
        if isinstance(r, UndefinedResult):
            restr.append("undefined: " + r.reason)
        native.setdefault("opaque_steps", 0)
        native["opaque_steps"] += sum(1 for s in st if s["rule"].strip().lower() == "compute exact result")
    if ln_normalised:
        native["input_normalisation"] = "ln( -> log(  (StepSolver natural-log token is 'log')"
        v, _, ve = common.timed(lambda: SOLVER.solve(raw_queries[0]))
        native["as_specified_variant"] = {"query": raw_queries[0], "result_type": type(v).__name__ if ve is None else type(ve).__name__,
                                          "raw_output": format_ascii(v) if ve is None else str(ve)[:200]}
    res = {**base, "api_call": " ; ".join("Solver().solve(%r)" % q for q in queries), "input_sent": queries, "time_ms": total_ms,
           "steps": steps, "restrictions_reported": restr, "native_flags": native, "raw_output": raws if len(raws) > 1 else raws[0]}
    r0 = results[0]
    for r in results:
        if isinstance(r, UnsolvedResult):
            return {**res, "outcome": "unsupported", "unsupported_reason": "UnsolvedResult: " + r.reason[:300], "native_status": "UnsolvedResult"}
    if len(results) > 1 and all(isinstance(r, ExactResult) for r in results):   # gradient composition
        items = [_val(r.value, op) for r in results]
        return {**res, "outcome": "answer", "native_status": "ExactResult", "answer": {"kind": "vector", "items": [i.get("text", str(i)) for i in items]}}
    if isinstance(r0, ExactResult):
        return {**res, "outcome": "answer", "native_status": "ExactResult", "answer": _canon(_val(r0.value, op), op)}
    if isinstance(r0, UndefinedResult):
        return {**res, "outcome": "answer", "native_status": "UndefinedResult", "answer": {"kind": "undefined", "reason": r0.reason}}
    if isinstance(r0, DivergentResult):
        return {**res, "outcome": "answer", "native_status": "DivergentResult", "answer": {"kind": "divergent", "reason": r0.reason, "divergence": r0.kind.value}}
    if isinstance(r0, NumericResult):
        residual_list = [float(item) for item in r0.residuals]
        tolerance = float(r0.value.tolerance)
        variable = next((arg.name for arg in r0.query.arguments if isinstance(arg, AstSymbol)), None)
        return {**res, "outcome": "answer", "native_status": "NumericResult",
                "answer": {"kind": "numeric",
                           "values": [format_expression(v) for v in r0.value.roots],
                           "tolerance": tolerance,
                           "variable": variable,
                           "method": r0.method,
                           "residuals": residual_list,
                           "converged": all(abs(item) <= tolerance for item in residual_list)}}
    if isinstance(r0, SingularMatrixResult):
        return {**res, "outcome": "answer", "native_status": "SingularMatrixResult",
                "answer": {"kind": "singular_matrix",
                           "reason": r0.reason,
                           "determinant": format_expression(r0.determinant) if r0.determinant is not None else None,
                           "rank": r0.rank,
                           "size": r0.size}}
    return {**res, "outcome": "error", "error": {"type": "UnexpectedResultType", "message": type(r0).__name__}}


def raw_probe(p):
    kind = p["kind"]
    if kind == "query":
        r, ms, e = common.timed(lambda: SOLVER.solve(p["query"]))
        if e:
            return {"outcome": "error", "error": {**common.err_dict(e), "contract_exception": isinstance(e, StepSolverError)}, "time_ms": ms}
        return {"outcome": "answer", "native_status": type(r).__name__, "raw_output": format_ascii(r), "time_ms": ms,
                "nsteps": len(r.steps)}
    return {"outcome": "error", "error": {"type": "UnknownProbe", "message": kind}}


if __name__ == "__main__":
    common.serve(sys.modules[__name__])
''')
print('wrote', 7, 'files')

wrote 7 files


In [5]:
#@title 3c. Benchmark source: runner, judge, derivation rubric { display-mode: "form" }
W('bench/runner.py', r'''#!/usr/bin/env python3
"""Run the dataset against each solver (each in its own venv + worker process). Raw replies are preserved verbatim.

  python -m bench.runner [--solver sympy-direct|smartsolver|stepsolver ...] [--category A] [--case C05]
                         [--repeats 5] [--timeout 20] [--no-timing] [--out results/raw]

Protocol per solver:
  1. cold-start trials  : N fresh processes; time to ready (spawn->first reply to 'ping') and first solve of C01.
  2. functional pass    : each selected case once, in dataset order, per-case timeout (process killed + restarted on timeout).
  3. warm repeats       : for each case that produced an answer: `repeats` extra runs, each preceded by a SymPy cache clear
                          (untimed), same process -> warm-process / cold-cache latency; canonical outputs hashed (repeatability).
Same cases, same timeout, same machine, same protocol for every solver. No solver gets special pre-processing.
"""
import argparse
import hashlib
import json
import os
import pathlib
import select
import statistics
import subprocess
import sys
import tempfile
import time

ROOT = pathlib.Path(__file__).resolve().parent.parent
WORKERS = {
    "sympy-direct": ("sympy", ROOT / "bench/workers/sympy_worker.py"),
    "smartsolver": ("smartsolver", ROOT / "bench/workers/smartsolver_worker.py"),
    "stepsolver": ("stepsolver", ROOT / "bench/workers/stepsolver_worker.py"),
}


class Worker:
    def __init__(self, name):
        env_name, script = WORKERS[name]
        self.name = name
        self.py = str(ROOT / "envs" / env_name / "bin" / "python")
        self.script = str(script)
        self.proc = None
        self.spawn_to_ready_ms = None
        self.start()

    def start(self):
        self.cwd = tempfile.mkdtemp(prefix="bench_wd_")       # neutral working directory (no repo files visible)
        env = {"PATH": "/usr/bin:/bin", "PYTHONHASHSEED": "0", "PYTHONNOUSERSITE": "1", "PYTHONDONTWRITEBYTECODE": "1",
               "HOME": self.cwd, "LC_ALL": "C.UTF-8"}
        t = time.perf_counter()
        self.proc = subprocess.Popen([self.py, "-s", self.script], stdin=subprocess.PIPE, stdout=subprocess.PIPE,
                                     stderr=subprocess.PIPE, text=True, bufsize=1, cwd=self.cwd, env=env)
        r = self.call({"cmd": "ping"}, timeout=120)
        self.spawn_to_ready_ms = (time.perf_counter() - t) * 1000.0
        return r

    def call(self, req, timeout):
        """Returns (reply|None, status) where status in ok|timeout|crash."""
        try:
            self.proc.stdin.write(json.dumps(req) + "\n")
            self.proc.stdin.flush()
        except BrokenPipeError:
            return None, "crash"
        end = time.time() + timeout
        buf = ""
        while time.time() < end:
            rem = max(0.0, end - time.time())
            r, _, _ = select.select([self.proc.stdout], [], [], min(rem, 0.5))
            if r:
                line = self.proc.stdout.readline()
                if not line:
                    return None, "crash"
                try:
                    return json.loads(line), "ok"
                except json.JSONDecodeError:
                    buf += line
            elif self.proc.poll() is not None:
                return None, "crash"
        return None, "timeout"

    def kill(self):
        if self.proc and self.proc.poll() is None:
            self.proc.kill()
            try:
                self.proc.wait(5)
            except Exception:   # noqa: BLE001
                pass

    def stderr_tail(self):
        try:
            if self.proc.poll() is not None:
                return self.proc.stderr.read()[-1500:]
        except Exception:   # noqa: BLE001
            pass
        return ""

    def close(self):
        try:
            self.proc.stdin.write(json.dumps({"cmd": "shutdown"}) + "\n")
            self.proc.stdin.flush()
            self.proc.wait(5)
        except Exception:   # noqa: BLE001
            self.kill()


def canon_hash(reply):
    """Hash of the semantically relevant output (answer, steps, outcome) -- excludes timings/memory."""
    keep = {k: reply.get(k) for k in ("outcome", "answer", "steps", "restrictions_reported", "unsupported_reason", "raw_output")}
    err = reply.get("error")
    if err:
        keep["error"] = {k: err.get(k) for k in ("type", "message")}
    return hashlib.sha256(json.dumps(keep, sort_keys=True, default=str).encode()).hexdigest()[:16]


def run_solver(name, cases, repeats, timeout, timing, cold_trials=5):
    out = {"solver": name, "cold_start": [], "rows": {}, "info": None, "maxrss_kb": 0, "timeouts": 0, "crashes": 0}
    w = Worker(name)
    reply, _ = w.call({"cmd": "info"}, 60)
    out["info"] = reply
    _d = json.load(open(ROOT / "data/cases.json"))
    c01 = next((c for c in _d["cases"] if c["id"] == _d.get("cold_start_case")), _d["cases"][0])   # cold-start probe case (declared in cases.json)
    if timing:
        for _ in range(cold_trials):                                  # cold-start trials: fresh process each time
            cw = Worker(name)
            t = time.perf_counter()
            r, st = cw.call({"cmd": "solve", "case": c01, "req_id": "cold"}, timeout)
            out["cold_start"].append({"spawn_to_ready_ms": cw.spawn_to_ready_ms, "first_solve_wall_ms": (time.perf_counter() - t) * 1000.0,
                                      "first_solve_inner_ms": (r or {}).get("time_ms"), "status": st})
            cw.close()
    for c in cases:
        t = time.perf_counter()
        r, st = w.call({"cmd": "solve", "case": c, "req_id": c["id"]}, timeout)
        wall = (time.perf_counter() - t) * 1000.0
        row = {"case_id": c["id"], "wall_ms": wall, "transport": st}
        if st == "ok":
            row["reply"] = r
            row["hash"] = canon_hash(r)
            out["maxrss_kb"] = max(out["maxrss_kb"], r.get("maxrss_kb", 0))
        else:
            row["reply"] = {"outcome": "timeout" if st == "timeout" else "infra_error", "solver": name,
                            "error": {"type": "WorkerTimeout" if st == "timeout" else "WorkerCrash",
                                      "message": "no reply within %ss" % timeout if st == "timeout" else w.stderr_tail()}}
            out["timeouts" if st == "timeout" else "crashes"] += 1
            w.kill()
            w = Worker(name)      # restart for the remaining cases (state of a killed process is not trusted)
        out["rows"][c["id"]] = row
    if timing and repeats:
        for c in cases:
            row = out["rows"][c["id"]]
            if row["transport"] != "ok" or row["reply"].get("outcome") != "answer":
                continue
            times, hashes = [], []
            for _ in range(repeats):
                w.call({"cmd": "clear_cache"}, 30)
                r, st = w.call({"cmd": "solve", "case": c, "req_id": c["id"] + "-rep"}, timeout)
                if st != "ok":
                    row.setdefault("repeat_failures", []).append(st)
                    if st != "ok":
                        w.kill(); w = Worker(name)
                    continue
                times.append(r.get("time_ms"))
                hashes.append(canon_hash(r))
            row["warm_times_ms"] = times
            row["repeat_hashes"] = hashes
            row["repeatable"] = len(set(hashes + [row["hash"]])) == 1 if hashes else None
    w.close()
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--solver", nargs="*", default=list(WORKERS))
    ap.add_argument("--category")
    ap.add_argument("--case", nargs="*")
    ap.add_argument("--repeats", type=int, default=5)
    ap.add_argument("--timeout", type=float, default=20.0)
    ap.add_argument("--no-timing", action="store_true")
    ap.add_argument("--out", default=str(ROOT / "results/raw"))
    a = ap.parse_args()
    data = json.load(open(ROOT / "data/cases.json"))
    cases = data["cases"]
    if a.category:
        cases = [c for c in cases if c["category"] == a.category.upper()]
    if a.case:
        cases = [c for c in cases if c["id"] in set(a.case)]
    pathlib.Path(a.out).mkdir(parents=True, exist_ok=True)
    print("cases loaded: %d (dataset total %d)" % (len(cases), data["count"]))
    for s in a.solver:
        t = time.time()
        res = run_solver(s, cases, a.repeats, a.timeout, not a.no_timing)
        res["protocol"] = {"repeats": a.repeats, "timeout_s": a.timeout, "cases_run": [c["id"] for c in cases],
                           "cpu_count": os.cpu_count(), "started_unix": t}
        (pathlib.Path(a.out) / ("%s.json" % s)).write_text(json.dumps(res, indent=1, default=str))
        oc = {}
        for r in res["rows"].values():
            o = r["reply"].get("outcome")
            oc[o] = oc.get(o, 0) + 1
        print("%-13s %5.1fs  outcomes=%s  maxrss=%.0fMB timeouts=%d crashes=%d" % (s, time.time() - t, oc, res["maxrss_kb"] / 1024, res["timeouts"], res["crashes"]))


if __name__ == "__main__":
    main()
''')

W('bench/judge.py', r'''"""Mathematical judge (runs in the controller env: SymPy used ONLY as a checker, never to produce a solver's answer).

Rules (declared in cases.json 'comparison' + here, applied identically to every solver):
  value        scalar equality: symbolic simplify(a-b)==0 OR high-precision numeric |a-b|<=tol  (numeric alone is flagged)
  expr_equiv   equivalence as functions: symbolic zero OR agreement at >=3 exact sample points (50-digit) [flagged if numeric only]
               + form check (below).  Complex/NaN at a real sample point is a MISMATCH (catches log(x) vs ln|x|).
  solution_set exact set equality over the case's declared domain (real|complex); multiplicity not part of correctness
  system_solution / inequality_set / matrix / gradient / ode_family / numeric_roots / antiderivative / undefined: see functions.
Form rules (uniform): simplify/evaluate -> structure of the RAW answer text must not be more than 1 operation more complex than the
  reference's raw structure (rejects echoing the input); named checks: expanded, factored, simplified_radical, no_fraction,
  reduced_fraction, partial_fractions.  The generic simplification-form rule was added after the first functional pass showed one solver
  echoing its input for E05; it is applied to all solvers, and results with/without form rules are both reported (sensitivity).
Domain rule: cases with required_restrictions need the solver to REPORT each restriction (constraint list or step text);
  strict_correct = value_correct AND form_ok AND domain_ok.  value-only correctness is reported alongside.
"""
import json
import re

import sympy as sp
from sympy.parsing.sympy_parser import convert_xor, parse_expr, standard_transformations

try:
    from .workers import safeinput
except ImportError:
    from workers import safeinput

TR = standard_transformations + (convert_xor,)
X, Y = sp.Symbol("x"), sp.Symbol("y")
XR, YR = sp.Symbol("x", real=True), sp.Symbol("y", real=True)
_ALLOWED = ["Integer", "Float", "Rational", "Symbol", "Function", "Mul", "Add", "Pow", "Derivative", "Eq", "sin", "cos", "tan", "exp", "log",
            "sqrt", "Abs", "pi", "E", "I", "oo", "zoo", "CRootOf", "Matrix", "Or", "And", "Not", "atan", "asin", "acos", "sinh", "cosh", "sign", "AccumBounds", "nan"]


def ns(extra=None):
    g = {k: getattr(sp, k) for k in _ALLOWED}
    g.update({"__builtins__": {}, "abs": sp.Abs, "ln": sp.log, "e": sp.E, "i": sp.I, "C": sp.Symbol("C"),
              "arctan": sp.atan, "arcsin": sp.asin, "arccos": sp.acos, "sgn": sp.sign})
    if extra:
        g.update(extra)
    return g


def P(text, evaluate=True, extra=None, syms=None):
    """Restricted parse of solver-produced text (allow-list; no builtins; no attribute access)."""
    safeinput.check(text.replace("\n", " "))
    loc = dict(syms or {})
    return parse_expr(text, local_dict=loc, global_dict=ns(extra), transformations=TR, evaluate=evaluate)


def strip_O(text):
    """Documented normalisation for series answers: drop the trailing truncation marker '+ O(...)' (every engine prints it last)."""
    import re as _re
    return _re.sub(r"\s*\+\s*O\(.*\)\s*$", "", text)


def N(e, n=50):
    return sp.N(e, n)


def ops_raw(text):
    try:
        return int(sp.count_ops(P(text, evaluate=False)))
    except Exception:   # noqa: BLE001
        return 10 ** 6


def strip_constant(e):
    """Remove an additive integration constant C/C1 (recorded)."""
    consts = [s for s in e.free_symbols if re.fullmatch(r"C\d*", s.name)]
    if consts:
        return e.subs({c: 0 for c in consts}), True
    return e, False


def sample_pts(case, var_names):
    sp_ = case.get("sample_points")
    if sp_:
        cols = [[sp.Rational(v) for v in sp_[n]] for n in var_names if n in sp_]
        if len(cols) == len(var_names):
            k = min(len(c) for c in cols)
            return [dict(zip(var_names, [c[i] for c in cols])) for i in range(k)]
    base = [sp.Rational(-37, 10), sp.Rational(-3, 5), sp.Rational(9, 10), sp.Rational(23, 10), sp.Rational(51, 10)]
    ycol = [sp.Rational(21, 10), sp.Rational(-7, 5), sp.Rational(1, 3), sp.Rational(17, 10), sp.Rational(-5, 2)]
    return [dict(zip(var_names, [base[i], ycol[i]][: len(var_names)])) for i in range(5)]


def equiv(a, b, case, tol=1e-9):
    """Returns (ok, proof) with proof in {'symbolic','numeric','mismatch'}."""
    names = sorted({s.name for s in (a.free_symbols | b.free_symbols) if s.name in ("x", "y")})
    sub = lambda e, pt: e.subs({sp.Symbol(k): v for k, v in pt.items()}).subs({sp.Symbol(k, real=True): v for k, v in pt.items()})
    pts = sample_pts(case, names) if names else [{}]
    numeric_ok = True
    for pt in pts:
        try:
            av, bv = N(sub(a, pt)), N(sub(b, pt))
        except Exception:   # noqa: BLE001
            return False, "mismatch"
        if not (av.is_number and bv.is_number) or av.has(sp.nan, sp.zoo, sp.oo, -sp.oo) or bv.has(sp.nan, sp.zoo, sp.oo, -sp.oo):
            return False, "mismatch"
        if abs(sp.im(av)) > tol or abs(sp.im(bv)) > tol:          # real input must give real output (cases are declared real)
            if case["domain"] == "real":
                return False, "mismatch"
        if abs(av - bv) > tol * max(1, abs(bv)):
            return False, "mismatch"
    try:
        if sp.simplify(a - b) == 0:
            return True, "symbolic"
    except Exception:   # noqa: BLE001
        pass
    return numeric_ok, "numeric"


def numeq(a, b, tol):
    try:
        return abs(N(a) - N(b)) <= max(tol["abs"], tol["rel"] * abs(N(b)))
    except Exception:   # noqa: BLE001
        return False


# --------------------------------------------------------------------------- form checks
def form_check(name, text, expected_text, case):
    """Returns (ok|None, note)."""
    try:
        if name is None and case["operation"] in ("simplify", "evaluate"):
            name = "generic_ops"
        if name is None:
            return None, ""
        if name == "generic_ops" or name in ("collected", "no_unexpanded_product"):
            a, b = ops_raw(text), ops_raw(expected_text)
            return a <= b + 1, "ops(raw)=%d vs reference %d (allowed +1)" % (a, b)
        e = P(text, syms={"x": X, "y": Y})
        if name == "simplified_radical":
            raw = P(text, evaluate=False)
            for pw in raw.atoms(sp.Pow):
                if pw.exp == sp.Rational(1, 2) and pw.base.is_Integer and any(k >= 2 for k in sp.factorint(pw.base).values()):
                    return False, "radical %s has a square factor" % pw
            return True, ""
        if name == "expanded":
            return e == sp.expand(e), "structurally expanded" if e == sp.expand(e) else "not expanded"
        if name == "closed_form":
            funcs = [a for a in e.atoms(sp.Function) if not isinstance(a, sp.core.function.AppliedUndef)]
            bad = any(pw.base.is_Add for pw in e.atoms(sp.Pow))
            ok = not funcs and not bad
            return ok, "fully evaluated closed form" if ok else "not evaluated: contains %s" % (funcs or "a power/quotient of a sum")
        if name == "factored_full":
            f_ = sp.factor(e)
            ok = (e == f_)
            return ok, "equals its own complete factorisation" if ok else "not completely factored (factor(e) = %s)" % f_
        if name == "factored":
            f = sp.factor(e)
            ok = isinstance(e, sp.Mul) and len(e.args) == len(f.args) and sp.simplify(f - e) == 0
            return ok, "product of irreducible factors" if ok else "not a fully factored product"
        if name == "no_fraction":
            n, d = sp.fraction(sp.together(e))
            return not d.has(X), "denominator has no x" if not d.has(X) else "still a fraction in x"
        if name == "reduced_fraction":
            n, d = sp.fraction(sp.together(e))
            return sp.gcd(n, d) == 1, "numerator/denominator coprime"
        if name == "partial_fractions":
            ok = isinstance(e, sp.Add) and len(e.args) >= 2 and all(sp.degree(sp.fraction(sp.together(t))[1], X) <= 1 for t in e.args)
            return ok, "sum of fractions with linear denominators" if ok else "not decomposed (not a sum of linear-denominator fractions)"
    except Exception as ex:   # noqa: BLE001
        return None, "form check could not run: %s" % type(ex).__name__
    return None, ""


# --------------------------------------------------------------------------- restrictions
def restriction_ok(required, reported_texts, step_texts):
    """required like 'x != 1'.  Satisfied if a reported constraint/step expresses the same exclusion."""
    m = re.fullmatch(r"(\w+)\s*!=\s*(.+)", required)
    v, c = m.group(1), sp.Rational(m.group(2))
    var = sp.Symbol(v)
    for t in reported_texts + step_texts:
        for part in re.split(r"[;,\n]| and ", t):
            mm = re.search(r"(.+?)\s*(?:!=|≠)\s*(.+)", part)
            if mm:
                try:
                    l, r = P(mm.group(1).strip().split("|")[-1].strip(), syms={v: var}), P(mm.group(2).strip().split("(")[0].strip() if "(" in mm.group(2) and not re.match(r"^[\w\s\+\-\*/\^\.]+$", mm.group(2).split("(")[0]) else mm.group(2).strip(), syms={v: var})
                    sols = sp.solve(l - r, var)
                    if sols and all(s == c for s in sols):
                        return True
                except Exception:   # noqa: BLE001
                    pass
            if re.search(r"\b%s\s*=\s*%s\b" % (re.escape(v), re.escape(str(c))), part) and re.search(r"denominator|undefined|excluded|reject|domain|zero", t, re.I):
                return True
    return False


# --------------------------------------------------------------------------- answer comparers
def _finite_set_equal(actual_vals, exp_vals, tol=1e-9):
    A, B = list(actual_vals), list(exp_vals)
    if len(A) != len(B):
        return False
    used = [False] * len(B)
    for a in A:
        hit = False
        for i, b in enumerate(B):
            if not used[i]:
                try:
                    if sp.simplify(a - b) == 0 or abs(N(a - b)) < tol:
                        used[i] = hit = True
                        break
                except Exception:   # noqa: BLE001
                    pass
        if not hit:
            return False
    return True


def judge(case, reply):
    """-> verdict dict.  'reply' is the preserved raw worker reply."""
    v = {"status": None, "sub": "", "value_correct": None, "form_ok": None, "domain_ok": None, "answer_produced": False,
         "covered": False, "strict_correct": False, "value_only_correct": False, "expected": case["spec_expected"], "actual": "",
         "proof": None, "manual_review": False, "manual_reasons": [], "form_note": "", "constant_shown": None, "detail": "",
         "domain_check": {"required": case["required_restrictions"], "reported": reply.get("restrictions_reported", []), "satisfied": None}}
    oc = reply.get("outcome")
    err = reply.get("error") or {}
    exp, cmp_ = case["expected"], case["comparison"]
    method = cmp_["method"]
    tol = case["tolerance"]

    def done(status, sub="", **kw):
        v.update(status=status, sub=sub, **kw)
        v["value_only_correct"] = bool(v["value_correct"]) and v["form_ok"] is not False
        v["strict_correct"] = bool(v["value_correct"]) and v["form_ok"] is not False and v["domain_ok"] is not False
        v["covered"] = v["answer_produced"] and v["form_ok"] is not False and not v.get("_not_in_form")
        v.pop("_not_in_form", None)
        return v

    if oc == "timeout":
        return done("timeout", "no reply within the per-case timeout")
    if oc == "infra_error":
        return done("infra_error", err.get("message", "")[:200])
    if oc == "unsupported":
        r = reply.get("unsupported_reason", "")
        sub = "no_api" if ("no public method" in r or "no mapping" in r or "documented for LINEAR" in r) else "unsolved_by_solver"
        v["actual"] = r[:200]
        # an engine that DECLARES the correct negative result in its refusal text has answered correctly (flagged for review)
        if sub == "unsolved_by_solver" and method in ("dne", "divergent", "expect_error"):
            pat = {"dne": r"does not exist|left hand limit|diverg", "divergent": r"diverg|does not converge|infinite", "expect_error": r"invertible|singular|det\s*==\s*0|determinant"}[method]
            if re.search(pat, r, re.I):
                v.update(answer_produced=True, value_correct=True, form_ok=True, manual_review=True)
                v["manual_reasons"].append("correct negative result stated in the engine's 'unsolved' message: " + r[:100])
                return done("valid", "negative_result_declared_in_unsolved_message")
        return done("unsupported", sub)
    if oc == "input_incompatible":
        v["actual"] = err.get("message", "")
        return done("input_incompatible", err.get("type", ""))
    if oc == "error":
        v["actual"] = "%s: %s" % (err.get("type"), err.get("message", "")[:120])
        pat_err = {"dne": r"does not exist|exist|oscillat|ambiguous|undefined|diverg", "divergent": r"diverg|infinite|does not converge|undefined",
                   "expect_error": r"singular|invertible|det\s*==\s*0|determinant|zero"}.get(method)
        if pat_err and re.search(pat_err, err.get("type", "") + " " + err.get("message", ""), re.I):
            v.update(answer_produced=True, value_correct=True, form_ok=True, manual_review=True)
            v["manual_reasons"].append("correct negative result signalled by an error: %s: %s" % (err.get("type"), err.get("message", "")[:80]))
            return done("valid", "error_signalled_negative_result")
        if method == "undefined" and re.search(r"zerodivision|division|undefined|zoo|domain", (err.get("type", "") + err.get("message", "")), re.I):
            v.update(answer_produced=True, value_correct=True, form_ok=True)
            v["manual_review"] = True
            v["manual_reasons"].append("L04 passed via raised division-by-zero error (no structured undefined result)")
            v["detail"] = "error signalled instead of a fabricated solution; contract_exception=%s" % err.get("contract_exception")
            return done("valid", "error_signalled_domain_error")
        return done("runtime_error", err.get("type", ""))

    # ------------- outcome == answer
    ans = reply.get("answer") or {}
    kind = ans.get("kind")
    v["answer_produced"] = True
    steps_text = " ".join((s.get("rule", "") + " " + s.get("description", "")) for s in reply.get("steps", []))
    try:
        # ---------------- negative-result families: does not exist / diverges
        if method in ("dne", "divergent") and kind in ("undefined", "divergent"):
            v.update(value_correct=True, form_ok=True, actual="%s: %s" % (kind, str(ans.get("reason", ""))[:100]))
            return done("valid", "explicit_%s_result" % kind)
        if method in ("dne", "divergent") and kind in ("expr", "numeric"):
            text = ans.get("text") or (ans.get("values") or [""])[0]
            v["actual"] = text
            e = P(text, syms={"x": X, "y": Y})
            inf_ = e in (sp.oo, -sp.oo, sp.zoo)
            osc = "AccumBounds" in text or e == sp.nan
            if method == "divergent":
                ok = inf_ or osc
                v.update(value_correct=ok, form_ok=True)
                return done("valid" if ok else "invalid_answer", "" if ok else "finite value returned for a divergent quantity")
            ok = osc
            v.update(value_correct=ok, form_ok=True)
            if ok:
                v["manual_review"] = True; v["manual_reasons"].append("DNE accepted via AccumBounds/nan (engine signals oscillation/undefined)")
            return done("valid" if ok else "invalid_answer", "" if ok else ("claims an infinite limit but the one-sided limits differ" if inf_ else "claims the limit exists (%s)" % text))
        if method == "series" and kind in ("expr", "numeric"):
            text = strip_O(ans.get("text") or "")
            v["actual"] = ans.get("text")
            e = P(text, syms={"x": X, "y": Y})
            ref = P(exp["expr"], syms={"x": X, "y": Y})
            ok, proof = equiv(e, ref, case)
            v.update(value_correct=ok, proof=proof, form_ok=True)
            return done("valid" if ok else "invalid_answer", "" if ok else "wrong_series")
        if method == "expect_error" and kind in ("undefined", "empty"):
            v.update(value_correct=True, form_ok=True, actual=kind)
            return done("valid", "explicit_%s_result" % kind)
        if method == "expect_error" and kind == "singular_matrix":
            determinant = ans.get("determinant")
            rank, size = ans.get("rank"), ans.get("size")
            structured_ok = (
                isinstance(determinant, str)
                and P(determinant, syms={"x": X, "y": Y}) == 0
                and isinstance(rank, int)
                and isinstance(size, int)
                and 0 <= rank < size
            )
            v["actual"] = "singular matrix: det=%s rank=%s/%s" % (determinant, rank, size)
            v.update(value_correct=structured_ok, form_ok=True)
            if structured_ok:
                return done("valid", "explicit_singular_matrix_result")
            return done("invalid_answer", "singular_matrix_result_missing_zero_determinant_or_rank_evidence")
        if method == "expect_error":
            v["actual"] = json.dumps(ans)[:150]
            v.update(value_correct=False, form_ok=True)
            return done("invalid_answer", "fabricated_result: returned a value where none exists")

        # ---------------- scalar / expression families
        if method in ("value", "expr_equiv", "antiderivative") and kind in ("expr", "numeric", "solutions", "equation"):
            text = ans.get("text") or (ans.get("values") or [""])[0]
            v["actual"] = text
            e = P(text, syms={"x": X, "y": Y})
            ref = P(exp["expr"], syms={"x": X, "y": Y})
            if method == "value" and ref in (sp.oo, -sp.oo):
                ok = (e == ref)
                v.update(value_correct=bool(ok), proof="symbolic" if ok else "mismatch")
            elif method == "value":
                ok = numeq(e, ref, tol)
                try:
                    proof = "symbolic" if sp.simplify(e - ref) == 0 else "numeric"
                except Exception:   # noqa: BLE001
                    proof = "numeric"
                v.update(value_correct=bool(ok), proof=proof if ok else "mismatch")
            elif method == "expr_equiv":
                ok, proof = equiv(e, ref, case)
                v.update(value_correct=ok, proof=proof)
                if ok and proof == "numeric":
                    v["manual_review"] = True; v["manual_reasons"].append("equivalence established numerically only (no symbolic zero)")
            else:   # antiderivative
                e0, shown = strip_constant(e)
                v["constant_shown"] = shown
                real_points = cmp_.get("require_real_on")
                real_ok = True
                if real_points:
                    # Evaluate the derivative in a real-symbol context.  A generic complex symbol
                    # leaves Derivative(re(x), x) / Derivative(Abs(x), x) unevaluated for
                    # log(Abs(u)) rules, which would reject a correct real antiderivative.
                    xr, yr = sp.Symbol("x", real=True), sp.Symbol("y", real=True)
                    eraw = P(text, syms={"x": xr, "y": yr})
                    er0, _ = strip_constant(eraw)
                    fr = P(case["problem"]["expr"], syms={"x": xr, "y": yr})
                    deriv_residual = sp.diff(er0, xr) - fr
                    sample_ok = True
                    for pt in real_points:
                        val = N(deriv_residual.subs(xr, sp.Rational(pt)), 40)
                        if (not val.is_number or val.has(sp.zoo, sp.nan, sp.oo, -sp.oo)
                                or abs(sp.im(val)) > 1e-12 or abs(sp.re(val)) > 1e-8):
                            sample_ok = False
                    ok, proof = sample_ok, "real-domain derivative samples" if sample_ok else "real-domain derivative mismatch"
                    for pt in real_points:
                        val = N(er0.subs(xr, sp.Rational(pt)))
                        if not val.is_number or val.has(sp.zoo, sp.nan, sp.oo, -sp.oo) or abs(sp.im(val)) > 1e-12:
                            real_ok = False
                    v["domain_ok"] = real_ok
                    v["detail"] = "F'==f at declared real-domain sample points: %s ; antiderivative real-valued at those points: %s" % (ok, real_ok)
                else:
                    d = sp.diff(e0, X)
                    f = P(case["problem"]["expr"], syms={"x": X, "y": Y})
                    ok, proof = equiv(d, f, {**case, "domain": "complex"})
                v.update(value_correct=bool(ok and real_ok), proof=proof)
            fo, note = form_check(cmp_.get("form"), text, exp["expr"], case)
            v["form_ok"], v["form_note"] = fo, note
            if case["required_restrictions"]:
                texts = list(reply.get("restrictions_reported", []))
                oks = [restriction_ok(r, texts, [steps_text]) for r in case["required_restrictions"]]
                v["domain_ok"] = all(oks)
                v["domain_check"]["satisfied"] = all(oks)
                v["manual_review"] = True; v["manual_reasons"].append("restriction-reporting decided by text matching")
            return done("valid" if v["value_correct"] and v["form_ok"] is not False else "invalid_answer",
                        "" if v["value_correct"] else ("domain_error" if v["domain_ok"] is False else "wrong_value"))

        # ---------------- equation solving
        if method == "solution_set":
            domain_cx = case["domain"] == "complex"
            ek = exp["kind"]
            if kind == "empty":
                v["actual"] = "no solution (empty set)"
                if ek == "empty":
                    v.update(value_correct=True, form_ok=True)
                elif ek == "finite" and domain_cx and re.search(r"no real|real number system|over the real|keep the real|real solutions|real values|in the reals|real domain", steps_text, re.I):
                    v.update(value_correct=False, form_ok=True)
                    return done("invalid_answer", "real_domain_only (states no real solutions; complex solutions missing)")
                else:
                    v.update(value_correct=False, form_ok=True)
                    return done("invalid_answer", "claims no solution but solutions exist" if ek != "empty" else "")
                return done("valid")
            if kind == "all":
                v["actual"] = "all %s" % ans.get("domain", "x")
                v.update(value_correct=(ek == "all"), form_ok=True)
                return done("valid" if ek == "all" else "invalid_answer", "" if ek == "all" else "claims every x is a solution")
            if kind == "solutions":
                vals = [P(t, syms={"x": X}) for t in ans["values"]]
                v["actual"] = "{" + ", ".join(ans["values"]) + "}"
                if ek == "finite":
                    ref = [P(t, syms={"x": X}) for t in exp["values"]]
                    ok = _finite_set_equal(vals, ref)
                    sub = ""
                    if not ok and domain_cx and len(vals) < len(ref) and all(any(_finite_set_equal([a], [b]) for b in ref) for a in vals):
                        sub = "real_domain_only (complex roots missing)"
                    v.update(value_correct=ok, form_ok=True)
                    return done("valid" if ok else "invalid_answer", sub or ("" if ok else "wrong_solution_set"))
                v.update(value_correct=False, form_ok=True)
                return done("invalid_answer", "returned solutions where %s expected" % ek)
            v["actual"] = json.dumps(ans)[:200]
            v["_not_in_form"] = True
            return done("invalid_answer", "unexpected answer form %r" % kind)

        if method == "system_solution":
            ek = exp["kind"]
            if kind == "empty":
                v.update(value_correct=(ek == "empty"), form_ok=True, actual="no solution")
                return done("valid" if ek == "empty" else "invalid_answer", "" if ek == "empty" else "claims no solution")
            if kind == "all":
                v.update(value_correct=False, form_ok=True, actual="all values")
                return done("invalid_answer", "claims every point is a solution")
            if kind == "system" and ek == "multi":
                sols = ans["solutions"]
                v["actual"] = json.dumps(sols)
                exp_l = exp["assignments"]
                used = [False] * len(exp_l)
                ok = len(sols) == len(exp_l)
                if ok:
                    for sol in sols:
                        hit = False
                        for i, ex in enumerate(exp_l):
                            if not used[i] and set(sol) == set(ex) and all(sp.simplify(P(sol[k_]) - sp.Rational(ex[k_])) == 0 for k_ in ex):
                                used[i] = hit = True
                                break
                        ok = ok and hit
                v.update(value_correct=bool(ok), form_ok=True)
                return done("valid" if ok else "invalid_answer", "" if ok else "wrong_or_incomplete_solution_set (%d of %d solutions)" % (len(sols), len(exp_l)))
            if kind == "infinite":
                v["actual"] = "infinitely many solutions (stated)"
                ok = ek == "line"
                v.update(value_correct=ok, form_ok=True)
                if ok:
                    v["manual_review"] = True; v["manual_reasons"].append("accepted: engine states infinitely many solutions without giving the parametric family")
                return done("valid" if ok else "invalid_answer", "" if ok else "claims infinitely many solutions")
            if kind == "system" and ek == "line":
                sols = ans["solutions"]
                v["actual"] = json.dumps(sols)
                free = any(P(t).free_symbols for sol in sols for t in sol.values())
                ok = bool(sols) and free
                if ok:
                    for sol in sols:
                        sub_ = {sp.Symbol(k_): P(t, syms={"x": X, "y": Y}) for k_, t in sol.items()}
                        for eq in case["problem"]["equations"]:
                            l_, r_ = [P(t, syms={"x": X, "y": Y, "z": sp.Symbol("z")}) for t in eq.split("=", 1)]
                            ok = ok and sp.simplify((l_ - r_).subs(sub_)) == 0
                v.update(value_correct=bool(ok), form_ok=True)
                return done("valid" if ok else "invalid_answer", "" if ok else "not a parametric family satisfying the equations (unique point returned for a dependent system)")
            if kind == "system":
                sols = ans["solutions"]
                v["actual"] = json.dumps(sols)
                if ek == "unique" and len(sols) == 1:
                    ok = all(k in sols[0] and sp.simplify(P(sols[0][k]) - sp.Rational(val)) == 0 for k, val in exp["assignments"].items()) and set(sols[0]) == set(exp["assignments"])
                    v.update(value_correct=ok, form_ok=True)
                    return done("valid" if ok else "invalid_answer", "" if ok else "wrong_solution")
                v.update(value_correct=False, form_ok=True)
                return done("invalid_answer", "returned %d solution(s) where %s expected" % (len(sols), ek))
            v["actual"] = json.dumps(ans)[:200]; v["_not_in_form"] = True
            return done("invalid_answer", "unexpected answer form %r" % kind)

        if method == "inequality_set":
            exp_set = sp.Union(*[sp.Interval(sp.sympify(a.replace("oo", "oo")), sp.sympify(b), l, r) for a, b, l, r in exp["intervals"]])
            if kind == "empty" or (kind == "bool" and ans.get("value") is False):
                act = sp.S.EmptySet
                v["actual"] = "empty set"
            elif kind == "all" or (kind == "bool" and ans.get("value") is True):
                act = sp.S.Reals
                v["actual"] = "all real numbers"
            elif kind == "intervals":
                act = sp.Union(*[sp.Interval(sp.sympify(a), sp.sympify(b), l, r) for a, b, l, r in ans["intervals"]])
                v["actual"] = str(act)
            elif kind == "relation" and ans["text"].strip() in ("False", "True"):
                act = sp.S.EmptySet if ans["text"].strip() == "False" else sp.S.Reals
                v["actual"] = ans["text"]
            elif kind == "relation":
                rel = P(ans["text"], syms={"x": XR})
                act = rel.as_set()
                v["actual"] = ans["text"]
            else:
                v["actual"] = json.dumps(ans)[:200]; v["_not_in_form"] = True
                return done("invalid_answer", "unexpected answer form %r" % kind)
            ok = act == exp_set or act.symmetric_difference(exp_set) == sp.S.EmptySet
            v.update(value_correct=bool(ok), form_ok=True)
            return done("valid" if ok else "invalid_answer", "" if ok else "wrong_interval_set")

        if method == "matrix":
            if kind != "matrix":
                v["actual"] = json.dumps(ans)[:200]; v["_not_in_form"] = True
                return done("invalid_answer", "unexpected answer form %r" % kind)
            A = sp.Matrix([[P(c) for c in r] for r in ans["rows"]])
            B = sp.Matrix([[sp.Rational(c) for c in r] for r in exp["matrix"]])
            v["actual"] = str(A.tolist())
            ok = A.shape == B.shape and (A - B).applyfunc(sp.simplify) == sp.zeros(*B.shape)
            v.update(value_correct=bool(ok), form_ok=True)
            return done("valid" if ok else "invalid_answer", "" if ok else "wrong_matrix")

        if method == "gradient":
            if kind != "vector":
                v["actual"] = json.dumps(ans)[:200]; v["_not_in_form"] = True
                return done("invalid_answer", "unexpected answer form %r" % kind)
            v["actual"] = "(" + ", ".join(ans["items"]) + ")"
            ok = len(ans["items"]) == len(exp["vector"]) and all(equiv(P(a, syms={"x": X, "y": Y}), P(b, syms={"x": X, "y": Y}), case)[0] for a, b in zip(ans["items"], exp["vector"]))
            v.update(value_correct=bool(ok), form_ok=True)
            return done("valid" if ok else "invalid_answer", "" if ok else "wrong_gradient")

        if method == "ode_family":
            if kind != "equation":
                v["actual"] = json.dumps(ans)[:200]; v["_not_in_form"] = True
                return done("invalid_answer", "unexpected answer form %r" % kind)
            f = sp.Function("y")
            eq = P(ans["text"], syms={"x": X}, extra={"y": f})
            v["actual"] = ans["text"]
            rhs = eq.rhs
            ode = P(case["problem"]["ode"], syms={"x": X}, extra={"y": f})
            consts = sorted([s for s in rhs.free_symbols if re.fullmatch(r"C\d*", s.name)], key=lambda s: s.name)
            resid = sp.simplify(ode.subs(f(X), rhs).doit())
            ok_ode = resid == 0
            ok_n = len(consts) == exp["n_constants"]
            basis = [sp.diff(rhs, c) for c in consts]
            indep = (basis[0] != 0) if len(basis) == 1 else (sp.simplify(sp.wronskian(basis, X)) != 0 if len(basis) >= 2 else False)
            ok = ok_ode and ok_n and indep
            v["detail"] = "ODE residual after substitution: %s ; constants=%d ; independent basis: %s" % (resid, len(consts), indep)
            v.update(value_correct=bool(ok), form_ok=True)
            return done("valid" if ok else "invalid_answer", "" if ok else "not a general solution")

        if method == "numeric_roots":
            vals = ans.get("values", [])
            if kind == "empty":
                v.update(value_correct=False, form_ok=True, actual="no solution")
                return done("invalid_answer", "claims no solution but a real root exists")
            if kind not in ("numeric", "solutions"):
                v["actual"] = json.dumps(ans)[:200]; v["_not_in_form"] = True
                return done("invalid_answer", "unexpected answer form %r" % kind)
            nums = []
            for t in vals:
                try:
                    nums.append(N(P(t, syms={"x": X}), 30))
                except Exception:   # noqa: BLE001
                    nums.append(None)
            v["actual"] = "[" + ", ".join(vals) + "]"
            ref = [sp.Float(t, 30) for t in exp["values"]]
            ok = all(any(n is not None and n.is_number and abs(sp.im(n)) < 1e-9 and abs(sp.re(n) - r) <= tol["abs"] for n in nums) for r in ref)
            spec_val = exp.get("spec_printed_value")
            if spec_val:
                v["detail"] = "matches spec-printed value %s within 1e-6: %s" % (spec_val, any(n is not None and abs(sp.im(n)) < 1e-9 and abs(sp.re(n) - sp.Float(spec_val, 30)) <= 1e-6 for n in nums))
            v.update(value_correct=bool(ok), form_ok=True)
            if any("CRootOf" in t for t in vals):
                v["manual_review"] = True; v["manual_reasons"].append("exact CRootOf accepted via numeric evaluation (declared rule)")
            return done("valid" if ok else "invalid_answer", "" if ok else "no root within tolerance")

        if method == "undefined":
            if kind == "undefined":
                v.update(value_correct=True, form_ok=True, actual="undefined: " + str(ans.get("reason", ""))[:120])
                return done("valid", "explicit_undefined_result")
            if kind == "empty":
                v.update(value_correct=False, form_ok=False, actual="no solution (empty set) - undefined not flagged")
                v["manual_review"] = True; v["manual_reasons"].append("L04 partial: empty set without undefined flag")
                return done("invalid_answer", "partial: empty solution set, expression not flagged undefined; no fabricated solution")
            v.update(value_correct=False, form_ok=False, actual=json.dumps(ans)[:150])
            return done("invalid_answer", "fabricated_solution")
    except Exception as ex:   # noqa: BLE001
        v["manual_review"] = True
        v["manual_reasons"].append("judge could not evaluate the output: %s: %s" % (type(ex).__name__, str(ex)[:100]))
        v["actual"] = v["actual"] or json.dumps(ans)[:200]
        v.update(value_correct=None, _not_in_form=True)
        return done("invalid_answer", "unjudgeable_output")
    v["manual_review"] = True; v["manual_reasons"].append("no comparer matched answer kind %r for method %r" % (kind, method))
    v["actual"] = json.dumps(ans)[:200]; v["_not_in_form"] = True
    return done("invalid_answer", "unexpected answer form %r" % kind)
''')

W('bench/derivations.py', r'''"""Derivation-quality scoring (rubric: correctness 0-3, completeness 0-2, explanation 0-2, consistency 0-1, domain 0-1, presentation 0-1).

Automated layer (deterministic, auditable; thresholds below are declared before results were inspected):
  correctness   0 final answer not (strictly) correct | 1 final correct but an automated check found an invalid step or the
                conclusion contradicts the final answer | 2 final correct, no invalid step, but the worked steps do not demonstrably
                produce the answer (solver flagged 'direct_evaluation' override, or final answer absent from every step) | 3 clean.
  completeness  share of the case's reference milestones (regex over step text) found: >=0.75 -> 2, >=0.34 -> 1, else 0; a derivation
                that is only opaque/structural steps scores 0; fewer than MIN_STEPS[op] substantive steps caps the score at 1.
  explanation   per substantive step: 2 = specific rule label + >=8 prose words ; 1 = specific rule label OR >=4 prose words ; 0 = opaque
                ('Compute exact result', 'Direct evaluation', ...).  mean>=1.5 -> 2 ; >=0.6 -> 1 ; else 0.
  consistency   1 if no invalid transition, no override, and the final answer appears in the workings; else 0.
  domain        domain-sensitive case: 1 iff the answer's domain handling is correct AND the workings mention the restriction/rejection
                (constraint list or domain vocabulary); else 0.  Non-sensitive case: 1 unless a domain error was detected (N/A -> full;
                reported separately as 'domain-sensitive-only mean').
  presentation  1 iff >=80% of steps have readable expression text free of Python-object artefacts (<...>, Symbol(, ...Rule( trees).
These are PROXIES.  A manual-review overlay (data/manual_reviews.json) records per-criterion adjustments with confidence and
justification; every row keeps both the automated and the final score.  The reviewer is a language model (Claude), so the overlay is
flagged 'model_review' and is NOT the authority on mathematical correctness (that comes from judge.py's symbolic/numeric checks).
"""
import json
import math
import re

import sympy as sp

try:
    from . import judge as J
except ImportError:
    import judge as J

MIN_STEPS = {"evaluate": 1, "simplify": 1, "expand": 1, "factor": 1, "solve": 2, "nsolve": 2, "solve_system": 3, "solve_inequality": 2,
             "diff": 2, "partial_derivative": 2, "gradient": 2, "integrate": 2, "limit": 2, "det": 2, "inverse": 2, "rref": 2,
             "dsolve": 2, "partial_fractions": 3}
OPAQUE = {"compute exact result", "direct evaluation", "general solving"}
GENERIC_RULES = {"apply algebraic identities"}      # boilerplate label reused across unrelated problems: not a specific rule identification
STRUCTURAL = {"setup", "final result", "limit setup", "partial derivative setup", "gradient setup", "augmented matrix",
              "identify linear equation", "identify quadratic equation", "identify polynomial", "identify rational form",
              "reduced row echelon form"}
DOMAIN_WORDS = re.compile(r"domain|restrict|undefined|denominator|extraneous|reject|!=|≠|no real|real number|nonzero|non-zero|positive|"
                          r"excluded|must not|cannot be zero|inconsistent|identity|contradict|absolute value|\|x\|", re.I)
ARTEFACT = re.compile(r"<[^>]*object|Symbol\(|Rule\(|\bRule\b.*integrand=|_sympy|\bzoo\b")


def prose_words(text):
    if ARTEFACT.search(text) or "integrand=" in text:      # raw object/rule-tree dumps are not prose
        return 0
    t = re.sub(r"[-+*/^=()\[\]{}<>|,.:;→←∫∂Δ∇]", " ", text)
    return len([w for w in t.split() if re.fullmatch(r"[A-Za-z']{3,}", w) and w.lower() not in ("sin", "cos", "tan", "log", "exp", "sqrt", "abs", "dx", "matrix")])


def _flat(steps):
    out = []
    for s in steps:
        out.append(" ".join([s.get("rule", ""), s.get("description", ""), s.get("before", ""), s.get("after", "")] + [" ".join(n) for n in s.get("notes", [])]))
        for sub in s.get("substeps", []) or []:
            out.append(json.dumps(sub))
    return " \n".join(out)


def _expr_or_none(t):
    try:
        if not t or any(tok in t for tok in ("Eq(", "[", "{", "<", ">", "diff(", "integrate(", "limit(", "matrix", "solve", "|")):
            return None
        return J.P(t, syms={"x": J.X, "y": J.Y})
    except Exception:   # noqa: BLE001
        return None


def check_steps(case, steps, final_ok):
    """-> (n_checked, n_invalid, notes).  Only algebra families are checked transition-by-transition."""
    op = case["operation"]
    checked = invalid = 0
    notes = []
    if op in ("evaluate", "simplify", "expand", "factor"):
        prev = _expr_or_none(case["problem"]["expr"].replace("^", "**"))
        for i, s in enumerate(steps):
            for key in ("before", "after"):
                e = _expr_or_none(s.get(key, "").replace("^", "**"))
                if e is None or prev is None:
                    continue
                if key == "before":
                    # StepSolver gives before/after pairs; 'before' must equal the previous 'after' (or input)
                    ok, _ = J.equiv(e, prev, {**case, "domain": "complex"})
                else:
                    ok, _ = J.equiv(e, prev, {**case, "domain": "complex"})
                    prev = e
                checked += 1
                if not ok:
                    invalid += 1
                    notes.append("step %d (%s): '%s' not equivalent to previous expression" % (i + 1, key, s.get(key)))
    elif op in ("solve", "nsolve"):
        truth = [J.P(t) for t in case["expected"].get("values", [])] if case["expected"].get("kind") == "finite" else None
        for i, s in enumerate(steps):
            m = re.match(r"^\s*Eq\(\s*x\s*,\s*(.+)\)\s*$", s.get("after", ""))
            if m and truth is not None:
                try:
                    val = J.P(m.group(1).replace("^", "**"), syms={"x": J.X})
                except Exception:   # noqa: BLE001
                    continue
                checked += 1
                if not any(sp.simplify(val - t) == 0 for t in truth):
                    # a step may legitimately show a REJECTED candidate (extraneous root); only flag if the text does not say so
                    if not re.search(r"reject|extraneous|does not satisfy|not a solution", s.get("description", "") + s.get("rule", ""), re.I):
                        invalid += 1
                        notes.append("step %d claims x = %s which is not a true solution" % (i + 1, m.group(1)))
    return checked, invalid, notes


def automated_score(case, reply, verdict):
    steps = [s for s in reply.get("steps", []) if s]
    rec = {"available": bool(steps), "n_steps": len(steps), "scored": False}
    if not steps or reply.get("outcome") != "answer":
        return rec
    flat = _flat(steps)
    nf = reply.get("native_flags") or {}
    rules = [s.get("rule", "").strip().lower() for s in steps]
    substantive = [s for s, r in zip(steps, rules) if r not in OPAQUE and r not in STRUCTURAL and not r.startswith("identify")]
    final_ok = bool(verdict.get("value_correct")) and verdict.get("form_ok") is not False
    n_chk, n_inv, inv_notes = check_steps(case, steps, final_ok)

    # --- correctness
    override = nf.get("steps_mode") == "direct_evaluation" or nf.get("workings_tied_to_answer") is False
    if not final_ok:
        c = 0
        why_c = "final answer not correct"
    elif n_inv:
        c, why_c = 1, "invalid step(s): " + "; ".join(inv_notes[:2])
    else:
        ans_text = json.dumps(reply.get("answer"))
        present = True
        # final answer present in workings? (value tokens of the answer must occur in some step text)
        toks = re.findall(r"[A-Za-z0-9_./^*+-]+", re.sub(r"[\[\]{}\"']", " ", ans_text))
        toks = [t for t in toks if len(t) > 0 and t not in ("kind", "expr", "text", "values", "solutions", "empty", "rows", "items", "system", "relation", "equation", "mapping", "entries")]
        if toks:
            norm = lambda t: re.sub(r"\s+", "", t.replace("**", "^"))
            present = any(norm(t) in norm(flat) for t in toks)
        if override:
            c, why_c = 2, "workings do not demonstrably produce the answer (steps_mode=direct_evaluation override, or workings from a code path separate from the answer, e.g. SymPy integral_steps vs integrate)"
        elif not present:
            c, why_c = 2, "final answer does not appear in the workings"
        else:
            c, why_c = 3, "no invalid step found (%d transitions auto-checked)" % n_chk

    # --- contradiction detection for integrals G04-like (derivation says ln|x| but answer is log(x))
    if case["id"] == "G04" and final_ok is False and re.search(r"ln\|x\||log\(abs", flat):
        why_c += "; workings state ln|x| but the returned antiderivative is not real-valued for x<0 (contradiction)"

    # --- completeness
    ms = case.get("reference_milestones") or []
    hits = sum(1 for m in ms if re.search(m, flat, re.I))
    frac = (hits / len(ms)) if ms else 1.0
    comp = 2 if frac >= 0.75 else 1 if frac >= 0.34 else 0
    if not substantive:
        comp = 0
    elif len(substantive) < MIN_STEPS.get(case["operation"], 1):
        comp = min(comp, 1)

    # --- explanation
    per = []
    for s in substantive:
        spec = s.get("rule", "").strip().lower() not in OPAQUE | GENERIC_RULES and bool(s.get("rule", "").strip())
        w = prose_words(s.get("description", ""))
        per.append(2 if (spec and w >= 8) else 1 if (spec or w >= 4) else 0)
    m_exp = (sum(per) / len(per)) if per else 0.0
    expl = 2 if m_exp >= 1.5 else 1 if m_exp >= 0.6 else 0

    # --- consistency
    cons = 1 if (n_inv == 0 and not override and c >= 2) else 0

    # --- domain
    if case["domain_sensitive"]:
        mentions = bool(reply.get("restrictions_reported")) or bool(DOMAIN_WORDS.search(flat))
        dom = 1 if (verdict.get("domain_ok") is not False and final_ok and mentions) else 0
    else:
        dom = 1 if verdict.get("domain_ok") is not False else 0

    # --- presentation
    readable = [s for s in steps if (s.get("after") or s.get("description")) and not ARTEFACT.search(s.get("after", "") + s.get("description", ""))]
    pres = 1 if len(readable) >= 0.8 * len(steps) else 0

    if not substantive:
        # Rubric rule 5/9 refinement (declared after the first inspection, applied uniformly to every solver):
        # a derivation with no substantive step shows no transformations -> correctness<=2, nothing to follow (consistency 0), nothing readable (presentation 0).
        c, cons, pres = min(c, 2), 0, 0
        why_c += "; opaque/structural steps only (no transformation shown)"
    rec.update(scored=True, automated={"correctness": c, "completeness": comp, "explanation": expl, "consistency": cons, "domain": dom,
                                       "presentation": pres}, automated_total=c + comp + expl + cons + dom + pres,
               evidence={"correctness": why_c, "milestones_hit": "%d/%d" % (hits, len(ms)), "mean_explanation_per_step": round(m_exp, 2),
                         "substantive_steps": len(substantive), "steps_auto_checked": n_chk, "invalid_steps": n_inv,
                         "opaque_steps": sum(1 for r in rules if r in OPAQUE), "native_flags": {k: nf.get(k) for k in ("steps_mode", "steps_verified", "workings_tied_to_answer")}})
    return rec


CRITERIA = ["correctness", "completeness", "explanation", "consistency", "domain", "presentation"]
CAPS = dict(correctness=3, completeness=2, explanation=2, consistency=1, domain=1, presentation=1)


def apply_manual(rec, review):
    """review: {'adjust': {criterion: new_value}, 'confidence': 'low|medium|high', 'justification': str}"""
    if not rec.get("scored"):
        return rec
    final = dict(rec["automated"])
    if review:
        for k, v in review.get("adjust", {}).items():
            assert k in CAPS and 0 <= v <= CAPS[k], (k, v)
            final[k] = v
        rec["manual"] = {"reviewer": "model_review (Claude) - requires human audit", "confidence": review["confidence"],
                         "justification": review["justification"], "adjusted": review.get("adjust", {})}
    rec["final"] = final
    rec["final_total"] = sum(final.values())
    return rec
''')

W('bench/reference_validation.py', r'''#!/usr/bin/env python3
"""Independently validate every reference answer of the HARD suite with stdlib (Fraction, cmath, math) and mpmath numerics.
No SymPy symbolic engine is used, so these checks share no code with any engine under test or with the judge."""
import json
import math
import pathlib
import random
import sys
from fractions import Fraction as Fr

import mpmath as mp

mp.mp.dps = 50
ROOT = pathlib.Path(__file__).resolve().parent.parent
RNG = random.Random(20260301)
R = {}
EPS = mp.mpf(10) ** -30


def rec(cid, ok, methods, detail=""):
    R[cid] = {"ok": bool(ok), "methods": methods, "detail": str(detail)}


def pts(n=8, avoid=(), lo=-9, hi=9):
    out = []
    while len(out) < n:
        p = Fr(RNG.randint(lo * 7, hi * 7), RNG.randint(2, 7))
        if p not in avoid and p not in out:
            out.append(p)
    return out


def ident(cid, f, g, avoid=(), n=12, note=""):
    rec(cid, all(f(p) == g(p) for p in pts(n, avoid)), ["exact rational identity test at %d random points" % n], note)


def undefined_at(f, *args):
    try:
        f(*args)
        return False
    except ZeroDivisionError:
        return True


def scan(pred, lo=-40, hi=40, den=4):
    return [Fr(k, den) for k in range(lo * den, hi * den + 1) if pred(Fr(k, den))]


# ------------------------------------------------------------------------------------------------ M
ident("M01", lambda x: x ** 4 - 5 * x ** 2 + 4, lambda x: (x - 2) * (x - 1) * (x + 1) * (x + 2))
ident("M02", lambda x: x ** 3 - 3 * x ** 2 + 3 * x - 1, lambda x: (x - 1) ** 3)
pp = [(a, b) for a in pts(5) for b in pts(4)]
rec("M03", all((a + b) ** 5 == a ** 5 + 5 * a ** 4 * b + 10 * a ** 3 * b ** 2 + 10 * a ** 2 * b ** 3 + 5 * a * b ** 4 + b ** 5 for a, b in pp),
    ["exact rational identity test at 20 (x,y) points"])
f4 = lambda x: (x ** 3 - 8) / (x ** 2 - 4)
rec("M04", all(f4(p) == (p * p + 2 * p + 4) / (p + 2) for p in pts(12, (Fr(2), Fr(-2)))) and undefined_at(f4, Fr(2)) and undefined_at(f4, Fr(-2)),
    ["exact identity at 12 points", "original undefined at x=2 and x=-2 (ZeroDivisionError)"])
pp = [(a, b) for a in pts(5, (Fr(0),)) for b in pts(4, (Fr(0),)) if a != b]
rec("M05", all(((1 / a + 1 / b) / (1 / a - 1 / b)) == (a + b) / (b - a) for a, b in pp), ["exact identity at %d (x,y) points (x,y != 0, x != y)" % len(pp)],
    "original undefined at x=0, y=0 and x=y")
rec("M06", 50 == 25 * 2 and 18 == 9 * 2 and 8 == 4 * 2 and 5 + 3 - 2 == 6 and abs(mp.sqrt(50) + mp.sqrt(18) - mp.sqrt(8) - 6 * mp.sqrt(2)) < EPS,
    ["integer decomposition 5+3-2=6", "mpmath 50 digits"])
rec("M07", abs((mp.sqrt(3) + mp.sqrt(2)) * (mp.sqrt(3) - mp.sqrt(2)) - 1) < EPS and 3 - 2 == 1, ["mpmath 50 digits", "difference of squares 3-2"])
rec("M08", abs(mp.sqrt(7 + 4 * mp.sqrt(3)) - (2 + mp.sqrt(3))) < EPS and (2 * 2 + 3) == 7 and 2 * 2 * 1 == 4, ["mpmath 50 digits", "(2+sqrt3)^2 = 4+3+4sqrt3 = 7+4sqrt3"])

# ------------------------------------------------------------------------------------------------ N
rec("N01", scan(lambda x: abs(2 * x - 3) == 5) == [Fr(-1), Fr(4)], ["exhaustive exact scan of 321 rationals", "substitution"])
f = lambda x: 1 / (x - 1) + 1 / (x + 1) - 2 / (x * x - 1)
sols = []
for x in scan(lambda x: x not in (1, -1)):
    if f(x) == 0:
        sols.append(x)
rec("N02", sols == [] and undefined_at(f, Fr(1)) and undefined_at(f, Fr(-1)),
    ["exhaustive exact scan of 321 rationals (x != +-1): no solution", "x=1 (the only algebraic candidate) makes the original undefined"], "2x/(x^2-1) = 2/(x^2-1) -> x = 1 (excluded)")
cand = [x for x in scan(lambda x: 2 * x + 3 >= 0, -12, 12, 8) if x >= 0 and x * x == 2 * x + 3]
rec("N03", cand == [Fr(3)] and math.sqrt(2 * -1 + 3) != -1, ["exact scan: sqrt(2x+3)=x requires x>=0 and x^2=2x+3 -> only 3", "x=-1 is extraneous (sqrt(1)=1 != -1)"])
g4 = lambda x: mp.sqrt(x + 5) + mp.sqrt(x)
rec("N04", abs(g4(4) - 5) < EPS and abs(mp.findroot(lambda x: g4(x) - 5, 3) - 4) < EPS and all(g4(a) < g4(b) for a, b in [(1, 2), (2, 7), (0, 0.5)]),
    ["substitution 3+2=5", "mpmath.findroot", "strictly increasing -> unique root"])
g5 = lambda x: mp.log(x) + mp.log(x - 3) - mp.log(4)
rec("N05", abs(g5(4)) < EPS and abs(mp.findroot(g5, 5) - 4) < EPS and (-1) - 3 < 0, ["substitution", "mpmath.findroot on the domain x>3", "x=-1 violates x>3 (log of negative)"])
x6 = mp.log(7) / mp.log(2)
rec("N06", abs(2 ** x6 - 7) < EPS and abs(mp.findroot(lambda t: 2 ** t - 7, 3) - x6) < EPS, ["mpmath substitution", "mpmath.findroot (2^x strictly increasing -> unique)"])
rec("N07", all(abs(mp.exp(2 * x) - 5 * mp.exp(x) + 6) < EPS for x in (mp.log(2), mp.log(3))) and sorted(mp.polyroots([1, -5, 6])) == [2, 3] or
    all(abs(a - b) < EPS for a, b in zip(sorted(mp.polyroots([1, -5, 6]), key=lambda z: z.real), [2, 3])),
    ["mpmath substitution", "quadratic in t=e^x has roots 2,3 (both > 0)"])
rts = sorted(mp.polyroots([1, 0, -5, 0, 4]), key=lambda z: z.real)
rec("N08", all(x ** 4 - 5 * x ** 2 + 4 == 0 for x in (-2, -1, 1, 2)) and all(abs(a - b) < EPS for a, b in zip(rts, [-2, -1, 1, 2])), ["substitution", "mpmath.polyroots"])
rts = sorted(mp.polyroots([1, -6, 11, -6]), key=lambda z: z.real)
rec("N09", all(x ** 3 - 6 * x ** 2 + 11 * x - 6 == 0 for x in (1, 2, 3)) and all(abs(a - b) < EPS for a, b in zip(rts, [1, 2, 3])), ["substitution", "mpmath.polyroots"])
rts = mp.polyroots([1, 0, 0, 0, 4], maxsteps=200, extraprec=100)
exp10 = [1 + 1j, 1 - 1j, -1 + 1j, -1 - 1j]
rec("N10", all(min(abs(r - e) for e in exp10) < EPS for r in rts) and all(abs(e ** 4 + 4) < EPS for e in exp10), ["complex substitution", "mpmath.polyroots (4 distinct roots)"])
rec("N11", all(math.sqrt(x) >= 0 for x in [k / 4 for k in range(0, 400)]), ["sqrt is non-negative on its whole real domain, so sqrt(x) = -2 is impossible (400-point check)"])

# ------------------------------------------------------------------------------------------------ O
rec("O01", all(a * a + b * b == 25 and a + b == 7 for a, b in [(3, 4), (4, 3)]) and sorted(mp.polyroots([1, -7, 12])) == [3, 4] or
    all(abs(a - b) < EPS for a, b in zip(sorted(mp.polyroots([1, -7, 12]), key=lambda z: z.real), [3, 4])),
    ["substitution", "reduction: x^2-7x+12=0 has exactly two roots (mpmath.polyroots)"])
rts = sorted(mp.polyroots([1, -1, -12]), key=lambda z: z.real)
rec("O02", all(a * b == 12 and a - b == 1 for a, b in [(4, 3), (-3, -4)]) and all(abs(a - b) < EPS for a, b in zip(rts, [-3, 4])), ["substitution", "reduction: x^2-x-12=0 roots (mpmath.polyroots)"])


def det3(m):
    return (m[0][0] * (m[1][1] * m[2][2] - m[1][2] * m[2][1]) - m[0][1] * (m[1][0] * m[2][2] - m[1][2] * m[2][0]) + m[0][2] * (m[1][0] * m[2][1] - m[1][1] * m[2][0]))


A3 = [[Fr(1), Fr(1), Fr(1)], [Fr(0), Fr(2), Fr(5)], [Fr(2), Fr(5), Fr(-1)]]
b3 = [Fr(6), Fr(-4), Fr(27)]
d = det3(A3)
cr = []
for j in range(3):
    M = [row[:] for row in A3]
    for i in range(3):
        M[i][j] = b3[i]
    cr.append(det3(M) / d)
rec("O03", cr == [5, 3, -2] and all(sum(A3[i][j] * [5, 3, -2][j] for j in range(3)) == b3[i] for i in range(3)), ["Cramer's rule (Fractions)", "substitution"])
rec("O04", all(p + (2 - p) == 2 and 2 * p + 2 * (2 - p) == 4 for p in pts(10)) and Fr(1) * 2 - Fr(1) * 2 == 0 and 2 * 2 == 4,
    ["det = 0, rows proportional and RHS consistent (rank 1 < 2)", "10 points on the line x+y=2 satisfy both equations"])


def member(x, ivs):
    for lo, hi, olo, ohi in ivs:
        ok_lo = lo is None or (x > lo if olo else x >= lo)
        ok_hi = hi is None or (x < hi if ohi else x <= hi)
        if ok_lo and ok_hi:
            return True
    return False


grid = [Fr(k, 4) for k in range(-80, 81)]
def pred5(x):
    return False if x == 2 else (x + 1) / (x - 2) >= 0
rec("O05", all(pred5(g) == member(g, [(None, Fr(-1), True, False), (Fr(2), None, True, True)]) for g in grid), ["exhaustive truth table on 161 grid points incl. boundaries x=-1 (closed) and x=2 (excluded)"])
rec("O06", all((abs(g - 1) < 3) == member(g, [(Fr(-2), Fr(4), True, True)]) for g in grid), ["grid truth table incl. boundaries"])
rec("O07", all((g ** 3 - g > 0) == member(g, [(Fr(-1), Fr(0), True, True), (Fr(1), None, True, True)]) for g in grid), ["grid truth table incl. boundaries x=-1,0,1"])
rec("O08", all(not (g * g + 1 < 0) for g in grid) and all(g * g + 1 >= 1 for g in grid), ["x^2+1 >= 1 > 0 on 161 grid points: the inequality has no real solution"])

# ------------------------------------------------------------------------------------------------ P
rec("P01", abs(mp.sin(mp.pi / 12) - (mp.sqrt(6) - mp.sqrt(2)) / 4) < EPS, ["mpmath 50 digits"])
rec("P02", abs(mp.cos(7 * mp.pi / 6) + mp.sqrt(3) / 2) < EPS, ["mpmath 50 digits"])
P = [mp.mpf(v) for v in ("-3.7", "-0.6", "0.9", "2.3", "5.1")]
rec("P03", all(abs((1 - mp.cos(x) ** 2) / mp.sin(x) ** 2 - 1) < EPS for x in P), ["mpmath at 5 points (sin x != 0)"])
rec("P04", all(abs(mp.tan(x) * mp.cos(x) - mp.sin(x)) < EPS for x in P), ["mpmath at 5 points"])
rec("P05", abs(mp.cos(mp.pi / 5) - (1 + mp.sqrt(5)) / 4) < EPS, ["mpmath 50 digits"])

# ------------------------------------------------------------------------------------------------ Q  (mpmath numerical differentiation)
def dchk(cid, f, d, points, order=1):
    err = max(abs(mp.diff(f, p, order) - d(p)) for p in points)
    rec(cid, err < mp.mpf(10) ** -20, ["mpmath.diff (order %d) numerical differentiation vs closed form at %d pts" % (order, len(points))], "max err %s" % mp.nstr(err, 3))


dchk("Q01", lambda x: (x ** 2 + 1) ** mp.sin(x), lambda x: (x ** 2 + 1) ** mp.sin(x) * (mp.cos(x) * mp.log(x ** 2 + 1) + 2 * x * mp.sin(x) / (x ** 2 + 1)), P)
dchk("Q02", lambda x: mp.atan(x ** 2), lambda x: 2 * x / (x ** 4 + 1), P)
dchk("Q03", lambda x: mp.sqrt(1 + x ** 2) / x, lambda x: -1 / (x ** 2 * mp.sqrt(1 + x ** 2)), P)
dchk("Q04", lambda x: x ** 4 * mp.exp(x), lambda x: x ** 2 * (x ** 2 + 8 * x + 12) * mp.exp(x), P, order=2)
dchk("Q05", lambda x: mp.log(x + mp.sqrt(x ** 2 + 1)), lambda x: 1 / mp.sqrt(x ** 2 + 1), P)

# ------------------------------------------------------------------------------------------------ R
def achk(cid, f, F, points, extra=""):
    err = max(abs(mp.diff(F, p) - f(p)) for p in points)
    rec(cid, err < mp.mpf(10) ** -20, ["mpmath.diff of the expected antiderivative equals the integrand at %d pts" % len(points)], "max err %s %s" % (mp.nstr(err, 3), extra))


POS = [mp.mpf(v) for v in ("0.5", "1.3", "2.7", "7.1")]
achk("R01", lambda x: x ** 2 * mp.exp(x), lambda x: mp.exp(x) * (x ** 2 - 2 * x + 2), P)
achk("R02", lambda x: mp.log(x), lambda x: x * mp.log(x) - x, POS)
achk("R03", lambda x: 1 / (x ** 2 + 1), mp.atan, P)
achk("R04", lambda x: 1 / mp.sqrt(1 - x ** 2), mp.asin, [mp.mpf(v) for v in ("-0.75", "-0.33", "0.25", "0.8")])
achk("R05", lambda x: mp.sin(x) ** 2, lambda x: x / 2 - mp.sin(2 * x) / 4, P)
F6 = lambda x: mp.log(abs((x - 1) / (x + 1))) / 2
p6 = [mp.mpf(v) for v in ("-3", "-0.5", "0.5", "3", "-1.7", "2.2")]
achk("R06", lambda x: 1 / (x ** 2 - 1), F6, p6, "(F real on all three intervals; log without |.| would be complex there)")
R["R06"]["ok"] = R["R06"]["ok"] and all(abs(mp.im(F6(p))) == 0 for p in p6) and abs(mp.im(mp.log((mp.mpf(-3) - 1) / (mp.mpf(-3) + 1)))) == 0 and abs(mp.im(mp.log(mp.mpf(-3) - 1))) > 1
rec("R07", abs(mp.quad(mp.log, [0, 1]) + 1) < mp.mpf(10) ** -20, ["mpmath.quad (tanh-sinh handles the endpoint singularity)"])
rec("R08", abs(mp.quad(lambda x: 1 / x ** 2, [1, mp.inf]) - 1) < mp.mpf(10) ** -25, ["mpmath.quad to infinity"])
rec("R09", abs(mp.quad(lambda x: mp.exp(-x ** 2), [-mp.inf, mp.inf]) - mp.sqrt(mp.pi)) < mp.mpf(10) ** -25, ["mpmath.quad over the real line"])
eps_list = [mp.mpf("1e-3"), mp.mpf("1e-6"), mp.mpf("1e-9")]
vals = [mp.quad(lambda x: 1 / x ** 2, [e, 1]) for e in eps_list]
rec("R10", all(abs(v - (1 / e - 1)) / (1 / e) < mp.mpf(10) ** -12 for v, e in zip(vals, eps_list)) and vals[2] > 9.9e8 and all(1 / mp.mpf(x) ** 2 > 0 for x in (-1, -0.5, 0.5, 1)),
    ["truncated integrals int_eps^1 x^-2 = 1/eps - 1 (matches mpmath.quad) grow without bound as eps -> 0", "integrand positive everywhere, so the naive FTC value -2 is impossible"], [mp.nstr(v, 5) for v in vals])
pv = lambda e1, e2: mp.quad(lambda x: 1 / x, [-1, -e2]) + mp.quad(lambda x: 1 / x, [e1, 1])
e = mp.mpf("1e-6")
rec("R11", abs(pv(e, e)) < mp.mpf(10) ** -20 and abs(pv(e, 2 * e) - mp.log(2)) < mp.mpf(10) ** -15,
    ["symmetric excision gives 0 (principal value) but excising (e, 2e) gives ln 2: the improper integral's limit depends on the path -> does not converge"],
    "pv(e,e)=%s pv(e,2e)=%s" % (mp.nstr(pv(e, e), 5), mp.nstr(pv(e, 2 * e), 8)))

# ------------------------------------------------------------------------------------------------ S
def lchk(cid, f, x0, target, sides=(1, -1), h="1e-12", tol="1e-9"):
    vals = [f(x0 + s * mp.mpf(h)) for s in sides] if x0 is not None else [f(mp.mpf(10) ** 14), f(mp.mpf(10) ** 18)]
    rec(cid, all(abs(v - target) < mp.mpf(tol) for v in vals), ["mpmath 50-digit evaluation approaching the point"], [mp.nstr(v, 12) for v in vals])


lchk("S01", lambda x: (mp.exp(x) - 1 - x) / x ** 2, mp.mpf(0), mp.mpf(1) / 2, h="1e-12", tol="1e-6")
lchk("S02", lambda x: (1 + 1 / x) ** x, None, mp.e, tol="1e-9")
lchk("S03", lambda x: x ** x, mp.mpf(0), mp.mpf(1), sides=(1,), h="1e-14", tol="1e-9")
v = mp.mpf(1) / mp.mpf("1e-12")
rec("S04", v > 1e11 and mp.mpf(1) / mp.mpf("1e-15") > 1e14, ["1/x grows without bound as x -> 0+"])
rec("S05", all(abs(abs(x) / x + 1) < EPS for x in (mp.mpf("-1e-6"), mp.mpf("-1e-12"))), ["abs(x)/x == -1 for x<0"])
rec("S06", abs(abs(mp.mpf("-1e-9")) / mp.mpf("-1e-9") + 1) < EPS and abs(abs(mp.mpf("1e-9")) / mp.mpf("1e-9") - 1) < EPS, ["left limit -1, right limit +1: they differ"])
a = [mp.sin(2 * mp.pi * n) for n in (10, 100, 1000)]
b = [mp.sin(2 * mp.pi * n + mp.pi / 2) for n in (10, 100, 1000)]
rec("S07", all(abs(x) < EPS for x in a) and all(abs(y - 1) < EPS for y in b), ["sin(1/x) at x=1/(2 pi n) -> 0 and at x=1/(2 pi n + pi/2) -> 1: two sequences to 0 with different limits"])
lchk("S08", lambda x: mp.sqrt(x ** 2 + x) - x, None, mp.mpf(1) / 2, tol="1e-9")
co = mp.taylor(mp.exp, 0, 4)
rec("S09", all(abs(c - t) < EPS for c, t in zip(co, [1, 1, mp.mpf(1) / 2, mp.mpf(1) / 6, mp.mpf(1) / 24])), ["mpmath.taylor coefficients"])
rec("S10", abs(mp.nsum(lambda k: 1 / k ** 2, [1, mp.inf]) - mp.pi ** 2 / 6) < mp.mpf(10) ** -20, ["mpmath.nsum (Richardson)"])
okh = True
for k in range(1, 15):
    s = sum(Fr(1, m) for m in range(2 ** (k - 1) + 1, 2 ** k + 1))
    okh = okh and s >= Fr(1, 2)
rec("S11", okh, ["each dyadic block sum_{2^(k-1)<m<=2^k} 1/m >= 1/2 (exact, k=1..14) -> partial sums are unbounded"])
rec("S12", abs(mp.nsum(lambda k: mp.mpf(1) / 2 ** k, [0, mp.inf]) - 2) < mp.mpf(10) ** -25 and all(sum(Fr(1, 2 ** k) for k in range(n + 1)) == 2 - Fr(1, 2 ** n) for n in range(1, 12)),
    ["mpmath.nsum", "exact partial sums 2 - 2^-n"])

# ------------------------------------------------------------------------------------------------ T
A = [[Fr(2), Fr(1), Fr(3)], [Fr(0), Fr(-1), Fr(4)], [Fr(1), Fr(2), Fr(5)]]


def elim_det(m):
    m = [r[:] for r in m]
    n, d = len(m), Fr(1)
    for c in range(n):
        p = next((i for i in range(c, n) if m[i][c] != 0), None)
        if p is None:
            return Fr(0)
        if p != c:
            m[c], m[p] = m[p], m[c]; d = -d
        d *= m[c][c]
        for i in range(c + 1, n):
            f_ = m[i][c] / m[c][c]
            m[i] = [u - f_ * w for u, w in zip(m[i], m[c])]
    return d


rec("T01", det3(A) == -19 and elim_det(A) == -19, ["cofactor expansion (Fractions)", "Gaussian elimination pivot product"])
exp_inv = json.load(open(ROOT / "data/cases.json"))
inv = next(c for c in exp_inv["cases"] if c["id"] == "T02")["expected"]["matrix"]
invF = [[Fr(v) for v in r] for r in inv]
prod = [[sum(A[i][k] * invF[k][j] for k in range(3)) for j in range(3)] for i in range(3)]
rec("T02", prod == [[1, 0, 0], [0, 1, 0], [0, 0, 1]], ["A * expected_inverse == I (exact Fractions)"])


def rank_fr(m):
    m = [[Fr(v) for v in r] for r in m]
    r = 0
    for c in range(len(m[0])):
        p = next((i for i in range(r, len(m)) if m[i][c] != 0), None)
        if p is None:
            continue
        m[r], m[p] = m[p], m[r]
        for i in range(r + 1, len(m)):
            f_ = m[i][c] / m[r][c]
            m[i] = [u - f_ * w for u, w in zip(m[i], m[r])]
        r += 1
    return r


rec("T03", rank_fr([[1, 2, 3], [2, 4, 6], [1, 0, 1]]) == 2 and [2 * v for v in [1, 2, 3]] == [2, 4, 6], ["Gaussian elimination rank (Fractions)", "row 2 = 2 x row 1; rows 1 and 3 independent"])
lam = [1, 3]
rec("T04", all((2 - l) * (2 - l) - 1 == 0 for l in lam) and sorted(mp.polyroots([1, -4, 3])) == [1, 3] or all(abs(a - b) < EPS for a, b in zip(sorted(mp.polyroots([1, -4, 3]), key=lambda z: z.real), [1, 3])),
    ["det(A - lambda I) = 0 substitution", "characteristic polynomial l^2-4l+3 (mpmath.polyroots)", "trace 4 = 1+3, det 3 = 1*3"])
rec("T05", all(abs((0 - l) * (0 - l) - (-1) * (1)) < EPS for l in (1j, -1j)) and all(abs(a - b) < EPS for a, b in zip(sorted(mp.polyroots([1, 0, 1]), key=lambda z: z.imag), [-1j, 1j])),
    ["complex substitution into det(A - lambda I)", "characteristic polynomial l^2+1"])
rec("T06", Fr(1) * 4 - Fr(2) * 2 == 0, ["det = 1*4 - 2*2 = 0 -> not invertible"])
RR = next(c for c in exp_inv["cases"] if c["id"] == "T07")["expected"]["matrix"]
A7 = [[1, 2, 1, 4], [2, 4, 0, 2], [1, 2, 2, 7]]
Rm = [[Fr(v) for v in r] for r in RR]
piv = []
for r_ in Rm:
    nz = next((j for j, v in enumerate(r_) if v != 0), None)
    if nz is not None:
        piv.append(nz)
shape_ok = all(Rm[i][piv[i]] == 1 and all(Rm[k][piv[i]] == 0 for k in range(len(Rm)) if k != i) for i in range(len(piv)))
rec("T07", shape_ok and rank_fr(A7) == rank_fr(RR) == rank_fr(A7 + RR), ["RREF shape (unit pivots, zero elsewhere in pivot columns)", "rank(A)=rank(R)=rank([A;R]) -> identical row space"])

# ------------------------------------------------------------------------------------------------ U
def odechk(cid, resid, consts, basis, note):
    err = max(abs(resid(x, *c)) for c in consts for x in (mp.mpf("-1.3"), mp.mpf("0.4"), mp.mpf("1.7")))
    ok = err < mp.mpf(10) ** -18
    if basis:
        ok = ok and all(abs(mp.diff(lambda t: basis[0](t), x) * basis[1](x) - basis[0](x) * mp.diff(lambda t: basis[1](t), x)) > 1e-6 for x in (mp.mpf("-1.3"), mp.mpf("0.4"), mp.mpf("1.7")))
    rec(cid, ok, ["substitute the general solution into the ODE (mpmath.diff), several constant values"] + (["Wronskian of the two basis solutions != 0 at 3 points"] if basis else []), "max residual %s %s" % (mp.nstr(err, 3), note))


y1 = lambda x, C: x - 1 + C * mp.exp(-x)
odechk("U01", lambda x, C: mp.diff(lambda t: y1(t, C), x) + y1(x, C) - x, [(mp.mpf(c),) for c in ("0", "1", "-2.5")], None, "")
y2 = lambda x, a, b: a * mp.exp(x) + b * mp.exp(2 * x)
odechk("U02", lambda x, a, b: mp.diff(lambda t: y2(t, a, b), x, 2) - 3 * mp.diff(lambda t: y2(t, a, b), x) + 2 * y2(x, a, b),
       [(mp.mpf(a), mp.mpf(b)) for a, b in (("1", "0"), ("0", "1"), ("2.2", "-0.7"))], (mp.exp, lambda t: mp.exp(2 * t)), "")
y3 = lambda x, a, b: (a + b * x) * mp.exp(x)
odechk("U03", lambda x, a, b: mp.diff(lambda t: y3(t, a, b), x, 2) - 2 * mp.diff(lambda t: y3(t, a, b), x) + y3(x, a, b),
       [(mp.mpf(a), mp.mpf(b)) for a, b in (("1", "0"), ("0", "1"), ("2.2", "-0.7"))], (mp.exp, lambda t: t * mp.exp(t)), "")
y4 = lambda x, a, b: mp.exp(-x) * (a * mp.cos(2 * x) + b * mp.sin(2 * x))
odechk("U04", lambda x, a, b: mp.diff(lambda t: y4(t, a, b), x, 2) + 2 * mp.diff(lambda t: y4(t, a, b), x) + 5 * y4(x, a, b),
       [(mp.mpf(a), mp.mpf(b)) for a, b in (("1", "0"), ("0", "1"), ("2.2", "-0.7"))], (lambda t: mp.exp(-t) * mp.cos(2 * t), lambda t: mp.exp(-t) * mp.sin(2 * t)), "")
y5 = lambda x, C: C * mp.exp(x ** 2 / 2)
odechk("U05", lambda x, C: mp.diff(lambda t: y5(t, C), x) - x * y5(x, C), [(mp.mpf(c),) for c in ("1", "-2.5", "0.3")], None, "")

# ------------------------------------------------------------------------------------------------ V
def gmul(a, b):
    return (a[0] * b[0] - a[1] * b[1], a[0] * b[1] + a[1] * b[0])


z = (1, 1)
w = (1, 0)
for _ in range(8):
    w = gmul(w, z)
rec("V01", w == (16, 0) and abs((1 + 1j) ** 8 - 16) < 1e-12, ["exact Gaussian-integer arithmetic", "Python complex"])
rec("V02", 3 * 3 + 4 * 4 == 25 and abs(abs(3 + 4j) - 5) < 1e-12, ["3^2+4^2 = 25 = 5^2", "Python complex abs"])
num = gmul((2, 3), (1, 1))     # multiply by the conjugate of (1 - i)
rec("V03", (Fr(num[0], 2), Fr(num[1], 2)) == (Fr(-1, 2), Fr(5, 2)) and abs((2 + 3j) / (1 - 1j) - (-0.5 + 2.5j)) < 1e-12, ["exact: (2+3i)(1+i)/2", "Python complex"])


# ------------------------------------------------------------------------------------------------ extra stress cases
POS1 = [mp.mpf(v) for v in ("1.3", "2", "2.7", "7")]
INS = [mp.mpf(v) for v in ("-0.75", "-0.33", "0.25", "0.8")]
achk("R12", lambda x: mp.exp(x) * mp.sin(x), lambda x: mp.exp(x) * (mp.sin(x) - mp.cos(x)) / 2, P)
achk("R13", lambda x: 1 / (x * mp.log(x)), lambda x: mp.log(mp.log(x)), POS1)
achk("R14", lambda x: x / (x ** 2 + x + 1), lambda x: mp.log(x ** 2 + x + 1) / 2 - mp.atan((2 * x + 1) / mp.sqrt(3)) / mp.sqrt(3), P)
achk("R15", lambda x: mp.sqrt(1 - x ** 2), lambda x: (x * mp.sqrt(1 - x ** 2) + mp.asin(x)) / 2, INS)
lchk("S13", lambda x: (mp.tan(x) - x) / x ** 3, mp.mpf(0), mp.mpf(1) / 3, h="1e-10", tol="1e-6")
lchk("S14", lambda x: mp.cos(x) ** (1 / x ** 2), mp.mpf(0), mp.exp(-mp.mpf(1) / 2), h="1e-8", tol="1e-7")
lchk("S15", lambda x: x ** (1 / x), None, mp.mpf(1), tol="1e-9")
rec("S16", all(abs(c - t) < EPS for c, t in zip(mp.taylor(mp.tan, 0, 5), [0, 1, 0, mp.mpf(1) / 3, 0, mp.mpf(2) / 15])), ["mpmath.taylor coefficients"])
rec("S17", all(abs(c - t) < EPS for c, t in zip(mp.taylor(lambda x: mp.log(1 + x), 0, 4), [0, 1, -mp.mpf(1) / 2, mp.mpf(1) / 3, -mp.mpf(1) / 4])), ["mpmath.taylor coefficients"])
A8 = [[Fr(2), Fr(0), Fr(0)], [Fr(0), Fr(3), Fr(4)], [Fr(0), Fr(4), Fr(9)]]
def charval(m, lam):
    return det3([[m[i][j] - (lam if i == j else 0) for j in range(3)] for i in range(3)])
rec("T08", all(charval(A8, Fr(l)) == 0 for l in (2, 1, 11)) and 2 + 1 + 11 == 14 and 2 * 1 * 11 == 22 == det3(A8) and
    all(abs(a - b) < EPS for a, b in zip(sorted(mp.polyroots([1, -14, 35, -22]), key=lambda z: z.real), [1, 2, 11])),
    ["det(A - lambda I) = 0 substitution", "trace 14 = 2+1+11 and det 22 = 2*1*11", "mpmath.polyroots of the characteristic polynomial"])
M4 = [[1, 2, 0, 1], [0, 1, 3, 2], [2, 0, 1, 1], [1, 1, 1, 0]]
def cof_det(m):
    if len(m) == 1:
        return Fr(m[0][0])
    return sum((-1) ** j * Fr(m[0][j]) * cof_det([r[:j] + r[j + 1:] for r in m[1:]]) for j in range(len(m)))
rec("T09", cof_det(M4) == -15 and elim_det([[Fr(v) for v in r] for r in M4]) == -15, ["recursive cofactor expansion (Fractions)", "Gaussian elimination pivot product"])
y6 = lambda x, a, b: x + a * mp.cos(x) + b * mp.sin(x)
odechk("U06", lambda x, a, b: mp.diff(lambda t: y6(t, a, b), x, 2) + y6(x, a, b) - x,
       [(mp.mpf(a), mp.mpf(b)) for a, b in (("1", "0"), ("0", "1"), ("2.2", "-0.7"))], (mp.cos, mp.sin), "")
y7 = lambda x, C: -1 / (x + C)
odechk("U07", lambda x, C: mp.diff(lambda t: y7(t, C), x) - y7(x, C) ** 2, [(mp.mpf(c),) for c in ("5", "-2.5", "10")], None, "(x + C != 0)")
def pred9(x):
    return (x - 1) * (x + 2) ** 2 * (x - 3) > 0
rec("O09", all(pred9(g) == member(g, [(None, Fr(-2), True, True), (Fr(-2), Fr(1), True, True), (Fr(3), None, True, True)]) for g in grid),
    ["grid truth table incl. the double root x=-2 (excluded), x=1 and x=3 (excluded)"])

ids = [c["id"] for c in json.load(open(ROOT / "data/cases.json"))["cases"]]
missing = [i for i in ids if i not in R]
out = {"validated": len(R), "all_ok": all(v["ok"] for v in R.values()) and not missing, "missing": missing,
       "failed": [k for k, v in R.items() if not v["ok"]], "checks": dict(sorted(R.items()))}
(ROOT / "results").mkdir(exist_ok=True)
(ROOT / "results" / "reference_validation.json").write_text(json.dumps(out, indent=1, default=str) + "\n")
print("validated", len(R), "cases | all_ok =", out["all_ok"], "| failed:", out["failed"], "| missing:", missing)
sys.exit(0 if out["all_ok"] else 1)
''')
print('wrote', 4, 'files')

wrote 4 files


In [6]:
#@title 3d. Benchmark source: evaluation, engineering probes, scoring, reports { display-mode: "form" }
W('bench/evaluate.py', r'''#!/usr/bin/env python3
"""raw replies (results/raw/*.json) + data/cases.json  ->  results/raw_results.json / .csv  (one row per solver per case).

Re-runnable without re-running any solver.  Usage: python -m bench.evaluate [--raw results/raw] [--out results]
"""
import argparse
import csv
import hashlib
import json
import pathlib
import statistics
import subprocess

try:
    from . import derivations as D
    from . import judge as J
except ImportError:
    import derivations as D
    import judge as J

ROOT = pathlib.Path(__file__).resolve().parent.parent
ORDER = ["sympy-direct", "smartsolver", "stepsolver"]


def load_evidence():
    """results/evidence/candidates.json is written by the setup step (commit, url, ref, licence, dependency count, history)."""
    p = ROOT / "results/evidence/candidates.json"
    return json.load(open(p)) if p.exists() else {}


def git_commit(name, evidence):
    return (evidence.get(name) or {}).get("commit")


def fingerprint(reply):
    """Identity of the derivation + answer a manual review was written for (a changed engine output invalidates the review)."""
    keep = {"steps": reply.get("steps"), "answer": reply.get("answer"), "outcome": reply.get("outcome")}
    return hashlib.sha256(json.dumps(keep, sort_keys=True, default=str).encode()).hexdigest()[:16]


def pct(sorted_vals, q):
    if not sorted_vals:
        return None
    k = (len(sorted_vals) - 1) * q
    f, c = int(k), min(int(k) + 1, len(sorted_vals) - 1)
    return sorted_vals[f] + (sorted_vals[c] - sorted_vals[f]) * (k - f)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--raw", default=str(ROOT / "results/raw"))
    ap.add_argument("--out", default=str(ROOT / "results"))
    a = ap.parse_args()
    cases = json.load(open(ROOT / "data/cases.json"))["cases"]
    mr_path = ROOT / "data/manual_reviews.json"
    manual = json.load(open(mr_path)) if mr_path.exists() else {}
    rows = []
    meta = {}
    evidence = load_evidence()
    stale_reviews = []
    for name in ORDER:
        p = pathlib.Path(a.raw) / ("%s.json" % name)
        if not p.exists():
            continue
        raw = json.load(open(p))
        info = raw["info"] or {}
        meta[name] = {"version": info.get("version"), "commit": git_commit(name, evidence), "package": info.get("package"),
                      "python": info.get("python"), "sympy_version": info.get("sympy_version") or info.get("version") if name == "sympy-direct" else info.get("sympy_version"),
                      "cold_start": raw["cold_start"], "maxrss_kb": raw["maxrss_kb"], "timeouts": raw["timeouts"], "crashes": raw["crashes"],
                      "protocol": raw["protocol"], "public_methods": info.get("public_methods"), "operations": info.get("operations")}
        for c in cases:
            rr = raw["rows"].get(c["id"])
            if rr is None:
                rows.append({"solver": name, "case_id": c["id"], "status": "skipped", "skip_reason": "case not selected in this run"})
                continue
            reply = rr["reply"]
            verdict = J.judge(c, reply)
            deriv = D.automated_score(c, reply, verdict)
            review = manual.get("%s|%s" % (name, c["id"]))
            if review is not None and review.get("fingerprint") not in (None, fingerprint(reply)):
                stale_reviews.append("%s|%s" % (name, c["id"]))      # engine output changed since the review: NOT applied
                review = None
            deriv = D.apply_manual(deriv, review)
            warm = sorted(t for t in rr.get("warm_times_ms", []) if t is not None)
            rows.append({
                "solver": name, "solver_version": meta[name]["version"], "solver_commit": meta[name]["commit"],
                "case_id": c["id"], "category": c["category"], "tier": c.get("tier", 1), "weight": c.get("weight", 1), "adversarial": c.get("adversarial", False), "operation": c["operation"], "display_input": c["display_input"],
                "normalised_operation": c["operation"], "problem": c["problem"], "domain": c["domain"], "domain_sensitive": c["domain_sensitive"],
                "adapter_api_call": reply.get("api_call"), "adapter_input": reply.get("input_sent"),
                "adapter_overrides": bool(reply.get("input_overrides_used")), "domain_assumption_passed": reply.get("domain_assumption_passed"),
                "raw_output": reply.get("raw_output"), "raw_answer_canonical": reply.get("answer"), "raw_steps": reply.get("steps"),
                "native_status": reply.get("native_status"), "native_flags": reply.get("native_flags"),
                "restrictions_reported": reply.get("restrictions_reported"),
                "expected_spec": c["spec_expected"], "expected_reference": c["expected"], "actual": verdict["actual"],
                "status": verdict["status"], "status_detail": verdict["sub"], "value_correct": verdict["value_correct"],
                "form_ok": verdict["form_ok"], "form_note": verdict["form_note"], "domain_ok": verdict["domain_ok"],
                "domain_check": verdict["domain_check"], "answer_produced": verdict["answer_produced"], "covered": verdict["covered"],
                "strict_correct": verdict["strict_correct"], "value_only_correct": verdict["value_only_correct"],
                "equivalence_proof": verdict["proof"], "judge_detail": verdict["detail"], "constant_shown": verdict["constant_shown"],
                "derivation": deriv, "unsupported_reason": reply.get("unsupported_reason"), "error": reply.get("error"),
                "time_first_call_ms": reply.get("time_ms"), "wall_ms": rr.get("wall_ms"), "warm_times_ms": warm,
                "warm_median_ms": statistics.median(warm) if warm else None, "repeatable": rr.get("repeatable"),
                "timeout": reply.get("outcome") == "timeout", "manual_review_flag": verdict["manual_review"],
                "manual_review_reasons": verdict["manual_reasons"],
            })
    out = pathlib.Path(a.out)
    out.mkdir(exist_ok=True)
    (out / "raw_results.json").write_text(json.dumps({"meta": meta, "rows": rows, "stale_manual_reviews": stale_reviews,
                                                       "manual_reviews_total": len(manual)}, indent=1, default=str))
    if stale_reviews:
        print("manual reviews NOT applied (engine output changed since review): %d" % len(stale_reviews))
    cols = ["solver", "solver_version", "solver_commit", "case_id", "category", "operation", "display_input", "status", "status_detail",
            "strict_correct", "value_only_correct", "covered", "form_ok", "domain_ok", "expected_spec", "actual", "derivation_available",
            "derivation_score_auto", "derivation_score_final", "time_first_call_ms", "warm_median_ms", "timeout", "error_type",
            "manual_review_flag"]
    with open(out / "raw_results.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=cols)
        w.writeheader()
        for r in rows:
            d = r.get("derivation") or {}
            w.writerow({**{k: r.get(k) for k in cols if k in r}, "derivation_available": d.get("available"),
                        "derivation_score_auto": d.get("automated_total"), "derivation_score_final": d.get("final_total"),
                        "error_type": (r.get("error") or {}).get("type")})
    print("rows:", len(rows), "->", out / "raw_results.json")
    for name in ORDER:
        rs = [r for r in rows if r["solver"] == name]
        if rs:
            from collections import Counter
            print("%-13s" % name, dict(Counter(r["status"] for r in rs)), "strict_correct=%d covered=%d" % (sum(1 for r in rs if r.get("strict_correct")), sum(1 for r in rs if r.get("covered"))))


if __name__ == "__main__":
    main()
''')

W('bench/engineering.py', r'''#!/usr/bin/env python3
"""Engineering / reliability probes (kept separate from mathematical failures).  Each probe goes through the same isolated worker
protocol (own venv, killed on timeout).  Benign canary only: math.factorial(4) -- proves evaluation of input as code without side effects.
Usage: python -m bench.engineering [--out results/engineering_results.json]"""
import argparse, json, pathlib, re, time
try:
    from .runner import Worker, WORKERS, ROOT
except ImportError:
    from runner import Worker, WORKERS, ROOT

CANARY = "x - __import__('math').factorial(4) = 0"


def probes(solver):
    """-> {probe_id: (description, request)}; None = not applicable to this solver's interface."""
    def sym(kind, **k): return {"cmd": "raw", "probe": {"kind": kind, **k}}
    def smart(method, *args): return {"cmd": "raw", "probe": {"kind": "call", "method": method, "args": list(args)}}
    def step(q): return {"cmd": "raw", "probe": {"kind": "query", "query": q}}
    P = {}
    if solver == "sympy-direct":
        P["malformed_syntax"] = sym("parse", text="2 +* 3")
        P["unbalanced_parens"] = sym("parse", text="(x + 1")
        P["empty_input"] = sym("eval", text="")
        P["zero_denominator"] = sym("eval", text="1/0")
        P["missing_variable"] = sym("solve_for", equation="x + 1 = 0", var="z")
        P["two_variable_parametric"] = sym("solve_for", equation="x*y = 6", var="x")
        P["unsupported_operation"] = None
        P["heavy_equation"] = sym("solve_for", equation="x^25 - x - 1 = 0", var="x")
        P["large_polynomial_equation"] = sym("solve_for", equation="(x + 1)^60 = 0", var="x")
        P["untrusted_input_canary"] = sym("canary", text="__import__('math').factorial(4)")
    elif solver == "smartsolver":
        P["malformed_syntax"] = smart("solve_equation", "2 +* 3 = x", "x")
        P["unbalanced_parens"] = smart("solve_equation", "(x + 1 = 2", "x")
        P["empty_input"] = smart("solve_equation", "", "x")
        P["zero_denominator"] = smart("solve_equation", "1/0 = x", "x")
        P["missing_variable"] = smart("solve_equation", "x + 1 = 0", "z")
        P["two_variable_parametric"] = smart("solve_equation", "x*y = 6", "x")
        P["unsupported_operation"] = None
        P["heavy_equation"] = smart("solve_equation", "x**25 - x - 1 = 0", "x")
        P["large_polynomial_equation"] = smart("solve_equation", "(x + 1)**60 = 0", "x")
        P["untrusted_input_canary"] = smart("solve_equation", CANARY, "x")
    else:
        P["malformed_syntax"] = step("solve(2 +* 3 = x, x)")
        P["unbalanced_parens"] = step("solve((x + 1 = 2, x)")
        P["empty_input"] = step("")
        P["zero_denominator"] = step("simplify(1/0)")
        P["missing_variable"] = step("solve(x + 1 = 0, z)")
        P["two_variable_parametric"] = step("solve(x*y = 6, x)")
        P["unsupported_operation"] = step("frobnicate(x)")
        P["heavy_equation"] = step("solve(x^25 - x - 1 = 0, x)")
        P["large_polynomial_equation"] = step("solve((x + 1)^60 = 0, x)")
        P["untrusted_input_canary"] = step("solve(" + CANARY + ", x)")
    return P


def score(pid, solver, r, status):
    """-> (score in {0,0.5,1,None}, note).  None = not applicable."""
    err = r.get("error") or {}
    oc = r.get("outcome")
    txt = json.dumps(r.get("raw_output", "")) if r.get("raw_output") is not None else ""   # full text: matches must not miss a truncated tail
    if status != "ok":
        return 0.0, "worker %s (hang/crash) - needed external kill" % status
    if pid in ("malformed_syntax", "unbalanced_parens", "empty_input"):
        if oc == "answer":
            return 0.0, "returned an answer for malformed input: %s" % txt[:100]
        return (1.0, "library-typed error %s" % err.get("type")) if err.get("contract_exception") else (0.5, "raw Python exception %s (caught by adapter)" % err.get("type"))
    ns_ = r.get("native_status")
    if pid == "zero_denominator":
        if ns_ == "UndefinedResult":
            return 1.0, "explicit UndefinedResult"
        if ns_ == "UnsolvedResult":
            return 0.5, "honest decline (UnsolvedResult), no fabricated value"
        if oc == "error" and re.search(r"zero|division|undefined|zoo|domain", err.get("type", "") + err.get("message", ""), re.I):
            return 1.0, "explicit error: %s" % err.get("type")
        if oc == "answer" and re.search(r"undefined", txt, re.I):
            return 1.0, "explicit undefined result"
        if oc == "answer" and "zoo" in txt:
            return 0.5, "returns 'zoo' (complex infinity) without an undefined/domain message"
        if oc == "answer" and re.search(r"no solution|\[\]|EmptySet|empty", txt, re.I):
            return 0.5, "silently reports no solution"
        return 0.0, "outcome=%s %s" % (oc, (err.get("message") or txt)[:100])
    if pid == "missing_variable":
        if oc == "error":
            return 1.0, "error: %s" % err.get("type")
        if oc == "answer" and not re.search(r"-1", txt):
            return 0.5, "answer without fabricating a root: %s" % txt[:100]
        return 0.0, "fabricated/unclear result: %s" % txt[:100]
    if pid == "two_variable_parametric":
        if oc == "answer" and re.search(r"6\s*/\s*y|6 / y|\{6/y\}|y\*\*\(-1\)", txt.replace(" ", "")) or (oc == "answer" and "6/y" in txt.replace(" ", "")):
            return 1.0, "parametric solution 6/y"
        if oc == "answer" and "6 / y" in txt:
            return 1.0, "parametric solution 6/y"
        return (0.5, "honest error/unsolved: %s" % (err.get("type") or txt[:80])) if oc in ("error", "unsupported") else (0.0, "wrong/absent parametric solution: %s" % txt[:100])
    if pid == "unsupported_operation":
        if ns_ == "ExactResult":
            return 0.0, "unknown call 'frobnicate(x)' echoed back as a Verified ExactResult (silent acceptance of unknown functions)"
        if ns_ == "UnsolvedResult":
            return 0.5, "honest decline (UnsolvedResult)"
        return (1.0, "typed error %s" % err.get("type")) if err.get("contract_exception") else (0.5, "raw exception %s" % err.get("type"))
    if pid == "large_polynomial_equation":
        ok = oc == "answer" and re.search(r"-1\b", txt) is not None
        return (1.0, "correct root -1 in %.0f ms" % r.get("time_ms", -1)) if ok else (0.0, "outcome=%s %s" % (oc, (err.get("message") or txt)[:100]))
    if pid == "heavy_equation":
        return 1.0, "completed in %.0f ms (%s)" % (r.get("time_ms", -1), r.get("native_status") or oc)
    if pid == "untrusted_input_canary":
        executed = bool(re.search(r"\b24\b", txt)) or (oc == "error" and re.search(r"unsupported operand.*str|'str'", err.get("message", "")))
        return (0.0, "INPUT EVALUATED AS PYTHON (math.factorial(4)=24 reached the solver): %s" % (txt[:80] or err.get("message", "")[:80])) if executed else (1.0, "payload rejected: %s" % (err.get("type") or txt[:60]))
    return None, ""


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", default=str(ROOT / "results/engineering_results.json"))
    a = ap.parse_args()
    out = {}
    ev_path = ROOT / "results/evidence/candidates.json"
    evidence = json.loads(ev_path.read_text()) if ev_path.exists() else {}
    present = [s for s in WORKERS if (ROOT / "envs" / WORKERS[s][0] / "bin" / "python").exists() and (evidence.get(s, {}).get("installed", True))]
    for s in present:
        res = {}
        w = Worker(s)
        for pid, req in probes(s).items():
            if req is None:
                res[pid] = {"score": None, "note": "not applicable: interface has no operation-name dispatch", "reply": None}
                continue
            timeout = 15 if pid in ("heavy_equation", "large_polynomial_equation") else 20
            t = time.perf_counter()
            r, st = w.call({**req, "req_id": pid}, timeout)
            wall = (time.perf_counter() - t) * 1000
            if st != "ok":
                sc, note = 0.0, "worker %s after %ss - external kill needed" % (st, timeout)
                w.kill(); w = Worker(s); r = {"outcome": st}
            else:
                sc, note = score(pid, s, r, st)
            res[pid] = {"score": sc, "note": note, "wall_ms": wall, "reply": r}
        w.close()
        out[s] = res
    # static evidence: network/process imports in candidate library code (offline claim)
    import subprocess
    ev = {}
    for name in [n for n in present if n != "sympy-direct"]:
        path = evidence.get(name, {}).get("source_dir")
        if not path:
            continue
        g = subprocess.run("grep -rnE '^\\s*(import|from)\\s+(requests|urllib|socket|httpx|http\\.client|aiohttp|subprocess|os\\.system)' %s --include=*.py | grep -v '/tests/' | head -20" % path,
                           shell=True, capture_output=True, text=True).stdout.strip().splitlines()
        ev[name] = g
    out["_static_network_imports"] = ev
    pathlib.Path(a.out).write_text(json.dumps(out, indent=1, default=str))
    for s in present:
        print(s, {k: v["score"] for k, v in out[s].items()})


if __name__ == "__main__":
    main()
''')

W('bench/score.py', r'''#!/usr/bin/env python3
"""results/raw_results.json + engineering_results.json -> results/scores.json   (all rules documented in docs/SCORING_RULES.md)"""
import json
import math
import pathlib
import statistics
from collections import Counter, defaultdict

ROOT = pathlib.Path(__file__).resolve().parent.parent
ALL_SOLVERS = ["sympy-direct", "smartsolver", "stepsolver"]
W = dict(correctness=0.35, derivations=0.25, coverage=0.15, honesty=0.10, reliability=0.10, performance=0.03, integration=0.02)

# ---- Integration & maintainability rubric (14 points). Every score is computed from evidence collected by the setup step
# (results/evidence/candidates.json) or measured in this run; the two fixed assumptions are labelled as such.
def integration_rubric(solver, ev, override_share):
    ev = ev or {}
    crit = {}
    deps = ev.get("runtime_deps")
    if not ev.get("installed", False):
        crit["env_compat"] = (0, "install failed: %s" % ev.get("install_error"))
        crit["install"] = (0, "install failed")
    else:
        crit["env_compat"] = ((2 if deps <= 4 else 1 if deps <= 30 else 0),
                              "%s third-party packages installed besides the engine (2 if <=4, 1 if <=30, else 0); requires-python %s" % (deps, ev.get("python_requires")))
        crit["install"] = ((1 if ev.get("sympy_pin_fallback") else 2), "single install command succeeded%s" % ("; needed an unpinned-SymPy fallback" if ev.get("sympy_pin_fallback") else ""))
    crit["adaptation"] = ((2 if override_share <= 0.02 else 1 if override_share <= 0.10 else 0),
                          "%.1f%% of cases needed per-case adapter handling (2 if <=2%%, 1 if <=10%%, else 0)" % (100 * override_share))
    if solver == "sympy-direct":
        crit["testability_extensibility"] = (0, "FIXED ASSUMPTION: bare engine has no derivation/step framework to test or extend")
    else:
        n = (ev.get("tests") or {}).get("static_test_functions", 0)
        own = (ev.get("tests") or {}).get("own_suite") or {}
        pts = 2 if n >= 100 else 1 if n >= 1 else 0
        note = "%d test functions found statically (2 if >=100, 1 if >=1)" % n
        if own.get("run"):
            note += "; own suite run: %s passed / %s failed" % (own.get("passed"), own.get("failed"))
            if own.get("failed"):
                pts = max(0, pts - 1)
        crit["testability_extensibility"] = (pts, note)
    cls = ev.get("licence_class", "unknown")
    crit["licence"] = ({"permissive": 2, "unknown": 1, "copyleft": 0}[cls], "licence %s (%s)" % (ev.get("licence"), cls))
    if solver == "sympy-direct":
        crit["maintenance"] = (2, "FIXED ASSUMPTION: mature, widely maintained project (not measurable from a PyPI install)")
    else:
        h = ev.get("history") or {}
        if h.get("age_days", 0) >= 730 and h.get("authors", 0) >= 5:
            m = 2
        elif h.get("commits", 0) >= 5 and h.get("days_since_last_commit", 10 ** 6) <= 365:
            m = 1
        else:
            m = 0
        crit["maintenance"] = (m, "history: %s commits, %s authors, age %s days, last commit %s days ago, %s tags (2 if >=2y and >=5 authors; 1 if >=5 commits and active within 1y)" %
                               (h.get("commits"), h.get("authors"), h.get("age_days"), h.get("days_since_last_commit"), h.get("tags")))
    crit["ui_independent"] = (1, "FIXED ASSUMPTION: used headless as a library in this benchmark")
    crit["no_rewrite"] = (1, "FIXED ASSUMPTION: plain library call; adapter-sized integration")
    return crit


def lat_score(ms):
    """100 at <=10 ms, 0 at >=2000 ms, log-linear in between."""
    if ms is None:
        return 0.0
    if ms <= 10:
        return 100.0
    if ms >= 2000:
        return 0.0
    return 100.0 * math.log10(2000.0 / ms) / math.log10(200.0)


def pctl(vals, q):
    v = sorted(vals)
    k = (len(v) - 1) * q
    f, c = int(k), min(int(k) + 1, len(v) - 1)
    return v[f] + (v[c] - v[f]) * (k - f)


def main():
    d = json.load(open(ROOT / "results/raw_results.json"))
    eng = json.load(open(ROOT / "results/engineering_results.json"))
    rows, meta = d["rows"], d["meta"]
    SOLVERS = [s for s in ALL_SOLVERS if s in meta]
    evidence = json.load(open(ROOT / "results/evidence/candidates.json")) if (ROOT / "results/evidence/candidates.json").exists() else {}
    by = {s: [r for r in rows if r["solver"] == s] for s in SOLVERS}
    n_app = len(by[SOLVERS[0]])
    out = {"applicable_cases": n_app, "weights": W, "solvers": {}, "solver_order": SOLVERS}

    # common set for fair latency comparison: cases where ALL solvers produced an answer
    answered = {s: {r["case_id"] for r in by[s] if r["answer_produced"] and r["warm_times_ms"]} for s in SOLVERS}
    common = sorted(set.intersection(*answered.values()))

    for s in SOLVERS:
        rs = by[s]
        cnt = Counter(r["status"] for r in rs)
        sub = Counter((r["status"], r["status_detail"].split(" ")[0] if r["status"] in ("unsupported", "invalid_answer") else "") for r in rs)
        strict = sum(1 for r in rs if r["strict_correct"])
        value_only = sum(1 for r in rs if r["value_only_correct"])
        value_nofrm = sum(1 for r in rs if r["value_correct"])
        covered = sum(1 for r in rs if r["covered"])
        attempted = [r for r in rs if not (r["status"] == "unsupported" and r["status_detail"] == "no_api")]
        crashes = sum(1 for r in attempted if r["status"] in ("runtime_error", "timeout", "infra_error"))
        dsens = [r for r in rs if r["domain_sensitive"]]
        dsens_ok = sum(1 for r in dsens if r["strict_correct"])
        rep = [r["repeatable"] for r in rs if r["repeatable"] is not None]
        eng_scores = [v["score"] for k, v in eng[s].items() if v["score"] is not None]
        canary = eng[s]["untrusted_input_canary"]["score"]
        R = dict(R1_repeatability=(sum(1 for x in rep if x) / len(rep)) if rep else 0.0,
                 R2_engineering_probes=statistics.mean(eng_scores),
                 R3_stability=1 - crashes / max(1, len(attempted)),
                 R4_domain_edge_cases=dsens_ok / len(dsens),
                 R5_untrusted_input_safety=canary)
        reliability = 100 * statistics.mean(R.values())
        # derivations
        scored = [r for r in rs if r["derivation"].get("scored")]
        avail = [r for r in rs if r["derivation"].get("available")]
        mean_final = statistics.mean(r["derivation"]["final_total"] for r in scored) if scored else 0.0
        mean_auto = statistics.mean(r["derivation"]["automated_total"] for r in scored) if scored else 0.0
        deriv_primary = mean_final * 10
        deriv_penalised = 10 * sum(r["derivation"]["final_total"] for r in scored) / n_app          # unavailable = 0 over ALL applicable cases
        dsens_scored = [r for r in scored if r["domain_sensitive"]]
        crit = defaultdict(list)
        for r in scored:
            for k, v in r["derivation"]["final"].items():
                crit[k].append(v)
        # performance
        warm_by_case = {r["case_id"]: r["warm_median_ms"] for r in rs if r["case_id"] in common}
        all_warm = [t for r in rs if r["case_id"] in common for t in r["warm_times_ms"]]
        med = statistics.median(warm_by_case.values()) if warm_by_case else None
        p95 = pctl(all_warm, 0.95) if all_warm else None
        own_med = statistics.median([r["warm_median_ms"] for r in rs if r["warm_median_ms"] is not None]) if any(r["warm_median_ms"] for r in rs) else None
        own_all = [t for r in rs for t in r["warm_times_ms"]]
        own_p95 = pctl(own_all, 0.95) if own_all else None
        fail_rate = (meta[s]["timeouts"] + meta[s]["crashes"]) / max(1, len(attempted))
        perf = 0.4 * lat_score(med) + 0.4 * lat_score(p95) + 0.2 * 100 * (1 - fail_rate)
        # integration
        override_share = sum(1 for r in rs if r["adapter_overrides"]) / n_app
        integ = integration_rubric(s, evidence.get(s), override_share)
        integ_pts = sum(v[0] for v in integ.values())
        integration = 100 * integ_pts / 14
        wtot = sum(r.get("weight", 1) for r in rs)
        wc = lambda pred: 100.0 * sum(r.get("weight", 1) for r in rs if pred(r)) / wtot       # difficulty-weighted share
        adv = [r for r in rs if r.get("adversarial")]
        dishonest = [r["case_id"] for r in adv if r["status"] == "invalid_answer"]            # a confident wrong/fabricated answer
        honesty = 100.0 * (1 - len(dishonest) / len(adv)) if adv else 100.0
        tiers = {}
        for t in sorted({r.get("tier", 1) for r in rs}):
            tr_ = [r for r in rs if r.get("tier", 1) == t]
            tiers[str(t)] = {"cases": len(tr_), "weight": t, "strict_correct": sum(1 for r in tr_ if r["strict_correct"]),
                             "pct": 100.0 * sum(1 for r in tr_ if r["strict_correct"]) / len(tr_), "covered": sum(1 for r in tr_ if r["covered"])}
        comp = dict(correctness=wc(lambda r: r["strict_correct"]), derivations=deriv_primary, coverage=100 * covered / n_app, honesty=honesty,
                    reliability=reliability, integration=integration, performance=perf)
        overall = sum(W[k] * comp[k] for k in W)
        # sensitivity variants
        def total(c):
            return sum(W[k] * c[k] for k in W)
        variants = {
            "primary (spec formula)": overall,
            "value-only correctness (no domain-reporting requirement)": total({**comp, "correctness": wc(lambda r: r["value_only_correct"])}),
            "value correctness ignoring form rules too": total({**comp, "correctness": wc(lambda r: bool(r["value_correct"]))}),
            "unweighted correctness (tiers ignored)": total({**comp, "correctness": 100 * strict / n_app}),
            "adversarial cases only for correctness": total({**comp, "correctness": (100.0 * sum(1 for r in adv if r["strict_correct"]) / len(adv)) if adv else comp["correctness"]}),
            "availability-penalised derivations (unavailable=0 over all cases)": total({**comp, "derivations": deriv_penalised}),
            "automated-only derivation scores (no manual overlay)": total({**comp, "derivations": mean_auto * 10}),
            "latency on each solver's own supported cases": total({**comp, "performance": 0.4 * lat_score(own_med) + 0.4 * lat_score(own_p95) + 0.2 * 100 * (1 - fail_rate)}),
        }
        # per-category
        cats = {}
        for c in sorted({r["category"] for r in rs}):
            cr = [r for r in rs if r["category"] == c]
            cats[c] = {"n": len(cr), "strict_correct": sum(1 for r in cr if r["strict_correct"]), "covered": sum(1 for r in cr if r["covered"]),
                       "unsupported": sum(1 for r in cr if r["status"] == "unsupported"), "invalid": sum(1 for r in cr if r["status"] == "invalid_answer"),
                       "errors": sum(1 for r in cr if r["status"] in ("runtime_error", "timeout", "infra_error", "input_incompatible"))}
        # per-operation timing (common set)
        ops = defaultdict(list)
        for r in rs:
            if r["case_id"] in common and r["warm_median_ms"] is not None:
                ops[r["operation"]].append(r["warm_median_ms"])
        cs = meta[s]["cold_start"]
        out["solvers"][s] = {
            "version": meta[s]["version"], "commit": meta[s]["commit"], "status_counts": dict(cnt),
            "defined": n_app, "executed": n_app, "skipped": cnt.get("skipped", 0), "passed(strict_correct)": strict, "failed(invalid+error)": cnt["invalid_answer"] + cnt["runtime_error"] + cnt["timeout"] + cnt["infra_error"] + cnt["input_incompatible"],
            "unsupported": cnt["unsupported"], "unsupported_no_api": sum(1 for r in rs if r["status"] == "unsupported" and r["status_detail"] == "no_api"),
            "unsolved_by_solver": sum(1 for r in rs if r["status"] == "unsupported" and r["status_detail"] != "no_api"),
            "value_only_correct": value_only, "covered": covered, "attempted": len(attempted), "crashes_timeouts": crashes,
            "reliability_metrics": R, "domain_sensitive_cases": len(dsens), "domain_sensitive_strict_correct": dsens_ok,
            "derivation": {"available": len(avail), "available_pct_of_applicable": 100 * len(avail) / n_app, "scored": len(scored),
                           "mean_automated": mean_auto, "mean_final": mean_final, "mean_final_domain_sensitive": statistics.mean(r["derivation"]["final_total"] for r in dsens_scored) if dsens_scored else None,
                           "criterion_means": {k: statistics.mean(v) for k, v in crit.items()}, "manual_reviewed": sum(1 for r in scored if r["derivation"].get("manual")),
                           "distribution": dict(Counter(int(r["derivation"]["final_total"]) for r in scored))},
            "performance": {"cold_start_ready_ms_median": statistics.median(c["spawn_to_ready_ms"] for c in cs) if cs else None,
                            "cold_first_solve_ms_median": statistics.median(c["first_solve_wall_ms"] for c in cs) if cs else None,
                            "warm_median_common_ms": med, "warm_p95_common_ms": p95, "warm_median_own_ms": own_med, "warm_p95_own_ms": own_p95,
                            "common_cases": len(common), "timeouts": meta[s]["timeouts"], "crashes": meta[s]["crashes"], "peak_rss_mb": meta[s]["maxrss_kb"] / 1024,
                            "per_operation_median_ms": {k: statistics.median(v) for k, v in sorted(ops.items())}},
            "integration": {"points": integ_pts, "max": 14, "criteria": {k: {"score": v[0], "evidence": v[1]} for k, v in integ.items()}},
            "tiers": tiers, "honesty": {"adversarial_cases": len(adv), "dishonest_cases": dishonest, "score": honesty},
            "components": comp, "overall": overall, "sensitivity": variants, "by_category": cats,
        }
    # ---- hard gates
    gates = {}
    for s in SOLVERS:
        rs = by[s]
        critical = [r for r in rs if r["status"] == "invalid_answer" and r["status_detail"].startswith(("claims no solution", "claims every x", "fabricated_solution", "wrong_", "returned solutions", "returned "))]
        majors = [r for r in rs if r["status"] == "invalid_answer" and r not in critical]
        g2 = []
        for r in rs:
            dd = r["derivation"]
            if not dd.get("scored"):
                continue
            nf = r.get("native_flags") or {}
            if nf.get("steps_mode") == "direct_evaluation":
                g2.append((r["case_id"], "steps_mode=direct_evaluation: worked steps replaced by direct solve"))
            ev = dd["evidence"]
            if ev["invalid_steps"] > 0:
                g2.append((r["case_id"], "automated invalid-step detection: " + ev["correctness"]))
            elif dd.get("manual") and dd["manual"]["adjusted"].get("correctness") is not None and "CONTRADICTION" in dd["manual"]["justification"]:
                g2.append((r["case_id"], "manual: contradiction"))
        g2_ids = sorted({c for c, _ in g2})
        # K01 final step shows the unchanged input (manual finding) counts as inconsistency
        for r in rs:
            dd = r["derivation"]
            if dd.get("manual") and "ORIGINAL expression" in dd["manual"]["justification"] and r["case_id"] not in g2_ids:
                g2_ids.append(r["case_id"]); g2.append((r["case_id"], "manual: final step shows the unchanged input"))
        c3 = eng[s]["untrusted_input_canary"]["score"]
        gates[s] = {
            "G1_critical_correctness": {"status": "FAIL" if critical else "PASS", "critical_cases": [(r["case_id"], r["status_detail"]) for r in critical],
                                         "major_defects(not gating)": [(r["case_id"], r["status_detail"][:70]) for r in majors]},
            "G2_fabricated_or_invalid_derivations": {"status": "FAIL" if len(g2_ids) >= 2 else "CONDITIONAL" if len(g2_ids) == 1 else "PASS", "cases": sorted(g2_ids), "evidence": g2},
            "G3_unsafe_execution": {"status": "PASS" if c3 == 1 else ("CONDITIONAL" if s == "sympy-direct" else "FAIL"), "canary_score": c3,
                                    "note": eng[s]["untrusted_input_canary"]["note"]},
            "G4_licence": {"status": ("PASS" if (evidence.get(s) or {}).get("licence_class") == "permissive" else "CONDITIONAL"), "licence": (evidence.get(s) or {}).get("licence", "unknown")},
        }
        st = [g["status"] for g in gates[s].values()]
        gates[s]["eligible_for_production_recommendation"] = "NO (as-is)" if "FAIL" in st else ("CONDITIONAL" if "CONDITIONAL" in st else "YES")
    out["hard_gates"] = gates
    out["common_latency_cases"] = common
    (ROOT / "results/scores.json").write_text(json.dumps(out, indent=1, default=str))
    for s in SOLVERS:
        o = out["solvers"][s]
        print("%-13s overall=%.1f  comps=%s  gates=%s" % (s, o["overall"], {k: round(v, 1) for k, v in o["components"].items()}, gates[s]["eligible_for_production_recommendation"]))
        print("      variants:", {k[:28]: round(v, 1) for k, v in o["sensitivity"].items()})


if __name__ == "__main__":
    main()
''')

W('bench/rank.py', r'''#!/usr/bin/env python3
"""results/raw_results.json + scores.json -> results/ranking.json and results/ranking.md

The ranking rules are fixed here, before any engine output is seen:
  1. Primary ranking: overall weighted score (see score.py).  Engines whose scores are within TIE_POINTS of the neighbour above them
     form a tie group; inside a tie group the order is decided, in this order, by
        (a) production eligibility (YES < CONDITIONAL < NO), (b) tier-3 (adversarial) correctness, (c) tier-2 correctness,
        (d) mean derivation score, (e) lower warm median latency.
  2. Cross-check: head-to-head (Copeland).  For every pair, each case is a win for the engine that is strictly correct when the other is
     not, weighted by the case's difficulty weight; the engine with the larger weighted margin wins the pair; Copeland score = pairs won.
  3. Stability: bootstrap over cases (2000 resamples, fixed seed).  The case-dependent components (difficulty-weighted correctness,
     coverage, honesty, derivation mean) are recomputed on each resample; reliability, performance and integration are held at their
     observed values.  We report rank probabilities, 95% intervals of the overall score and P(A outranks B).  Two engines are called
     'statistically separable' only if P(A outranks B) >= 0.90.
"""
import json
import pathlib
import random
import statistics

ROOT = pathlib.Path(__file__).resolve().parent.parent
RES = ROOT / "results"
NAMES = {"sympy-direct": "SymPy (direct)", "smartsolver": "SmartSolver", "stepsolver": "StepSolver"}
TIE_POINTS = 2.0
BOOT = 2000
SEP = 0.90
ELIG = {"YES": 0, "CONDITIONAL": 1, "NO (as-is)": 2}


def main():
    d = json.load(open(RES / "raw_results.json"))
    sc = json.load(open(RES / "scores.json"))
    S = sc["solver_order"]
    W = sc["weights"]
    sv, g = sc["solvers"], sc["hard_gates"]
    rows = d["rows"]
    case_ids = []
    for r in rows:
        if r["case_id"] not in case_ids:
            case_ids.append(r["case_id"])
    by = {s: {r["case_id"]: r for r in rows if r["solver"] == s} for s in S}

    # ---------------------------------------------------------------- 1. primary ranking with declared tie-breaks
    def key(s):
        t = sv[s]["tiers"]
        return (ELIG.get(g[s]["eligible_for_production_recommendation"], 2), -t.get("3", {}).get("pct", 0), -t.get("2", {}).get("pct", 0),
                -sv[s]["derivation"]["mean_final"], sv[s]["performance"]["warm_median_common_ms"] or 1e9)
    order = sorted(S, key=lambda s: -sv[s]["overall"])
    groups = []
    for s in order:
        if groups and groups[-1][-1][1] - sv[s]["overall"] <= TIE_POINTS:
            groups[-1].append((s, sv[s]["overall"]))
        else:
            groups.append([(s, sv[s]["overall"])])
    final, tied = [], {}
    for grp in groups:
        members = sorted([m[0] for m in grp], key=key)
        for m in members:
            final.append(m)
            tied[m] = len(members) > 1
    ranks = {s: i + 1 for i, s in enumerate(final)}

    # ---------------------------------------------------------------- 2. head-to-head
    h2h = {}
    copeland = {s: 0 for s in S}
    for a in S:
        for b in S:
            if a == b:
                continue
            wa = sum(by[a][c].get("weight", 1) for c in case_ids if by[a][c]["strict_correct"] and not by[b][c]["strict_correct"])
            wb = sum(by[b][c].get("weight", 1) for c in case_ids if by[b][c]["strict_correct"] and not by[a][c]["strict_correct"])
            only_a = [c for c in case_ids if by[a][c]["strict_correct"] and not by[b][c]["strict_correct"]]
            h2h["%s>%s" % (a, b)] = {"a_only_weight": wa, "b_only_weight": wb, "margin": wa - wb, "a_only_cases": only_a}
            if wa > wb:
                copeland[a] += 1
    cope_order = sorted(S, key=lambda s: (-copeland[s], ranks[s]))

    # ---------------------------------------------------------------- 3. bootstrap
    rng = random.Random(20260301)
    arr = {}
    for s in S:
        arr[s] = [(by[s][c].get("weight", 1), 1 if by[s][c]["strict_correct"] else 0, 1 if by[s][c]["covered"] else 0, bool(by[s][c].get("adversarial")),
                   1 if by[s][c]["status"] == "invalid_answer" else 0, (by[s][c]["derivation"].get("final_total") if by[s][c]["derivation"].get("scored") else None)) for c in case_ids]
    n = len(case_ids)
    rank_counts = {s: [0] * len(S) for s in S}
    overalls = {s: [] for s in S}
    beats = {(a, b): 0 for a in S for b in S if a != b}
    for _ in range(BOOT):
        idx = [rng.randrange(n) for _ in range(n)]
        ov = {}
        for s in S:
            smp = [arr[s][i] for i in idx]
            wsum = sum(x[0] for x in smp)
            corr = 100.0 * sum(x[0] * x[1] for x in smp) / wsum
            cov = 100.0 * sum(x[2] for x in smp) / n
            adv = [x for x in smp if x[3]]
            hon = 100.0 * (1 - sum(x[4] for x in adv) / len(adv)) if adv else sv[s]["components"]["honesty"]
            ds = [x[5] for x in smp if x[5] is not None]
            der = 10.0 * statistics.mean(ds) if ds else 0.0
            c = sv[s]["components"]
            ov[s] = (W["correctness"] * corr + W["derivations"] * der + W["coverage"] * cov + W["honesty"] * hon + W["reliability"] * c["reliability"] +
                     W["performance"] * c["performance"] + W["integration"] * c["integration"])
            overalls[s].append(ov[s])
        srt = sorted(S, key=lambda s: -ov[s])
        for k, s in enumerate(srt):
            rank_counts[s][k] += 1
        for a in S:
            for b in S:
                if a != b and ov[a] > ov[b]:
                    beats[(a, b)] += 1

    def pct(v, q):
        v = sorted(v)
        k = (len(v) - 1) * q
        f, c = int(k), min(int(k) + 1, len(v) - 1)
        return v[f] + (v[c] - v[f]) * (k - f)

    # ---------------------------------------------------------------- 4. weight sensitivity (components fixed at observed values)
    comps = list(W.keys())
    def dirichlet(alpha):
        g_ = [rng.gammavariate(a, 1.0) for a in alpha]
        t_ = sum(g_)
        return [x / t_ for x in g_]
    wsens = {}
    for label, alpha in (("moderate reweighting (Dirichlet around the declared weights, concentration 20)", [20.0 * W[k] + 0.01 for k in comps]),
                         ("arbitrary reweighting (uniform over all weight vectors)", [1.0] * len(comps))):
        tops = {s: 0 for s in S}
        for _ in range(BOOT):
            wv = dirichlet(alpha)
            ov_ = {s: sum(wv[i] * sv[s]["components"][k] for i, k in enumerate(comps)) for s in S}
            tops[max(S, key=lambda s: ov_[s])] += 1
        wsens[label] = {s: tops[s] / BOOT for s in S}
    boot = {s: {"rank_probability": [x / BOOT for x in rank_counts[s]], "overall_ci95": [pct(overalls[s], 0.025), pct(overalls[s], 0.975)], "overall_mean": statistics.mean(overalls[s])} for s in S}
    pair_p = {"%s>%s" % (a, b): beats[(a, b)] / BOOT for (a, b) in beats}
    out = {"tie_points": TIE_POINTS, "bootstrap_resamples": BOOT, "separable_threshold": SEP, "final_rank": final, "ranks": ranks, "tie_group_decided_by_tiebreak": tied,
           "copeland": copeland, "copeland_order": cope_order, "head_to_head": h2h, "bootstrap": boot, "pairwise_prob_outranks": pair_p, "weight_sensitivity_p_first": wsens,
           "primary_vs_copeland_agree": final == cope_order}
    (RES / "ranking.json").write_text(json.dumps(out, indent=1))

    # ---------------------------------------------------------------- markdown fragment
    def table(head, body):
        return "\n".join(["| " + " | ".join(head) + " |", "|" + "|".join("---" for _ in head) + "|"] + ["| " + " | ".join(str(c) for c in r) + " |" for r in body])
    L = []
    L.append("## Leaderboard\n")
    L.append("Rule: rank by overall weighted score; engines within %.1f points of the one above are a tie group and are ordered by (a) production eligibility, (b) tier-3 adversarial correctness, (c) tier-2 correctness, (d) mean derivation score, (e) warm latency. Hard gates are shown, never averaged away.\n" % TIE_POINTS)
    body = []
    for s in final:
        t = sv[s]["tiers"]
        body.append([ranks[s] if not tied[s] else "%d (tie-break)" % ranks[s], NAMES[s], "**%.1f**" % sv[s]["overall"], "%.0f%%" % t.get("1", {}).get("pct", 0), "%.0f%%" % t.get("2", {}).get("pct", 0),
                     "%.0f%%" % t.get("3", {}).get("pct", 0), "%.0f" % sv[s]["components"]["honesty"], "%.1f" % sv[s]["derivation"]["mean_final"], g[s]["eligible_for_production_recommendation"]])
    L.append(table(["Rank", "Engine", "Overall /100", "Tier 1 (hard)", "Tier 2 (very hard)", "Tier 3 (adversarial)", "Honesty", "Mean derivation /10", "Eligible as-is"], body))
    L.append("\nTier percentages are strictly-correct shares within each tier (weights 1 / 2 / 3). Honesty = share of adversarial cases where the engine did not return a confident wrong answer.\n")
    L.append("### Head-to-head cross-check (difficulty-weighted case wins)\n")
    hb = []
    for a in S:
        row = [NAMES[a]]
        for b in S:
            row.append("-" if a == b else "%d : %d" % (h2h["%s>%s" % (a, b)]["a_only_weight"], h2h["%s>%s" % (a, b)]["b_only_weight"]))
        row.append(copeland[a])
        hb.append(row)
    L.append(table(["Row vs column (weight won : weight lost)"] + [NAMES[b] for b in S] + ["Copeland"], hb))
    L.append("\nCopeland order: %s. Agrees with the primary ranking: **%s**.\n" % (" > ".join(NAMES[s] for s in cope_order), "yes" if out["primary_vs_copeland_agree"] else "NO - treat the order as unsettled"))
    L.append("### How stable is the ranking? (bootstrap over cases, %d resamples)\n" % BOOT)
    L.append(table(["Engine", "Overall (mean)", "95% interval"] + ["P(rank %d)" % (k + 1) for k in range(len(S))], [[NAMES[s], "%.1f" % boot[s]["overall_mean"], "%.1f - %.1f" % tuple(boot[s]["overall_ci95"])] + ["%.0f%%" % (100 * x) for x in boot[s]["rank_probability"]] for s in final]))
    L.append("")
    for i, a in enumerate(final):
        for b in final[i + 1:]:
            p = pair_p["%s>%s" % (a, b)]
            L.append("- %s outranks %s in %.0f%% of resamples -> %s." % (NAMES[a], NAMES[b], 100 * p, "statistically separable" if p >= SEP else ("leaning, but NOT separable (below %.0f%%)" % (100 * SEP))))
    L.append("\nOnly the case-dependent components are resampled; reliability, performance and integration are held fixed, so these intervals describe test-sample uncertainty, not run-to-run noise.\n")
    L.append("### How much does the ranking depend on the weights?\n")
    L.append("The bootstrap above is conditional on the declared weights. Re-drawing the component weights and counting how often each engine ranks first (components fixed at their observed values):\n")
    L.append(table(["Weight scheme"] + [NAMES[s] for s in final], [[k] + ["%.0f%%" % (100 * v[s]) for s in final] for k, v in wsens.items()]))
    L.append("\nIf the engine on top changes between schemes, the order is a statement about the weights as much as about the engines: compare the per-component table in section 8 and the Copeland cross-check.\n")
    (RES / "ranking.md").write_text("\n".join(L) + "\n")
    print("ranking:", " > ".join("%s (%.1f)" % (s, sv[s]["overall"]) for s in final), "| copeland agrees:", out["primary_vs_copeland_agree"])


if __name__ == "__main__":
    main()
''')

W('bench/compare_runs.py', r'''#!/usr/bin/env python3
"""Reproducibility check: compare two full runs (raw replies and judged verdicts). Timings are expected to differ; everything else must not."""
import json, pathlib, statistics, subprocess, sys
ROOT = pathlib.Path(__file__).resolve().parent.parent
a, b = sys.argv[1], sys.argv[2]
out = pathlib.Path(sys.argv[3]); out.mkdir(exist_ok=True, parents=True)
subprocess.run([sys.executable, str(ROOT / "bench/evaluate.py"), "--raw", b, "--out", str(out)], check=True, capture_output=True)
A = json.load(open(ROOT / "results/raw_results.json"))["rows"]; B = json.load(open(out / "raw_results.json"))["rows"]
Bi = {(r["solver"], r["case_id"]): r for r in B}
diff_hash = diff_status = diff_score = n = 0; details = []
ra = {s: json.load(open(pathlib.Path(a) / f"{s}.json")) for s in ("sympy-direct", "smartsolver", "stepsolver") if (pathlib.Path(a) / f"{s}.json").exists()}
rb = {s: json.load(open(pathlib.Path(b) / f"{s}.json")) for s in ra}
for r in A:
    n += 1
    o = Bi[(r["solver"], r["case_id"])]
    h1, h2 = ra[r["solver"]]["rows"][r["case_id"]].get("hash"), rb[r["solver"]]["rows"][r["case_id"]].get("hash")
    if h1 != h2: diff_hash += 1; details.append((r["solver"], r["case_id"], "raw canonical output differs"))
    if r["status"] != o["status"] or r["strict_correct"] != o["strict_correct"]: diff_status += 1; details.append((r["solver"], r["case_id"], "verdict differs"))
    if (r["derivation"].get("automated_total"), r["derivation"].get("final_total")) != (o["derivation"].get("automated_total"), o["derivation"].get("final_total")): diff_score += 1; details.append((r["solver"], r["case_id"], "derivation score differs"))
res = {"rows_compared": n, "raw_output_differences": diff_hash, "verdict_differences": diff_status, "derivation_score_differences": diff_score, "details": details[:20], "timing_median_ms_run1_vs_run2": {}}
for s in ra:
    m = lambda R: statistics.median([x["warm_median_ms"] for x in R if x["solver"] == s and x["warm_median_ms"] is not None])
    res["timing_median_ms_run1_vs_run2"][s] = [round(m(A), 2), round(m(B), 2)]
(ROOT / "results/reproducibility_check.json").write_text(json.dumps(res, indent=1))
print(json.dumps(res, indent=1))
''')

W('bench/report.py', r'''#!/usr/bin/env python3
"""results/*.json -> results/summary.md, results/failure_analysis.md (and results/changes_vs_baseline.md if data/baseline.json exists).

EVERYTHING in these reports is computed from the preserved results. There is no run-specific narrative: strongest/weakest areas and the
recommendation are produced by documented rules from the measured data, so the reports stay valid for any engine version."""
import json
import pathlib
from collections import defaultdict

ROOT = pathlib.Path(__file__).resolve().parent.parent
RES = ROOT / "results"
NAMES = {"sympy-direct": "SymPy (direct)", "smartsolver": "SmartSolver", "stepsolver": "StepSolver"}
CRIT = ["correctness", "completeness", "explanation", "consistency", "domain", "presentation"]


def f(x, n=1):
    return "n/a" if x is None else ("%.*f" % (n, x))


def table(head, rows):
    return "\n".join(["| " + " | ".join(head) + " |", "|" + "|".join("---" for _ in head) + "|"] + ["| " + " | ".join(str(c) for c in r) + " |" for r in rows])


def cut(x, n):
    return str(x if x is not None else "").replace("\n", " ").replace("|", "/")[:n]


def main():
    d = json.load(open(RES / "raw_results.json"))
    sc = json.load(open(RES / "scores.json"))
    eng = json.load(open(RES / "engineering_results.json"))
    rep = json.load(open(RES / "reproducibility_check.json")) if (RES / "reproducibility_check.json").exists() else None
    ref = json.load(open(RES / "reference_validation.json"))
    evp = RES / "evidence/candidates.json"
    evidence = json.load(open(evp)) if evp.exists() else {}
    envtxt = (RES / "evidence/environment.txt").read_text().strip() if (RES / "evidence/environment.txt").exists() else ""
    cases = {c["id"]: c for c in json.load(open(ROOT / "data/cases.json"))["cases"]}
    cats = json.load(open(ROOT / "data/cases.json"))["categories"]
    rows, meta = d["rows"], d["meta"]
    S3 = sc["solver_order"]
    by = {s: [r for r in rows if r["solver"] == s] for s in S3}
    sv, g = sc["solvers"], sc["hard_gates"]
    pf = {s: sv[s]["performance"] for s in S3}
    n_app = sc["applicable_cases"]
    scored_total = sum(sv[s]["derivation"]["scored"] for s in S3)
    reviewed_total = sum(sv[s]["derivation"]["manual_reviewed"] for s in S3)
    stale = len(d.get("stale_manual_reviews", []))
    cases_meta = json.load(open(ROOT / "data/cases.json"))
    suite = cases_meta.get("suite", "standard")
    rk = json.load(open(RES / "ranking.json")) if (RES / "ranking.json").exists() else None
    L = []
    A = L.append

    A("# Math-solver benchmark (%s suite): " % suite + " vs ".join(NAMES[s] for s in S3) + "\n")
    A("Generated from `results/raw_results.json`, `scores.json`, `engineering_results.json`. Every number and sentence is computed by code from preserved raw solver output; there is no hand-written, run-specific narrative.\n")
    A("> **Read this first.** (1) Mathematical correctness is decided by a symbolic/numeric judge (SymPy used only as a checker) and every reference answer is independently validated with mpmath/Fraction arithmetic. (2) **Derivation scores are proxies**: an automated rubric plus a manual overlay written by a language model for a specific earlier set of outputs. This run scored %d derivations; %d use that overlay; %d overlay entries were NOT applied because the engine's output changed since they were written (their automated score is used). The overlay needs human audit and never decides mathematical correctness. (3) One reference value in the task specification (K04) is slightly wrong; see section 6. (4) SymPy is evaluated as a **bare engine** (no step layer by design).\n" % (scored_total, reviewed_total, stale))

    if (RES / "ranking.md").exists():
        A((RES / "ranking.md").read_text())
        wl = sc["weights"]
        A("Difficulty tiers: tier 1 = hard (weight 1), tier 2 = very hard (weight 2), tier 3 = adversarial (weight 3; the correct behaviour is to report no-solution / undefined / diverges / does-not-exist / not-invertible). Overall weights: " + ", ".join("%s %d%%" % (k, round(100 * v)) for k, v in wl.items()) + ".\n")

    # 1
    A("## 1. Environment and versions\n")
    A("```\n" + envtxt + "\n```")
    A(table(["Engine", "Version", "Commit / source", "Python", "SymPy in its env", "Isolated env"],
            [[NAMES[s], meta[s]["version"], (meta[s]["commit"] or (evidence.get(s, {}).get("source") or "PyPI release"))[:44], meta[s]["python"], meta[s].get("sympy_version") or "n/a",
              "`envs/%s`" % {"sympy-direct": "sympy", "smartsolver": "smartsolver", "stepsolver": "stepsolver"}[s]] for s in S3]))
    A("\nEach engine ran in its own virtualenv and worker process from an empty working directory. The judge runs in a separate pinned environment. Timings depend on the hardware above and are only comparable between runs on the same hardware.\n")

    # 2
    A("## 2. Installation and execution status\n")
    tr = []
    for s in S3:
        e = evidence.get(s, {})
        own = (e.get("tests") or {}).get("own_suite") or {}
        tr.append([NAMES[s], "yes" if e.get("installed", True) else "NO: " + cut(e.get("install_error"), 80),
                   "%d/%d attempted" % (sv[s]["executed"], sv[s]["defined"]),
                   ("%s passed / %s failed" % (own.get("passed"), own.get("failed"))) if own.get("run") else "not run (static test count: %s)" % ((e.get("tests") or {}).get("static_test_functions") if (e.get("tests") or {}).get("static_test_functions") is not None else "n/a: bare engine"),
                   e.get("runtime_deps", "n/a"), "%s (%s)" % (e.get("licence"), e.get("licence_class")), e.get("python_requires") or "n/a"])
    A(table(["Engine", "Installed", "Executed", "Own test-suite", "Third-party deps", "Licence", "requires-python"], tr))
    A("\nSkipped cases: %s. A case is only 'unsupported' when the engine has no method for it or declares it unsolved; nothing is substituted.\n" % ", ".join("%s %d" % (NAMES[s], sv[s]["skipped"]) for s in S3))

    # 3
    A("## 3. Results by category (strictly correct / cases)\n")
    A(table(["Category"] + [NAMES[s] for s in S3],
            [[c + " " + nm] + ["%d/%d" % (sv[s]["by_category"][c]["strict_correct"], sv[s]["by_category"][c]["n"]) for s in S3] for c, nm in cats.items()] +
            [["**Total**"] + ["**%d/%d**" % (sv[s]["passed(strict_correct)"], sv[s]["defined"]) for s in S3]]))
    A("\n**By difficulty tier** (strictly correct / cases; weight):\n")
    A(table(["Tier"] + [NAMES[s] for s in S3], [["%s (weight %s)" % (t, t)] + ["%d/%d (%.0f%%)" % (sv[s]["tiers"][t]["strict_correct"], sv[s]["tiers"][t]["cases"], sv[s]["tiers"][t]["pct"]) for s in S3] for t in sorted(sv[S3[0]]["tiers"])]))
    A("\n*Strictly correct* = correct value AND required form AND (where required) the domain restriction reported. Value-only counts: " + ", ".join("%s %d" % (NAMES[s], sv[s]["value_only_correct"]) for s in S3) + ".\n")

    # 4
    A("## 4. Overall correctness and coverage\n")
    A(table(["Metric", *[NAMES[s] for s in S3]], [
        ["Cases defined / executed / skipped", *["%d / %d / %d" % (sv[s]["defined"], sv[s]["executed"], sv[s]["skipped"]) for s in S3]],
        ["Strictly correct (passed)", *[sv[s]["passed(strict_correct)"] for s in S3]],
        ["Invalid answers (produced, wrong)", *[sv[s]["status_counts"].get("invalid_answer", 0) for s in S3]],
        ["Unsupported: no API for the operation", *[sv[s]["unsupported_no_api"] for s in S3]],
        ["Unsupported: engine declared it unsolved", *[sv[s]["unsolved_by_solver"] for s in S3]],
        ["Runtime errors / timeouts / infra errors / input-incompatible", *["%d / %d / %d / %d" % (sv[s]["status_counts"].get("runtime_error", 0), sv[s]["status_counts"].get("timeout", 0), sv[s]["status_counts"].get("infra_error", 0), sv[s]["status_counts"].get("input_incompatible", 0)) for s in S3]],
        ["Coverage (result in the expected form)", *["%d (%.1f%%)" % (sv[s]["covered"], 100 * sv[s]["covered"] / sv[s]["defined"]) for s in S3]],
        ["Correctness (strict, difficulty-weighted) %", *[f(sv[s]["components"]["correctness"]) for s in S3]],
        ["Honesty: adversarial cases NOT answered with a confident wrong result %", *[f(sv[s]["components"]["honesty"]) for s in S3]],
        ["Dishonest adversarial cases", *[", ".join(sv[s]["honesty"]["dishonest_cases"]) or "none" for s in S3]]]))
    A("\nCoverage counts a result only if it is in the requested form; correctness is reported separately.\n")

    # 5
    A("## 5. Derivation availability and quality\n")
    A(table(["Metric", *[NAMES[s] for s in S3]], [
        ["Derivation available (cases / %d)" % n_app, *["%d (%.0f%%)" % (sv[s]["derivation"]["available"], sv[s]["derivation"]["available_pct_of_applicable"]) for s in S3]],
        ["Mean automated score /10", *[f(sv[s]["derivation"]["mean_automated"], 2) for s in S3]],
        ["Mean final score /10 (after any applied manual overlay)", *[f(sv[s]["derivation"]["mean_final"], 2) for s in S3]],
        ["Mean final, domain-sensitive cases only", *[f(sv[s]["derivation"]["mean_final_domain_sensitive"], 2) for s in S3]],
        ["Derivations with an applied manual review", *[sv[s]["derivation"]["manual_reviewed"] for s in S3]],
        *[["Mean " + k, *[f(sv[s]["derivation"]["criterion_means"].get(k), 2) for s in S3]] for k in CRIT]]))
    opaque = {s: [r["case_id"] for r in by[s] if r["derivation"].get("scored") and r["derivation"]["evidence"]["substantive_steps"] == 0] for s in S3}
    A("\nMax per criterion: correctness 3, completeness 2, explanation 2, consistency 1, domain 1, presentation 1. Opaque-only derivations (no transformation shown): " +
      "; ".join("%s %d%s" % (NAMES[s], len(opaque[s]), (" (" + ", ".join(opaque[s]) + ")") if opaque[s] else "") for s in S3) + ".\n")
    A("Score distribution (final /10 -> number of derivations): " + "; ".join("%s %s" % (NAMES[s], dict(sorted(sv[s]["derivation"]["distribution"].items()))) for s in S3) + ".\n")

    # 6
    A("## 6. Edge-case, domain and engineering behaviour\n")
    A("Engineering probes (1 = clean, 0.5 = acceptable but raw/implicit, 0 = failure, n/a = does not apply):\n")
    pids = list(eng[S3[0]].keys())
    A(table(["Probe", *[NAMES[s] for s in S3]], [[p, *[("n/a" if eng[s][p]["score"] is None else eng[s][p]["score"]) for s in S3]] for p in pids]))
    A("\nProbe outcomes that were not clean (from the raw probe notes):\n")
    for s in S3:
        for p in pids:
            if eng[s][p]["score"] is not None and eng[s][p]["score"] < 1:
                A("- **%s / %s** (%s): %s" % (NAMES[s], p, eng[s][p]["score"], cut(eng[s][p]["note"], 220)))
    A("\n`untrusted_input_canary` sends the benign payload `__import__('math').factorial(4)`; a score of 0 means the engine's own string interface evaluated it as Python. `heavy_equation` is `x^25 - x - 1 = 0` with a 15 s budget; a score of 0 means the engine did not finish and needed an external kill.\n")
    dsn = sv[S3[0]]["domain_sensitive_cases"]
    A("Domain-sensitive cases (%d): strictly correct %s.\n" % (dsn, ", ".join("%s %d" % (NAMES[s], sv[s]["domain_sensitive_strict_correct"]) for s in S3)))
    k4 = [(NAMES[r["solver"]], r["judge_detail"]) for s in S3 for r in by[s] if r["case_id"] == "K04" and r["judge_detail"]]
    A("**Specification discrepancy (K04):** the task prints \"approximately 1.167304015\", but the real root of x^5 - x - 1 is 1.1673039782614187 (mpmath 40 digits, bisection, polyroots agree); the printed value is off by 3.7e-8. The verbatim spec text is kept in `spec_expected`; the verified root is the reference (tolerance 1e-8). Per-engine: " + "; ".join("%s: %s" % (n, cut(t, 90)) for n, t in k4) + ".\n")
    A("Reference validation: %d/%d references independently validated by a second method, all_ok=%s.\n" % (ref["validated"] - len(ref["failed"]), ref["validated"], ref["all_ok"]))

    # 7
    A("## 7. Reliability and performance\n")
    A(table(["Reliability metric (0-1)", *[NAMES[s] for s in S3]], [[k, *[f(sv[s]["reliability_metrics"][k], 3) for s in S3]] for k in sv[S3[0]]["reliability_metrics"]] +
            [["**Reliability score (mean x100)**", *["**%s**" % f(sv[s]["components"]["reliability"]) for s in S3]]]))
    A("\nReliability = 100 x mean(R1 repeatability, R2 engineering probes, R3 = 1 - crashes/attempted, R4 strict-correct rate on domain-sensitive cases, R5 untrusted-input safety as shipped); see `docs/SCORING_RULES.md`.\n")
    A(table(["Performance (ms)", *[NAMES[s] for s in S3]], [
        ["Cold start: process spawn -> ready (median)", *[f(pf[s]["cold_start_ready_ms_median"], 0) for s in S3]],
        ["Cold first solve of the cold-start case (median)", *[f(pf[s]["cold_first_solve_ms_median"], 0) for s in S3]],
        ["Warm median, %d common cases (cache-cleared repeats)" % pf[S3[0]]["common_cases"], *[f(pf[s]["warm_median_common_ms"], 2) for s in S3]],
        ["Warm p95, common cases", *[f(pf[s]["warm_p95_common_ms"], 2) for s in S3]],
        ["Warm median, own supported cases", *[f(pf[s]["warm_median_own_ms"], 2) for s in S3]],
        ["Warm p95, own supported cases", *[f(pf[s]["warm_p95_own_ms"], 2) for s in S3]],
        ["Timeouts / crashes (main pass)", *["%d / %d" % (pf[s]["timeouts"], pf[s]["crashes"]) for s in S3]],
        ["Peak RSS of worker (MB)", *[f(pf[s]["peak_rss_mb"], 0) for s in S3]],
        ["**Performance score**", *["**%s**" % f(sv[s]["components"]["performance"]) for s in S3]]]))
    A("\nLatency score: 100 at <=10 ms, 0 at >=2000 ms, log-linear between; Performance = 0.4 x score(median) + 0.4 x score(p95) + 0.2 x 100 x (1 - failure rate), on the cases ALL engines answered. Cold start is reported, not scored.\n")
    if rep:
        A("Reproducibility (run 2 vs run 1): **%d/%d** rows identical in raw output, **%d** verdict differences, **%d** derivation-score differences. Median warm latency run1 -> run2: %s.\n" %
          (rep["rows_compared"] - rep["raw_output_differences"], rep["rows_compared"], rep["verdict_differences"], rep["derivation_score_differences"],
           "; ".join("%s %s -> %s ms" % (NAMES[k], v[0], v[1]) for k, v in rep["timing_median_ms_run1_vs_run2"].items())))

    # 8
    A("## 8. Weighted scores and hard gates\n")
    LAB = {"correctness": "Correctness (difficulty-weighted)", "derivations": "Derivation quality", "coverage": "Coverage", "honesty": "Honesty on adversarial cases",
           "reliability": "Reliability", "performance": "Performance", "integration": "Integration/maintainability"}
    A(table(["Component (weight)", *[NAMES[s] for s in S3]], [["%s (%d%%)" % (LAB[k], round(100 * w)), *[f(sv[s]["components"][k]) for s in S3]] for k, w in sc["weights"].items()] +
            [["**Overall /100**", *["**%s**" % f(sv[s]["overall"]) for s in S3]]]))
    A("\n`Overall = " + " + ".join("%.2f*%s" % (w, k.capitalize()) for k, w in sc["weights"].items()) + "`. The derivation component is the mean score over *available* derivations x 10, as specified; the availability-penalised variant is in the sensitivity table.\n")
    A("**Integration evidence (computed):**\n")
    for s in S3:
        A("- **%s** (%d/14): " % (NAMES[s], sv[s]["integration"]["points"]) + "; ".join("%s=%d (%s)" % (k, v["score"], cut(v["evidence"], 110)) for k, v in sv[s]["integration"]["criteria"].items()))
    A("\n**Sensitivity (ranking stability):**\n")
    vs = list(sv[S3[0]]["sensitivity"].keys())
    A(table(["Variant", *[NAMES[s] for s in S3], "Order"], [[v, *[f(sv[s]["sensitivity"][v]) for s in S3], " > ".join(NAMES[s] for s in sorted(S3, key=lambda s: -sv[s]["sensitivity"][v]))] for v in vs]))
    A("\n**Hard gates** (a weighted score never overrides these):\n")
    A(table(["Gate", *[NAMES[s] for s in S3]], [
        ["G1 critical correctness defect (confidently wrong answer)", *[g[s]["G1_critical_correctness"]["status"] + (" " + ", ".join(c for c, _ in g[s]["G1_critical_correctness"]["critical_cases"]) if g[s]["G1_critical_correctness"]["critical_cases"] else "") for s in S3]],
        ["G2 invalid / self-contradicting derivation (FAIL >=2 cases, CONDITIONAL 1)", *[g[s]["G2_fabricated_or_invalid_derivations"]["status"] + (" " + ", ".join(g[s]["G2_fabricated_or_invalid_derivations"]["cases"]) if g[s]["G2_fabricated_or_invalid_derivations"]["cases"] else "") for s in S3]],
        ["G3 untrusted input executed as code (as shipped)", *[g[s]["G3_unsafe_execution"]["status"] for s in S3]],
        ["G4 licence", *[g[s]["G4_licence"]["status"] + " (" + str(g[s]["G4_licence"]["licence"]) + ")" for s in S3]],
        ["**Eligible for production recommendation**", *["**" + g[s]["eligible_for_production_recommendation"] + "**" for s in S3]]]))
    A("\nGate evidence:\n")
    for s in S3:
        for k in ("G1_critical_correctness", "G2_fabricated_or_invalid_derivations", "G3_unsafe_execution"):
            gg = g[s][k]
            if gg["status"] == "PASS":
                continue
            if k.startswith("G1"):
                ev = "; ".join("%s: %s" % (c, cut(t, 80)) for c, t in gg["critical_cases"])
            elif k.startswith("G2"):
                ev = "; ".join("%s: %s" % (c, cut(t, 120)) for c, t in gg["evidence"][:6])
            else:
                ev = cut(gg["note"], 200)
            A("- **%s %s = %s**: %s" % (NAMES[s], k[:2], gg["status"], ev))
    A("\nG3 is CONDITIONAL for SymPy-direct by definition: only its *string* parser evaluates code, it offers programmatic construction, and this benchmark's allow-list adapter blocks the payload.\n")

    # 9
    A("## 9. Strongest and weakest areas (computed)\n")
    for s in S3:
        o, cs = sv[s], sv[s]["by_category"]
        full = [c for c, v in cs.items() if v["strict_correct"] == v["n"]]
        zero = [c for c, v in cs.items() if v["strict_correct"] == 0]
        partial = sorted([(v["strict_correct"] / v["n"], c) for c, v in cs.items() if 0 < v["strict_correct"] < v["n"]])
        fails = defaultdict(list)
        for r in by[s]:
            if r["status"] == "invalid_answer":
                fails["invalid answer"].append(r["case_id"])
            elif r["status"] == "runtime_error":
                fails["runtime error"].append(r["case_id"])
            elif r["status"] == "unsupported":
                fails["unsupported (%s)" % ("no API" if r["status_detail"] == "no_api" else "declared unsolved")].append(r["case_id"])
            elif r["status"] == "valid" and not r["strict_correct"]:
                fails["right value, domain/form requirement missed"].append(r["case_id"])
        dv = o["derivation"]
        low = [p for p in pids if eng[s][p]["score"] is not None and eng[s][p]["score"] < 1]
        A("**%s** (overall %.1f)\n" % (NAMES[s], o["overall"]))
        A("- Strongest: %s; strict %d/%d, coverage %d/%d, derivations available %d/%d (mean %.2f/10), warm median %.1f ms." % (
            ("all cases correct in categories " + ", ".join(full)) if full else "no category fully correct", o["passed(strict_correct)"], o["defined"], o["covered"], o["defined"],
            dv["available"], o["defined"], dv["mean_final"], pf[s]["warm_median_common_ms"] or 0))
        A("- Weakest: %s%s." % (("no correct answers in categories " + ", ".join(zero)) if zero else "no category entirely wrong",
                                ("; lowest partial categories: " + ", ".join("%s (%.0f%%)" % (c, 100 * p) for p, c in partial[:3])) if partial else ""))
        A("- Not-passed cases by type: " + ("; ".join("%s: %s" % (k, ", ".join(v)) for k, v in fails.items()) if fails else "none") + ".")
        A("- Derivation concerns: %d opaque-only, %d with auto-detected invalid steps, %d with override/untied workings." % (
            len(opaque[s]), sum(1 for r in by[s] if r["derivation"].get("scored") and r["derivation"]["evidence"]["invalid_steps"] > 0),
            sum(1 for r in by[s] if (r.get("native_flags") or {}).get("steps_mode") == "direct_evaluation" or (r.get("native_flags") or {}).get("workings_tied_to_answer") is False)))
        A("- Engineering probes below 1.0: %s; gates not PASS: %s.\n" % (", ".join("%s (%s)" % (p, eng[s][p]["score"]) for p in low) or "none",
                                                                        ", ".join("%s=%s" % (k[:2], v["status"]) for k, v in g[s].items() if k.startswith("G") and v["status"] != "PASS") or "none"))

    # 10
    A("## 10. Recommendation (rule-based, with limits)\n")
    elig = {s: g[s]["eligible_for_production_recommendation"] for s in S3}
    order = rk["final_rank"] if rk else sorted(S3, key=lambda s: -sv[s]["overall"])
    A("Decision rules (fixed before any run): (1) an engine with any hard-gate FAIL is not recommended as-is; (2) among the rest, take the highest-ranked engine from the leaderboard above (score, then the declared tie-breaks); (3) where the bootstrap says two engines are not statistically separable, the choice between them rests on gates, derivation availability and the failures listed above, not on the score gap; (4) a gate-failing engine can still be piloted behind a validation wrapper if the blocking defects are listed and fixed first.\n")
    A(table(["Rank", "Engine", "Overall", "Eligible as-is", "Blocking gates"], [[i + 1, NAMES[s], f(sv[s]["overall"]), elig[s], ", ".join("%s=%s" % (k[:2], v["status"]) for k, v in g[s].items() if k.startswith("G") and v["status"] in ("FAIL", "CONDITIONAL")) or "none"] for i, s in enumerate(order)]))
    ok = [s for s in order if elig[s] == "YES"]
    cond = [s for s in order if elig[s] == "CONDITIONAL"]
    top = order[0]
    A("")
    if ok:
        pick = ok[0]
        A("**Recommended as-is:** %s (highest-scoring engine with no failed or conditional gate; overall %.1f)." % (NAMES[pick], sv[pick]["overall"]))
    elif cond:
        pick = cond[0]
        A("**Recommended with conditions:** %s (highest-scoring engine with no FAIL gate; overall %.1f). Conditions: %s." % (NAMES[pick], sv[pick]["overall"],
          "; ".join("%s=%s" % (k[:2], v["status"]) for k, v in g[pick].items() if k.startswith("G") and v["status"] == "CONDITIONAL")))
    else:
        pick = None
        A("**No engine is eligible as-is.**")
    if top != pick:
        gap = (sv[top]["overall"] - sv[pick]["overall"]) if pick else None
        reasons = []
        for k, v in g[top].items():
            if k.startswith("G") and v["status"] in ("FAIL", "CONDITIONAL"):
                ids = v.get("cases") or [c for c, _ in v.get("critical_cases", [])]
                reasons.append("%s = %s%s" % (k[:2], v["status"], (" (" + ", ".join(ids) + ")") if ids else ""))
        A("")
        A("**Highest weighted score:** %s (%.1f)%s. It is not eligible as-is because: %s." % (NAMES[top], sv[top]["overall"], (", %.1f points above the recommended engine" % gap) if pick else "", "; ".join(reasons)))
        A("Piloting it behind a validation wrapper is reasonable only after the listed failures are fixed and this notebook is re-run to confirm the gates pass.")
    A("\nFrom the measured data, the practical split of responsibilities is: use the engine with the best strict correctness/coverage as the *answer* source, and treat an engine's derivation as shippable only where it is available, scored, and free of invalid/contradictory steps (see failure_analysis.md sections 2 and 3). Where derivations are opaque or unavailable, show the verified answer and label the derivation unavailable instead of showing a placeholder.\n")
    A("### Limitations of this benchmark\n")
    A("- 72 cases are one sample of school/early-university mathematics; this does **not** show that any engine supports all of mathematics or handles word problems, images, or free-form phrasing.\n"
      "- Derivation scores are proxy rubrics plus a model-assigned overlay (for earlier outputs only); human audit is required. Completeness uses regex milestones.\n"
      "- Latency is hardware-dependent and a small part of the score (5%); warm latency is measured cache-cleared in a warm process.\n"
      "- K03/K04 (numeric solving) use each engine's nearest documented API (SymPy `nsolve`/`nroots`; engines without a numeric root-finder use their generic equation solver). SmartSolver `limit` uses its documented one-sided default.\n"
      "- StepSolver `ln` is adapted to `log`; the as-specified result is preserved in `native_flags.as_specified_variant`.\n"
      "- Adapters map to the public API names that existed when the benchmark was written; a renamed/removed method is reported as unsupported (SmartSolver) or an infrastructure error (import failure), never silently substituted.\n"
      "- Offline operation was assessed only by static search for network imports in each library core (`engineering_results.json`).\n")
    A("### Artefacts\n")
    A("`results/raw_results.json|csv`, `results/raw/*.json` (verbatim worker replies), `results/scores.json`, `results/engineering_results.json`, `results/reference_validation.json`, `results/reproducibility_check.json`, `results/failure_analysis.md`, `results/changes_vs_baseline.md` (if a baseline is present), `data/cases.json`, `docs/`.\n")

    # 11 baseline
    base_path = ROOT / "data/baseline.json"
    if base_path.exists():
        base = json.load(open(base_path))
        B = []
        B.append("# Changes vs the baseline run\n")
        B.append("Baseline: **%s**.\n" % base["label"])
        B.append(table(["Engine", "Baseline version", "This run", "Baseline overall", "This overall", "Delta"], [
            [NAMES[s], "%s %s" % (base["engines"].get(s, {}).get("version"), (base["engines"].get(s, {}).get("commit") or "")[:8]), "%s %s" % (meta[s]["version"], (meta[s]["commit"] or "")[:8]),
             f(base["overall"].get(s)), f(sv[s]["overall"]), f(sv[s]["overall"] - base["overall"][s]) if s in base["overall"] else "n/a"] for s in S3]))
        B.append("\n" + table(["Component"] + ["%s (base -> now)" % NAMES[s] for s in S3], [[k] + ["%s -> %s" % (f(base["components"].get(s, {}).get(k)), f(sv[s]["components"][k])) for s in S3] for k in sv[S3[0]]["components"]]))
        reg, imp, other = [], [], []
        for r in rows:
            b = base["cases"].get("%s|%s" % (r["solver"], r["case_id"]))
            if not b:
                continue
            if b["strict_correct"] and not r["strict_correct"]:
                reg.append((r["solver"], r["case_id"], b["status"], r["status"], r["status_detail"]))
            elif (not b["strict_correct"]) and r["strict_correct"]:
                imp.append((r["solver"], r["case_id"], b["status"], r["status"]))
            elif b["status"] != r["status"]:
                other.append((r["solver"], r["case_id"], b["status"], r["status"]))
        B.append("\n## Regressions (strictly correct before, not now): %d\n" % len(reg))
        B.append(table(["Engine", "Case", "Before", "Now", "Detail"], [[NAMES[a], b, c, dd, cut(e, 90)] for a, b, c, dd, e in reg]) if reg else "None.")
        B.append("\n## Improvements (not strictly correct before, correct now): %d\n" % len(imp))
        B.append(table(["Engine", "Case", "Before", "Now"], [[NAMES[a], b, c, dd] for a, b, c, dd in imp]) if imp else "None.")
        B.append("\n## Other status changes: %d\n" % len(other))
        B.append(table(["Engine", "Case", "Before", "Now"], [[NAMES[a], b, c, dd] for a, b, c, dd in other]) if other else "None.")
        B.append("\n*Derivation scores that relied on the earlier manual overlay fall back to automated scores when output changed; compare derivation means with that in mind.*\n")
        (RES / "changes_vs_baseline.md").write_text("\n".join(B) + "\n")
        A("## 11. Changes vs the baseline\n")
        A("See `results/changes_vs_baseline.md`: %d regressions, %d improvements, %d other status changes versus *%s*.\n" % (len(reg), len(imp), len(other), base["label"]))
    (RES / "summary.md").write_text("\n".join(L) + "\n")

    # ------------------------------------------------------------------ failure analysis
    F = []
    Z = F.append
    Z("# Failure analysis (generated from this run's data)\n")
    Z("Reproduce any row after setup: `envs/judge/bin/python bench/runner.py --solver <name> --case <ID> --no-timing --out /tmp/x`, then inspect `/tmp/x/<name>.json` (raw reply: `api_call`, `input_sent`, `raw_output`, `steps`, traceback tail). Verdicts: `results/raw_results.json` (`status`, `status_detail`, `judge_detail`).\n")

    def rowline(r):
        c = cases[r["case_id"]]
        return "| %s | %s | `%s` | %s | `%s` | `%s` | %s |" % (r["case_id"], NAMES[r["solver"]], cut(r["display_input"], 44), cut(c["spec_expected"], 40), cut(r["actual"], 60), cut(r.get("adapter_api_call"), 70), cut(r["status_detail"] or r["status"], 80))

    hdr = "| Case | Engine | Input | Expected (spec) | Actual | Call | Verdict |\n|---|---|---|---|---|---|---|"
    Z("## 1. Incorrect answers (and values correct but not strictly correct)\n")
    inc = [r for s in S3 for r in by[s] if r["status"] == "invalid_answer" or (r["status"] == "valid" and not r["strict_correct"])]
    Z(hdr + "\n" + "\n".join(rowline(r) for r in inc) if inc else "None.")
    Z("\nRows with status `valid` but not strictly correct have the right value and miss a declared domain-reporting or form requirement.\n")
    Z("## 2. Missing or invalid intermediate steps\n")
    for s in S3:
        Z("- **%s** opaque/structural-only derivations (%d): %s" % (NAMES[s], len(opaque[s]), ", ".join(opaque[s]) or "none"))
        unavailable = [r["case_id"] for r in by[s] if r["answer_produced"] and not r["derivation"].get("available")]
        Z("  - answered but no derivation at all (%d): %s" % (len(unavailable), (", ".join(unavailable[:40]) + (" ..." if len(unavailable) > 40 else "")) or "none"))
    Z("")
    inv = [(r, r["derivation"]["evidence"]) for s in S3 for r in by[s] if r["derivation"].get("scored") and r["derivation"]["evidence"]["invalid_steps"] > 0]
    Z("**Automatically detected invalid/misrendered steps** (consecutive-step equivalence check; a displayed step that is not equivalent to the one before it):\n")
    Z(table(["Engine", "Case", "Evidence"], [[NAMES[r["solver"]], r["case_id"], cut(e["correctness"], 180)] for r, e in inv]) if inv else "None detected.")
    ov = [r for s in S3 for r in by[s] if (r.get("native_flags") or {}).get("steps_mode") == "direct_evaluation"]
    Z("\n**Derivations replaced by a direct-evaluation override (steps do not produce the answer):** " + (", ".join("%s %s" % (NAMES[r["solver"]], r["case_id"]) for r in ov) or "none") + ".")
    untied = [r for s in S3 for r in by[s] if (r.get("native_flags") or {}).get("workings_tied_to_answer") is False]
    Z("**Workings from a code path separate from the answer:** " + (", ".join("%s %s" % (NAMES[r["solver"]], r["case_id"]) for r in untied) or "none") + ".")
    for s in S3:
        flags = [r for r in by[s] if r["answer_produced"] and (r.get("native_flags") or {}).get("steps_verified") is not None]
        if flags and all(r["native_flags"]["steps_verified"] for r in flags) and any(r["status"] == "invalid_answer" for r in by[s]):
            Z("- **%s** reports `steps_verified=True` on all %d answered cases, including %d that are invalid: the flag carries no information." % (NAMES[s], len(flags), sum(1 for r in by[s] if r["status"] == "invalid_answer")))
    contra = [(r["solver"], r["case_id"], r["derivation"]["manual"]["justification"]) for s in S3 for r in by[s] if r["derivation"].get("manual") and any(w in r["derivation"]["manual"]["justification"] for w in ("CONTRADICTION", "ORIGINAL expression", "Displayed first step"))]
    if contra:
        Z("\n**Manual-review findings (overlay applied; model-assigned, needs audit):**\n")
        for a, b, c in contra:
            Z("- %s %s: %s" % (NAMES[a], b, cut(c, 400)))
    Z("")
    Z("## 3. Domain and extraneous-solution errors\n")
    dom = [r for s in S3 for r in by[s] if r["status"] in ("invalid_answer", "valid") and ((r["domain_ok"] is False) or any(w in (r["status_detail"] or "") for w in ("domain", "real_domain_only", "partial", "claims no solution", "claims every x")))]
    Z(hdr + "\n" + "\n".join(rowline(r) for r in dom) if dom else "None detected.")
    Z("")
    Z("## 4. Unsupported operations\n")
    for s in S3:
        na = defaultdict(list)
        un = []
        for r in by[s]:
            if r["status"] == "unsupported":
                if r["status_detail"] == "no_api":
                    na[r["category"]].append(r["case_id"])
                else:
                    un.append("%s (%s)" % (r["case_id"], cut(r["unsupported_reason"], 70)))
        Z("- **%s** no API: %s" % (NAMES[s], "; ".join("%s: %s" % (k, ", ".join(v)) for k, v in sorted(na.items())) or "none"))
        Z("  - declared unsolved by the engine: %s" % (", ".join(un) or "none"))
    Z("")
    Z("## 5. Parsing / input-format incompatibilities\n")
    pe = [r for s in S3 for r in by[s] if r["status"] == "input_incompatible"]
    Z("Input-incompatible results (engine rejected the adapted syntax): " + (", ".join("%s %s (%s)" % (NAMES[r["solver"]], r["case_id"], cut((r.get("error") or {}).get("message"), 60)) for r in pe) or "none") + ".\n")
    adp = [r for s in S3 for r in by[s] if r["adapter_overrides"]]
    Z("Cases that needed a documented adapter override/normalisation/composition: " + (", ".join("%s %s" % (NAMES[r["solver"]], r["case_id"]) for r in adp) or "none") + ".")
    for r in adp:
        v = (r.get("native_flags") or {}).get("as_specified_variant")
        if v:
            Z("- %s %s as specified (`%s`) -> %s: %s" % (NAMES[r["solver"]], r["case_id"], v["query"], v["result_type"], cut(v["raw_output"], 160)))
    Z("")
    for s in S3:
        for p in ("malformed_syntax", "unbalanced_parens", "empty_input", "unsupported_operation"):
            if eng[s][p]["score"] == 0:
                Z("- **%s / %s**: %s" % (NAMES[s], p, cut(eng[s][p]["note"], 200)))
    Z("")
    Z("## 6. Runtime and dependency failures\n")
    for s in S3:
        e = evidence.get(s, {})
        if not e.get("installed", True):
            Z("- **%s INSTALL FAILED**: %s" % (NAMES[s], cut(e.get("install_error"), 300)))
    rt = [r for s in S3 for r in by[s] if r["status"] in ("runtime_error", "timeout", "infra_error")]
    for r in rt:
        er = r.get("error") or {}
        Z("- **%s %s** %s: `%s: %s`  (call: `%s`)" % (NAMES[r["solver"]], r["case_id"], r["status"], er.get("type"), cut(er.get("message"), 140), cut(r.get("adapter_api_call"), 80)))
    if not rt:
        Z("- No runtime errors, timeouts or infrastructure failures in the main pass.")
    for s in S3:
        for p in ("heavy_equation", "large_polynomial_equation", "untrusted_input_canary", "zero_denominator"):
            if eng[s][p]["score"] is not None and eng[s][p]["score"] == 0:
                Z("- **%s / %s**: %s" % (NAMES[s], p, cut(eng[s][p]["note"], 220)))
    Z("")
    Z("## 7. Cases the evaluator cannot judge reliably (manual-review flags)\n")
    fl = [r for s in S3 for r in by[s] if r["manual_review_flag"]]
    Z(table(["Engine", "Case", "Reason"], [[NAMES[r["solver"]], r["case_id"], cut("; ".join(r["manual_review_reasons"]), 150)] for r in fl]) if fl else "None.")
    Z("\n- Restriction reporting is decided by text matching of reported constraints.\n- K03/K04 use the closest documented API; K04's reference differs from the specification's printed digits (specification typo).\n- Derivation completeness/explanation are regex/word-count proxies; a manual overlay applies only to the exact output it reviewed (%d applied, %d stale and ignored).\n- Expression equivalence is symbolic where possible, otherwise numeric at exact rational sample points (flagged when numeric-only).\n" % (reviewed_total, stale))
    (RES / "failure_analysis.md").write_text("\n".join(F) + "\n")
    print("wrote summary.md, failure_analysis.md" + (", changes_vs_baseline.md" if base_path.exists() else ""))


if __name__ == "__main__":
    main()
''')
print('wrote', 6, 'files')

wrote 6 files


In [7]:
#@title 3e. README { display-mode: "form" }
W("README.md", """# Math-solver benchmark (generated by the Colab notebook)

Layout: data/cases.json (89 hard cases in 3 difficulty tiers, generated by tools/make_cases.py), data/manual_reviews.json (derivation overlay, fingerprinted; empty for this suite),
data/baseline.json (original run, for the change report), bench/ (workers, runner, judge, derivations, engineering, score, report), docs/, pins/, results/.
Start with results/summary.md (leaderboard first), results/ranking.md and results/failure_analysis.md (and results/changes_vs_baseline.md).
Re-run: open the notebook, set the engine versions in the configuration cell, Run all.
""")
print("ok")

ok


In [8]:
#@title 4. Clone, install (isolated environments) and record evidence { display-mode: "form" }
# ---- clone candidates, build isolated environments, record evidence ---------------------------------------------
import tomllib
(ROOT / "results/evidence").mkdir(parents=True, exist_ok=True)
(ROOT / "pins").mkdir(exist_ok=True)

def clone(name, url, ref):
    d = ROOT / "candidates" / name
    if not (d / ".git").exists():
        sh(["git", "clone", "--quiet", url, str(d)])
    sh(["git", "-C", str(d), "fetch", "--quiet", "--tags", "origin"], check=False)
    if ref and ref != "HEAD":
        sh(["git", "-C", str(d), "checkout", "--quiet", ref])
    else:
        sh(["git", "-C", str(d), "checkout", "--quiet", "-"], check=False)
        sh(["git", "-C", str(d), "pull", "--quiet", "--ff-only"], check=False)
    return d

def mkenv(name, packages, pin_note=None):
    env = ROOT / "envs" / name
    if env.exists():
        shutil.rmtree(env)
    sh([UV, "venv", "--quiet", "--python", PYTHON_VERSION, str(env)])
    py = str(env / "bin" / "python")
    t = time.time()
    r = sh([UV, "pip", "install", "--quiet", "--python", py] + packages, check=False)
    return py, r, time.time() - t

def licence_of(d):
    for f in sorted(d.glob("LICENSE*")) + sorted(d.glob("LICENCE*")) + sorted(d.glob("COPYING*")):
        t = f.read_text(errors="ignore")
        for key, name, cls in [(r"MIT License|Permission is hereby granted, free of charge", "MIT", "permissive"), (r"Apache License", "Apache-2.0", "permissive"),
                               (r"BSD|Redistribution and use in source and binary forms", "BSD", "permissive"), (r"\bISC License", "ISC", "permissive"),
                               (r"GNU AFFERO", "AGPL", "copyleft"), (r"GNU LESSER", "LGPL", "copyleft"), (r"GNU GENERAL PUBLIC", "GPL", "copyleft"), (r"Mozilla Public License", "MPL", "copyleft")]:
            if re.search(key, t):
                return name, cls
    return "unknown (no recognised LICENSE file)", "unknown"

def history_of(d):
    def g(*a):
        return sh(["git", "-C", str(d)] + list(a), check=False).stdout.strip()
    try:
        first = int(g("log", "--reverse", "--format=%ct").splitlines()[0]); last = int(g("log", "-1", "--format=%ct"))
        return {"commits": int(g("rev-list", "--count", "HEAD")), "authors": len(set(g("log", "--format=%an").splitlines())), "tags": len(g("tag").split()),
                "age_days": int((time.time() - first) / 86400), "days_since_last_commit": int((time.time() - last) / 86400)}
    except Exception as e:
        return {"error": str(e)}

def deps_count(py):
    r = sh([UV, "pip", "list", "--python", py, "--format", "json"], check=False)
    try:
        return len(json.loads(r.stdout)) - 1
    except Exception:
        return None

def static_tests(d):
    n = 0
    for tdir in list(d.rglob("tests")) + list(d.rglob("test")):
        if tdir.is_dir() and ".git" not in tdir.parts:
            for f in tdir.rglob("*.py"):
                n += len(re.findall(r"^\s*(?:async\s+)?def test_", f.read_text(errors="ignore"), re.M))
    return n

def pyproject_info(d):
    p = d / "pyproject.toml"
    if not p.exists():
        return {}
    try:
        return tomllib.loads(p.read_text())
    except Exception:
        return {}

evidence = {}
SY = [SYMPY_SPEC, "mpmath"]

# judge env (fixed SymPy, used by the judge and the reference validation only)
jpy, r, _ = mkenv("judge", [JUDGE_SYMPY_SPEC, "mpmath"])
assert r.returncode == 0, "judge env failed: " + r.stderr[-500:]

# standalone SymPy
py, r, secs = mkenv("sympy", SY)
if r.returncode == 0:
    md_ = sh([py, "-c", "import importlib.metadata as m; x=m.metadata('sympy'); print((x.get('License-Expression') or x.get('License') or '').strip()); print(x.get('Requires-Python') or '')"]).stdout.splitlines()
    lic = md_[0] if md_ else ""
    cls = "permissive" if re.search(r"BSD|MIT|Apache", lic) else "unknown"
    evidence["sympy-direct"] = {"installed": True, "install_seconds": secs, "runtime_deps": deps_count(py), "python_requires": md_[1] if len(md_) > 1 else None,
                                "licence": lic or "BSD-3-Clause (not in metadata)", "licence_class": cls if lic else "permissive", "source": "PyPI: " + SYMPY_SPEC, "commit": None,
                                "tests": {"static_test_functions": None, "own_suite": {"run": False}}}
else:
    evidence["sympy-direct"] = {"installed": False, "install_error": r.stderr[-600:]}

def candidate(key, name, url, ref):
    try:
        d = clone(name, url, ref)
    except Exception as e:
        evidence[key] = {"installed": False, "install_error": "clone/checkout failed: %s" % e, "source": "%s @ %s" % (url, ref)}
        return
    commit = sh(["git", "-C", str(d), "rev-parse", "HEAD"]).stdout.strip()
    pin = [SYMPY_SPEC, "mpmath"] if PIN_SYMPY_IN_CANDIDATES else []
    py, r, secs = mkenv(key, [str(d)] + pin)
    fallback = False
    if r.returncode != 0 and pin:
        py, r, secs = mkenv(key, [str(d)])
        fallback = r.returncode == 0
    lic, cls = licence_of(d)
    pp = pyproject_info(d)
    proj = pp.get("project", {})
    src = d / "src" if (d / "src").exists() else d
    evidence[key] = {"installed": r.returncode == 0, "install_error": None if r.returncode == 0 else r.stderr[-800:], "sympy_pin_fallback": fallback,
                     "install_seconds": secs, "commit": commit, "url": url, "ref": ref, "source": "%s @ %s" % (url, commit[:12]), "licence": lic, "licence_class": cls,
                     "python_requires": proj.get("requires-python"), "runtime_deps": deps_count(py) if r.returncode == 0 else None, "history": history_of(d),
                     "tests": {"static_test_functions": static_tests(d), "own_suite": {"run": False}}, "source_dir": str(src), "checkout_dir": str(d)}

candidate("smartsolver", "smartsolver", SMARTSOLVER_URL, SMARTSOLVER_REF)
candidate("stepsolver", "stepsolver", STEPSOLVER_URL, STEPSOLVER_REF)

for k in ("sympy", "smartsolver", "stepsolver", "judge"):
    p = ROOT / "envs" / k / "bin" / "python"
    if p.exists():
        (ROOT / "pins" / ("requirements-%s.txt" % k)).write_text(sh([UV, "pip", "freeze", "--python", str(p)], check=False).stdout)

# optional: candidates' own test suites (separate throw-away venvs; never mixed into the benchmark envs)
if RUN_OWN_TEST_SUITES:
    for key in ("smartsolver", "stepsolver"):
        ev = evidence[key]
        if not ev.get("installed"):
            continue
        d = pathlib.Path(ev["checkout_dir"])
        pp = pyproject_info(d)
        extra = ["pytest"]
        for item in (pp.get("dependency-groups", {}) or {}).get("dev", []):
            name = re.split(r"[ (<>=!~\[]", str(item).strip(), 1)[0]
            if name.startswith("pytest") or name.startswith("httpx") or name == "hypothesis":
                extra.append(re.sub(r"[ ()]", "", str(item)))
        py, r, _ = mkenv("dev-" + key, [str(d)] + sorted(set(extra)))
        if r.returncode:
            ev["tests"]["own_suite"] = {"run": True, "passed": 0, "failed": "n/a", "note": "dev env failed: " + r.stderr[-300:]}
            continue
        t = sh([py, "-m", "pytest", "tests", "-q", "-p", "no:cacheprovider", "-o", "addopts="], check=False, timeout=1500, cwd=str(d))
        out = t.stdout + t.stderr
        m_pass = re.search(r"(\d+) passed", out); m_fail = re.search(r"(\d+) (?:failed|error)", out)
        ev["tests"]["own_suite"] = {"run": True, "passed": int(m_pass.group(1)) if m_pass else 0, "failed": int(m_fail.group(1)) if m_fail else 0, "tail": out[-300:]}
        print(key, "own suite:", ev["tests"]["own_suite"]["passed"], "passed /", ev["tests"]["own_suite"]["failed"], "failed")

(ROOT / "results/evidence/candidates.json").write_text(json.dumps(evidence, indent=1))
cpu = ""
try:
    cpu = [l for l in open("/proc/cpuinfo") if l.startswith("model name")][0].split(":", 1)[1].strip()
except Exception:
    pass
env_txt = "date: %s\nos: %s\ncpu: %d core(s), %s\nmem: %s\npython (notebook): %s  uv: %s\nengine python: %s\nexperiment config: SYMPY_SPEC=%s SMARTSOLVER_REF=%s STEPSOLVER_REF=%s JUDGE=%s" % (
    time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), platform.platform(), os.cpu_count(), cpu,
    sh("free -m | awk '/Mem/{print $2\" MB\"}'", check=False).stdout.strip(), platform.python_version(), sh([UV, "--version"]).stdout.strip(), PYTHON_VERSION,
    SYMPY_SPEC, SMARTSOLVER_REF, STEPSOLVER_REF, JUDGE_SYMPY_SPEC)
(ROOT / "results/evidence/environment.txt").write_text(env_txt + "\n")
print(env_txt, "\n")
for k, v in evidence.items():
    print("%-13s installed=%s commit=%s deps=%s licence=%s" % (k, v.get("installed"), (v.get("commit") or "PyPI")[:10], v.get("runtime_deps"), v.get("licence")))
    if not v.get("installed"):
        print("   INSTALL FAILED:", (v.get("install_error") or "")[-300:])

date: 2026-11-04T12:15:30Z
os: Linux-6.6.159+-x86_64-with-glibc2.39
cpu: 2 core(s), Intel(R) Xeon(R) CPU @ 2.20GHz
mem: 12975 MB
python (notebook): 3.13.18  uv: uv 0.13.5 (x86_64-unknown-linux-gnu)
engine python: 3.12
experiment config: SYMPY_SPEC=sympy SMARTSOLVER_REF=HEAD STEPSOLVER_REF=main JUDGE=sympy==1.14.0 

sympy-direct  installed=True commit=PyPI deps=1 licence=BSD
smartsolver   installed=True commit=f0d46a1089 deps=2 licence=MIT
stepsolver    installed=True commit=d965b46ee2 deps=2 licence=MIT


In [9]:
#@title 5. Dataset (89 hard cases) and reference validation { display-mode: "form" }
# ---- dataset + independent validation of every reference answer -------------------------------------------------
JPY = str(ROOT / "envs/judge/bin/python")
run_stream([sys.executable, "tools/make_cases.py"])
run_stream([JPY, "bench/reference_validation.py"], check=False)
ref = json.load(open(ROOT / "results/reference_validation.json"))
print("references validated: %d/%d  all_ok=%s  failed=%s" % (ref["validated"] - len(ref["failed"]), ref["validated"], ref["all_ok"], ref["failed"]))

wrote /content/math_solver_hard_suite/data/cases.json cases: 89 tiers: {1: 37, 2: 42, 3: 10} weight total: 151
validated 89 cases | all_ok = True | failed: [] | missing: []
references validated: 89/89  all_ok=True  failed=[]


In [10]:
#@title 6. Run the benchmark (run 1) { display-mode: "form" }
# ---- run 1 (functional pass, cold-start trials, warm repeats); raw replies are preserved verbatim -------------------
WORKER_ENV = {"sympy-direct": "sympy", "smartsolver": "smartsolver", "stepsolver": "stepsolver"}
installed = [s for s, e in WORKER_ENV.items() if json.load(open(ROOT / "results/evidence/candidates.json")).get(s, {}).get("installed")]
print("engines to run:", installed)
if (ROOT / "results/raw").exists():
    shutil.rmtree(ROOT / "results/raw")
run_stream([JPY, "bench/runner.py", "--solver", *installed, "--repeats", str(REPEATS), "--timeout", str(TIMEOUT_S), "--out", "results/raw"])

engines to run: ['sympy-direct', 'smartsolver', 'stepsolver']
cases loaded: 89 (dataset total 89)
sympy-direct   46.1s  outcomes={'answer': 87, 'error': 2}  maxrss=66MB timeouts=0 crashes=0
smartsolver    50.2s  outcomes={'unsupported': 44, 'answer': 45}  maxrss=66MB timeouts=0 crashes=0
stepsolver     83.4s  outcomes={'answer': 88, 'error': 1}  maxrss=72MB timeouts=0 crashes=0


0

In [11]:
#@title 7. Evaluate, engineering probes, scoring, ranking { display-mode: "form" }
# ---- judge + derivation rubric -> raw_results; engineering probes; weighted scores and hard gates ------------------
run_stream([JPY, "bench/evaluate.py"])
run_stream([JPY, "bench/engineering.py"])
run_stream([JPY, "bench/score.py"])
run_stream([JPY, "bench/rank.py"])

rows: 267 -> /content/math_solver_hard_suite/results/raw_results.json
sympy-direct  {'valid': 85, 'invalid_answer': 4} strict_correct=84 covered=87
smartsolver   {'unsupported': 44, 'invalid_answer': 2, 'valid': 43} strict_correct=43 covered=45
stepsolver    {'valid': 88, 'invalid_answer': 1} strict_correct=88 covered=89
sympy-direct {'malformed_syntax': 0.5, 'unbalanced_parens': 0.5, 'empty_input': 0.5, 'zero_denominator': 0.5, 'missing_variable': 0.5, 'two_variable_parametric': 1.0, 'unsupported_operation': None, 'heavy_equation': 0.0, 'large_polynomial_equation': 1.0, 'untrusted_input_canary': 0.0}
smartsolver {'malformed_syntax': 0.5, 'unbalanced_parens': 0.5, 'empty_input': 0.5, 'zero_denominator': 0.5, 'missing_variable': 0.5, 'two_variable_parametric': 1.0, 'unsupported_operation': None, 'heavy_equation': 0.0, 'large_polynomial_equation': 1.0, 'untrusted_input_canary': 0.0}
stepsolver {'malformed_syntax': 1.0, 'unbalanced_parens': 1.0, 'empty_input': 1.0, 'zero_denominator': 1.0

0

In [12]:
#@title 8. Reproducibility run (run 2) { display-mode: "form" }
# ---- second full run -> reproducibility check ------------------------------------------------------------------
rp = ROOT / "results/reproducibility_check.json"
if rp.exists():
    rp.unlink()
if RUN_TWICE:
    if (ROOT / "results/raw_run2").exists():
        shutil.rmtree(ROOT / "results/raw_run2")
    run_stream([JPY, "bench/runner.py", "--solver", *installed, "--repeats", str(REPEATS), "--timeout", str(TIMEOUT_S), "--out", "results/raw_run2"])
    run_stream([JPY, "bench/compare_runs.py", "results/raw", "results/raw_run2", "results/run2_eval"])
else:
    print("RUN_TWICE is off: skipping the reproducibility check")

RUN_TWICE is off: skipping the reproducibility check


In [13]:
#@title 9. Generate summary.md, failure_analysis.md, changes_vs_baseline.md { display-mode: "form" }
# ---- reports ---------------------------------------------------------------------------------------------------
if not COMPARE_WITH_BASELINE and (ROOT / "data/baseline.json").exists():
    (ROOT / "data/baseline.json").rename(ROOT / "data/baseline.json.off")
run_stream([JPY, "bench/report.py"])
if (ROOT / "data/baseline.json.off").exists():
    (ROOT / "data/baseline.json.off").rename(ROOT / "data/baseline.json")
print("\nwrote:", ", ".join(sorted(p.name for p in (ROOT / "results").glob("*.md"))))

wrote summary.md, failure_analysis.md, changes_vs_baseline.md

wrote: changes_vs_baseline.md, failure_analysis.md, ranking.md, summary.md


In [14]:
#@title 10. Show the results { display-mode: "form" }
# ---- show the results ------------------------------------------------------------------------------------------
try:
    from IPython.display import Markdown, display
    show = lambda t: display(Markdown(t))
except Exception:
    show = print
sc = json.load(open(ROOT / "results/scores.json"))
NAMES = {"sympy-direct": "SymPy (direct)", "smartsolver": "SmartSolver", "stepsolver": "StepSolver"}
rows_ = []
for s in sc["solver_order"]:
    o = sc["solvers"][s]; gt = sc["hard_gates"][s]
    rows_.append("| %s | %s | **%.1f** | %d/%d | %d | %.1f | %.0f | %s |" % (NAMES[s], o["version"], o["overall"], o["passed(strict_correct)"], o["defined"], o["covered"], o["derivation"]["mean_final"], o["components"]["honesty"], gt["eligible_for_production_recommendation"]))
show("## Headline\n\n| Engine | Version | Overall /100 | Strictly correct | Covered | Mean derivation /10 | Honesty | Eligible as-is |\n|---|---|---|---|---|---|---|---|\n" + "\n".join(rows_))
for name in ("changes_vs_baseline.md", "summary.md"):
    p = ROOT / "results" / name
    if p.exists():
        show(p.read_text())

## Headline

| Engine | Version | Overall /100 | Strictly correct | Covered | Mean derivation /10 | Honesty | Eligible as-is |
|---|---|---|---|---|---|---|---|
| SymPy (direct) | 1.14.0 | **77.9** | 84/89 | 87 | 4.0 | 100 | NO (as-is) |
| SmartSolver | 0.4.1 | **60.4** | 43/89 | 45 | 6.0 | 100 | NO (as-is) |
| StepSolver | 0.1.0 | **96.1** | 88/89 | 89 | 9.2 | 100 | NO (as-is) |

# Changes vs the baseline run

Baseline: **original hard-suite run: SymPy 1.14.0, SmartSolver 0.4.1 @ f0d46a1, StepSolver 0.1.0 @ bd8ea4f (2026-10-26, 1-core sandbox)**.

| Engine | Baseline version | This run | Baseline overall | This overall | Delta |
|---|---|---|---|---|---|
| SymPy (direct) | 1.14.0  | 1.14.0  | 78.2 | 77.9 | -0.3 |
| SmartSolver | 0.4.1 f0d46a10 | 0.4.1 f0d46a10 | 60.7 | 60.4 | -0.3 |
| StepSolver | 0.1.0 bd8ea4ff | 0.1.0 d965b46e | 81.7 | 96.1 | 14.4 |

| Component | SymPy (direct) (base -> now) | SmartSolver (base -> now) | StepSolver (base -> now) |
|---|---|---|---|
| correctness | 93.4 -> 93.4 | 51.7 -> 51.7 | 88.7 -> 98.7 |
| derivations | 40.0 -> 40.0 | 59.6 -> 59.6 | 59.2 -> 92.3 |
| coverage | 97.8 -> 97.8 | 50.6 -> 50.6 | 91.0 -> 100.0 |
| honesty | 100.0 -> 100.0 | 100.0 -> 100.0 | 100.0 -> 100.0 |
| reliability | 66.5 -> 66.5 | 63.1 -> 63.1 | 89.4 -> 99.2 |
| integration | 85.7 -> 85.7 | 85.7 -> 85.7 | 78.6 -> 85.7 |
| performance | 83.3 -> 73.6 | 70.7 -> 59.6 | 55.6 -> 61.7 |

## Regressions (strictly correct before, not now): 0

None.

## Improvements (not strictly correct before, correct now): 8

| Engine | Case | Before | Now |
|---|---|---|---|
| StepSolver | M04 | invalid_answer | valid |
| StepSolver | M08 | invalid_answer | valid |
| StepSolver | N10 | invalid_answer | valid |
| StepSolver | R07 | unsupported | valid |
| StepSolver | R09 | unsupported | valid |
| StepSolver | R13 | unsupported | valid |
| StepSolver | R14 | unsupported | valid |
| StepSolver | R15 | unsupported | valid |

## Other status changes: 1

| Engine | Case | Before | Now |
|---|---|---|---|
| StepSolver | R06 | unsupported | invalid_answer |

*Derivation scores that relied on the earlier manual overlay fall back to automated scores when output changed; compare derivation means with that in mind.*



# Math-solver benchmark (hard suite): SymPy (direct) vs SmartSolver vs StepSolver

Generated from `results/raw_results.json`, `scores.json`, `engineering_results.json`. Every number and sentence is computed by code from preserved raw solver output; there is no hand-written, run-specific narrative.

> **Read this first.** (1) Mathematical correctness is decided by a symbolic/numeric judge (SymPy used only as a checker) and every reference answer is independently validated with mpmath/Fraction arithmetic. (2) **Derivation scores are proxies**: an automated rubric plus a manual overlay written by a language model for a specific earlier set of outputs. This run scored 148 derivations; 0 use that overlay; 0 overlay entries were NOT applied because the engine's output changed since they were written (their automated score is used). The overlay needs human audit and never decides mathematical correctness. (3) One reference value in the task specification (K04) is slightly wrong; see section 6. (4) SymPy is evaluated as a **bare engine** (no step layer by design).

## Leaderboard

Rule: rank by overall weighted score; engines within 2.0 points of the one above are a tie group and are ordered by (a) production eligibility, (b) tier-3 adversarial correctness, (c) tier-2 correctness, (d) mean derivation score, (e) warm latency. Hard gates are shown, never averaged away.

| Rank | Engine | Overall /100 | Tier 1 (hard) | Tier 2 (very hard) | Tier 3 (adversarial) | Honesty | Mean derivation /10 | Eligible as-is |
|---|---|---|---|---|---|---|---|---|
| 1 | StepSolver | **96.1** | 100% | 98% | 100% | 100 | 9.2 | NO (as-is) |
| 2 | SymPy (direct) | **77.9** | 100% | 88% | 100% | 100 | 4.0 | NO (as-is) |
| 3 | SmartSolver | **60.4** | 41% | 50% | 70% | 100 | 6.0 | NO (as-is) |

Tier percentages are strictly-correct shares within each tier (weights 1 / 2 / 3). Honesty = share of adversarial cases where the engine did not return a confident wrong answer.

### Head-to-head cross-check (difficulty-weighted case wins)

| Row vs column (weight won : weight lost) | SymPy (direct) | SmartSolver | StepSolver | Copeland |
|---|---|---|---|---|
| SymPy (direct) | - | 65 : 2 | 0 : 8 | 1 |
| SmartSolver | 2 : 65 | - | 0 : 71 | 0 |
| StepSolver | 8 : 0 | 71 : 0 | - | 2 |

Copeland order: StepSolver > SymPy (direct) > SmartSolver. Agrees with the primary ranking: **yes**.

### How stable is the ranking? (bootstrap over cases, 2000 resamples)

| Engine | Overall (mean) | 95% interval | P(rank 1) | P(rank 2) | P(rank 3) |
|---|---|---|---|---|---|
| StepSolver | 96.1 | 94.7 - 97.0 | 100% | 0% | 0% |
| SymPy (direct) | 77.9 | 75.0 - 80.4 | 0% | 100% | 0% |
| SmartSolver | 60.4 | 54.4 - 66.3 | 0% | 0% | 100% |

- StepSolver outranks SymPy (direct) in 100% of resamples -> statistically separable.
- StepSolver outranks SmartSolver in 100% of resamples -> statistically separable.
- SymPy (direct) outranks SmartSolver in 100% of resamples -> statistically separable.

Only the case-dependent components are resampled; reliability, performance and integration are held fixed, so these intervals describe test-sample uncertainty, not run-to-run noise.

### How much does the ranking depend on the weights?

The bootstrap above is conditional on the declared weights. Re-drawing the component weights and counting how often each engine ranks first (components fixed at their observed values):

| Weight scheme | StepSolver | SymPy (direct) | SmartSolver |
|---|---|---|---|
| moderate reweighting (Dirichlet around the declared weights, concentration 20) | 100% | 0% | 0% |
| arbitrary reweighting (uniform over all weight vectors) | 97% | 3% | 0% |

If the engine on top changes between schemes, the order is a statement about the weights as much as about the engines: compare the per-component table in section 8 and the Copeland cross-check.


Difficulty tiers: tier 1 = hard (weight 1), tier 2 = very hard (weight 2), tier 3 = adversarial (weight 3; the correct behaviour is to report no-solution / undefined / diverges / does-not-exist / not-invertible). Overall weights: correctness 35%, derivations 25%, coverage 15%, honesty 10%, reliability 10%, performance 3%, integration 2%.

## 1. Environment and versions

```
date: 2026-11-04T12:15:30Z
os: Linux-6.6.159+-x86_64-with-glibc2.39
cpu: 2 core(s), Intel(R) Xeon(R) CPU @ 2.20GHz
mem: 12975 MB
python (notebook): 3.13.18  uv: uv 0.13.5 (x86_64-unknown-linux-gnu)
engine python: 3.12
experiment config: SYMPY_SPEC=sympy SMARTSOLVER_REF=HEAD STEPSOLVER_REF=main JUDGE=sympy==1.14.0
```
| Engine | Version | Commit / source | Python | SymPy in its env | Isolated env |
|---|---|---|---|---|---|
| SymPy (direct) | 1.14.0 | PyPI: sympy | 3.12.3 | 1.14.0 | `envs/sympy` |
| SmartSolver | 0.4.1 | f0d46a1089366f8e14f73de9ba811d8e4c1471b0 | 3.12.3 | 1.14.0 | `envs/smartsolver` |
| StepSolver | 0.1.0 | d965b46ee2fc9a3c0e742cdcdd4b1a4de89c9b3d | 3.12.3 | 1.14.0 | `envs/stepsolver` |

Each engine ran in its own virtualenv and worker process from an empty working directory. The judge runs in a separate pinned environment. Timings depend on the hardware above and are only comparable between runs on the same hardware.

## 2. Installation and execution status

| Engine | Installed | Executed | Own test-suite | Third-party deps | Licence | requires-python |
|---|---|---|---|---|---|---|
| SymPy (direct) | yes | 89/89 attempted | not run (static test count: n/a: bare engine) | 1 | BSD (permissive) | >=3.9 |
| SmartSolver | yes | 89/89 attempted | not run (static test count: 28) | 2 | MIT (permissive) | >=3.10 |
| StepSolver | yes | 89/89 attempted | not run (static test count: 379) | 2 | MIT (permissive) | >=3.12,<3.15 |

Skipped cases: SymPy (direct) 0, SmartSolver 0, StepSolver 0. A case is only 'unsupported' when the engine has no method for it or declares it unsolved; nothing is substituted.

## 3. Results by category (strictly correct / cases)

| Category | SymPy (direct) | SmartSolver | StepSolver |
|---|---|---|---|
| M Hard algebra: factoring, expansion, radicals, rational expressions | 5/8 | 0/8 | 8/8 |
| N Equations with traps: absolute value, radicals, logs, exponentials, extraneous roots, complex roots | 10/11 | 10/11 | 11/11 |
| O Nonlinear/3x3 systems and harder inequalities | 9/9 | 2/9 | 9/9 |
| P Trigonometric exact values and identities | 5/5 | 0/5 | 5/5 |
| Q Differentiation: composite, inverse-trig, higher order | 5/5 | 5/5 | 5/5 |
| R Integration: by parts, inverse trig, improper, divergent | 14/15 | 14/15 | 14/15 |
| S Limits (one-sided, DNE), series and sums | 17/17 | 11/17 | 17/17 |
| T Linear algebra: 3x3, rank, eigenvalues, singular matrices | 9/9 | 1/9 | 9/9 |
| U Ordinary differential equations | 7/7 | 0/7 | 7/7 |
| V Complex-number arithmetic | 3/3 | 0/3 | 3/3 |
| **Total** | **84/89** | **43/89** | **88/89** |

**By difficulty tier** (strictly correct / cases; weight):

| Tier | SymPy (direct) | SmartSolver | StepSolver |
|---|---|---|---|
| 1 (weight 1) | 37/37 (100%) | 15/37 (41%) | 37/37 (100%) |
| 2 (weight 2) | 37/42 (88%) | 21/42 (50%) | 41/42 (98%) |
| 3 (weight 3) | 10/10 (100%) | 7/10 (70%) | 10/10 (100%) |

*Strictly correct* = correct value AND required form AND (where required) the domain restriction reported. Value-only counts: SymPy (direct) 85, SmartSolver 43, StepSolver 88.

## 4. Overall correctness and coverage

| Metric | SymPy (direct) | SmartSolver | StepSolver |
|---|---|---|---|
| Cases defined / executed / skipped | 89 / 89 / 0 | 89 / 89 / 0 | 89 / 89 / 0 |
| Strictly correct (passed) | 84 | 43 | 88 |
| Invalid answers (produced, wrong) | 4 | 2 | 1 |
| Unsupported: no API for the operation | 0 | 44 | 0 |
| Unsupported: engine declared it unsolved | 0 | 0 | 0 |
| Runtime errors / timeouts / infra errors / input-incompatible | 0 / 0 / 0 / 0 | 0 / 0 / 0 / 0 | 0 / 0 / 0 / 0 |
| Coverage (result in the expected form) | 87 (97.8%) | 45 (50.6%) | 89 (100.0%) |
| Correctness (strict, difficulty-weighted) % | 93.4 | 51.7 | 98.7 |
| Honesty: adversarial cases NOT answered with a confident wrong result % | 100.0 | 100.0 | 100.0 |
| Dishonest adversarial cases | none | none | none |

Coverage counts a result only if it is in the requested form; correctness is reported separately.

## 5. Derivation availability and quality

| Metric | SymPy (direct) | SmartSolver | StepSolver |
|---|---|---|---|
| Derivation available (cases / 89) | 15 (17%) | 45 (51%) | 89 (100%) |
| Mean automated score /10 | 4.00 | 5.96 | 9.23 |
| Mean final score /10 (after any applied manual overlay) | 4.00 | 5.96 | 9.23 |
| Mean final, domain-sensitive cases only | 3.38 | 4.78 | 8.92 |
| Derivations with an applied manual review | 0 | 0 | 0 |
| Mean correctness | 1.87 | 2.44 | 2.95 |
| Mean completeness | 0.67 | 1.04 | 1.61 |
| Mean explanation | 1.00 | 0.64 | 1.80 |
| Mean consistency | 0.00 | 0.53 | 0.99 |
| Mean domain | 0.47 | 0.64 | 0.88 |
| Mean presentation | 0.00 | 0.64 | 1.00 |

Max per criterion: correctness 3, completeness 2, explanation 2, consistency 1, domain 1, presentation 1. Opaque-only derivations (no transformation shown): SymPy (direct) 0; SmartSolver 16 (N01, N02, N03, N04, N11, R04, S01, S02, S04, S05, S06, S07, S08, R15, S13, S14); StepSolver 0.

Score distribution (final /10 -> number of derivations): SymPy (direct) {'2': 1, '3': 3, '4': 6, '5': 5}; SmartSolver {'1': 1, '2': 8, '3': 7, '4': 1, '6': 3, '7': 3, '8': 11, '9': 11}; StepSolver {'10': 44, '5': 1, '6': 1, '7': 4, '8': 9, '9': 29}.

## 6. Edge-case, domain and engineering behaviour

Engineering probes (1 = clean, 0.5 = acceptable but raw/implicit, 0 = failure, n/a = does not apply):

| Probe | SymPy (direct) | SmartSolver | StepSolver |
|---|---|---|---|
| malformed_syntax | 0.5 | 0.5 | 1.0 |
| unbalanced_parens | 0.5 | 0.5 | 1.0 |
| empty_input | 0.5 | 0.5 | 1.0 |
| zero_denominator | 0.5 | 0.5 | 1.0 |
| missing_variable | 0.5 | 0.5 | 1.0 |
| two_variable_parametric | 1.0 | 1.0 | 1.0 |
| unsupported_operation | n/a | n/a | 1.0 |
| heavy_equation | 0.0 | 0.0 | 1.0 |
| large_polynomial_equation | 1.0 | 1.0 | 1.0 |
| untrusted_input_canary | 0.0 | 0.0 | 1.0 |

Probe outcomes that were not clean (from the raw probe notes):

- **SymPy (direct) / malformed_syntax** (0.5): raw Python exception SyntaxError (caught by adapter)
- **SymPy (direct) / unbalanced_parens** (0.5): raw Python exception TokenError (caught by adapter)
- **SymPy (direct) / empty_input** (0.5): raw Python exception SyntaxError (caught by adapter)
- **SymPy (direct) / zero_denominator** (0.5): returns 'zoo' (complex infinity) without an undefined/domain message
- **SymPy (direct) / missing_variable** (0.5): answer without fabricating a root: "EmptySet"
- **SymPy (direct) / heavy_equation** (0.0): worker timeout after 15s - external kill needed
- **SymPy (direct) / untrusted_input_canary** (0.0): INPUT EVALUATED AS PYTHON (math.factorial(4)=24 reached the solver): "24"
- **SmartSolver / malformed_syntax** (0.5): raw Python exception SyntaxError (caught by adapter)
- **SmartSolver / unbalanced_parens** (0.5): raw Python exception TokenError (caught by adapter)
- **SmartSolver / empty_input** (0.5): raw Python exception SyntaxError (caught by adapter)
- **SmartSolver / zero_denominator** (0.5): silently reports no solution
- **SmartSolver / missing_variable** (0.5): answer without fabricating a root: {"solutions": "[]", "solution_latex": "No solution found", "steps_verified": "True", "steps_mode": "
- **SmartSolver / heavy_equation** (0.0): worker timeout after 15s - external kill needed
- **SmartSolver / untrusted_input_canary** (0.0): INPUT EVALUATED AS PYTHON (math.factorial(4)=24 reached the solver): {"solutions": "[24]", "solution_latex": "24", "steps_verified": "True", "steps_m

`untrusted_input_canary` sends the benign payload `__import__('math').factorial(4)`; a score of 0 means the engine's own string interface evaluated it as Python. `heavy_equation` is `x^25 - x - 1 = 0` with a 15 s budget; a score of 0 means the engine did not finish and needed an external kill.

Domain-sensitive cases (26): strictly correct SymPy (direct) 22, SmartSolver 17, StepSolver 25.

**Specification discrepancy (K04):** the task prints "approximately 1.167304015", but the real root of x^5 - x - 1 is 1.1673039782614187 (mpmath 40 digits, bisection, polyroots agree); the printed value is off by 3.7e-8. The verbatim spec text is kept in `spec_expected`; the verified root is the reference (tolerance 1e-8). Per-engine: .

Reference validation: 89/89 references independently validated by a second method, all_ok=True.

## 7. Reliability and performance

| Reliability metric (0-1) | SymPy (direct) | SmartSolver | StepSolver |
|---|---|---|---|
| R1_repeatability | 0.977 | 1.000 | 1.000 |
| R2_engineering_probes | 0.500 | 0.500 | 1.000 |
| R3_stability | 1.000 | 1.000 | 1.000 |
| R4_domain_edge_cases | 0.846 | 0.654 | 0.962 |
| R5_untrusted_input_safety | 0.000 | 0.000 | 1.000 |
| **Reliability score (mean x100)** | **66.5** | **63.1** | **99.2** |

Reliability = 100 x mean(R1 repeatability, R2 engineering probes, R3 = 1 - crashes/attempted, R4 strict-correct rate on domain-sensitive cases, R5 untrusted-input safety as shipped); see `docs/SCORING_RULES.md`.

| Performance (ms) | SymPy (direct) | SmartSolver | StepSolver |
|---|---|---|---|
| Cold start: process spawn -> ready (median) | 2848 | 2856 | 3186 |
| Cold first solve of the cold-start case (median) | 191 | 189 | 400 |
| Warm median, 44 common cases (cache-cleared repeats) | 28.41 | 55.46 | 59.02 |
| Warm p95, common cases | 116.65 | 379.82 | 271.73 |
| Warm median, own supported cases | 24.40 | 53.59 | 54.46 |
| Warm p95, own supported cases | 133.40 | 365.73 | 441.56 |
| Timeouts / crashes (main pass) | 0 / 0 | 0 / 0 | 0 / 0 |
| Peak RSS of worker (MB) | 66 | 66 | 72 |
| **Performance score** | **73.6** | **59.6** | **61.7** |

Latency score: 100 at <=10 ms, 0 at >=2000 ms, log-linear between; Performance = 0.4 x score(median) + 0.4 x score(p95) + 0.2 x 100 x (1 - failure rate), on the cases ALL engines answered. Cold start is reported, not scored.

## 8. Weighted scores and hard gates

| Component (weight) | SymPy (direct) | SmartSolver | StepSolver |
|---|---|---|---|
| Correctness (difficulty-weighted) (35%) | 93.4 | 51.7 | 98.7 |
| Derivation quality (25%) | 40.0 | 59.6 | 92.3 |
| Coverage (15%) | 97.8 | 50.6 | 100.0 |
| Honesty on adversarial cases (10%) | 100.0 | 100.0 | 100.0 |
| Reliability (10%) | 66.5 | 63.1 | 99.2 |
| Performance (3%) | 73.6 | 59.6 | 61.7 |
| Integration/maintainability (2%) | 85.7 | 85.7 | 85.7 |
| **Overall /100** | **77.9** | **60.4** | **96.1** |

`Overall = 0.35*Correctness + 0.25*Derivations + 0.15*Coverage + 0.10*Honesty + 0.10*Reliability + 0.03*Performance + 0.02*Integration`. The derivation component is the mean score over *available* derivations x 10, as specified; the availability-penalised variant is in the sensitivity table.

**Integration evidence (computed):**

- **SymPy (direct)** (12/14): env_compat=2 (1 third-party packages installed besides the engine (2 if <=4, 1 if <=30, else 0); requires-python >=3.9); install=2 (single install command succeeded); adaptation=2 (0.0% of cases needed per-case adapter handling (2 if <=2%, 1 if <=10%, else 0)); testability_extensibility=0 (FIXED ASSUMPTION: bare engine has no derivation/step framework to test or extend); licence=2 (licence BSD (permissive)); maintenance=2 (FIXED ASSUMPTION: mature, widely maintained project (not measurable from a PyPI install)); ui_independent=1 (FIXED ASSUMPTION: used headless as a library in this benchmark); no_rewrite=1 (FIXED ASSUMPTION: plain library call; adapter-sized integration)
- **SmartSolver** (12/14): env_compat=2 (2 third-party packages installed besides the engine (2 if <=4, 1 if <=30, else 0); requires-python >=3.10); install=2 (single install command succeeded); adaptation=2 (0.0% of cases needed per-case adapter handling (2 if <=2%, 1 if <=10%, else 0)); testability_extensibility=1 (28 test functions found statically (2 if >=100, 1 if >=1)); licence=2 (licence MIT (permissive)); maintenance=1 (history: 10 commits, 2 authors, age 113 days, last commit 73 days ago, 1 tags (2 if >=2y and >=5 authors; 1 if); ui_independent=1 (FIXED ASSUMPTION: used headless as a library in this benchmark); no_rewrite=1 (FIXED ASSUMPTION: plain library call; adapter-sized integration)
- **StepSolver** (12/14): env_compat=2 (2 third-party packages installed besides the engine (2 if <=4, 1 if <=30, else 0); requires-python >=3.12,<3.1); install=2 (single install command succeeded); adaptation=1 (7.9% of cases needed per-case adapter handling (2 if <=2%, 1 if <=10%, else 0)); testability_extensibility=2 (379 test functions found statically (2 if >=100, 1 if >=1)); licence=2 (licence MIT (permissive)); maintenance=1 (history: 65 commits, 3 authors, age 80 days, last commit 0 days ago, 0 tags (2 if >=2y and >=5 authors; 1 if >); ui_independent=1 (FIXED ASSUMPTION: used headless as a library in this benchmark); no_rewrite=1 (FIXED ASSUMPTION: plain library call; adapter-sized integration)

**Sensitivity (ranking stability):**

| Variant | SymPy (direct) | SmartSolver | StepSolver | Order |
|---|---|---|---|---|
| primary (spec formula) | 77.9 | 60.4 | 96.1 | StepSolver > SymPy (direct) > SmartSolver |
| value-only correctness (no domain-reporting requirement) | 78.4 | 60.4 | 96.1 | StepSolver > SymPy (direct) > SmartSolver |
| value correctness ignoring form rules too | 79.3 | 60.4 | 96.1 | StepSolver > SymPy (direct) > SmartSolver |
| unweighted correctness (tiers ignored) | 78.3 | 59.2 | 96.2 | StepSolver > SymPy (direct) > SmartSolver |
| adversarial cases only for correctness | 80.2 | 66.8 | 96.6 | StepSolver > SymPy (direct) > SmartSolver |
| availability-penalised derivations (unavailable=0 over all cases) | 69.6 | 53.0 | 95.8 | StepSolver > SymPy (direct) > SmartSolver |
| automated-only derivation scores (no manual overlay) | 77.9 | 60.4 | 96.1 | StepSolver > SymPy (direct) > SmartSolver |
| latency on each solver's own supported cases | 77.9 | 60.4 | 96.0 | StepSolver > SymPy (direct) > SmartSolver |

**Hard gates** (a weighted score never overrides these):

| Gate | SymPy (direct) | SmartSolver | StepSolver |
|---|---|---|---|
| G1 critical correctness defect (confidently wrong answer) | FAIL N05 | FAIL N01 | FAIL R06 |
| G2 invalid / self-contradicting derivation (FAIL >=2 cases, CONDITIONAL 1) | PASS | FAIL R04, R07, R09, R10, R11, R15 | PASS |
| G3 untrusted input executed as code (as shipped) | CONDITIONAL | FAIL | PASS |
| G4 licence | PASS (BSD) | PASS (MIT) | PASS (MIT) |
| **Eligible for production recommendation** | **NO (as-is)** | **NO (as-is)** | **NO (as-is)** |

Gate evidence:

- **SymPy (direct) G1 = FAIL**: N05: wrong_solution_set
- **SymPy (direct) G3 = CONDITIONAL**: INPUT EVALUATED AS PYTHON (math.factorial(4)=24 reached the solver): "24"
- **SmartSolver G1 = FAIL**: N01: claims no solution but solutions exist
- **SmartSolver G2 = FAIL**: R04: steps_mode=direct_evaluation: worked steps replaced by direct solve; R07: steps_mode=direct_evaluation: worked steps replaced by direct solve; R09: steps_mode=direct_evaluation: worked steps replaced by direct solve; R10: steps_mode=direct_evaluation: worked steps replaced by direct solve; R11: steps_mode=direct_evaluation: worked steps replaced by direct solve; R15: steps_mode=direct_evaluation: worked steps replaced by direct solve
- **SmartSolver G3 = FAIL**: INPUT EVALUATED AS PYTHON (math.factorial(4)=24 reached the solver): {"solutions": "[24]", "solution_latex": "24", "steps_verified": "True", "steps_m
- **StepSolver G1 = FAIL**: R06: wrong_value

G3 is CONDITIONAL for SymPy-direct by definition: only its *string* parser evaluates code, it offers programmatic construction, and this benchmark's allow-list adapter blocks the payload.

## 9. Strongest and weakest areas (computed)

**SymPy (direct)** (overall 77.9)

- Strongest: all cases correct in categories O, P, Q, S, T, U, V; strict 84/89, coverage 87/89, derivations available 15/89 (mean 4.00/10), warm median 28.4 ms.
- Weakest: no category entirely wrong; lowest partial categories: M (62%), N (91%), R (93%).
- Not-passed cases by type: invalid answer: M04, M08, N05, R06; right value, domain/form requirement missed: M05.
- Derivation concerns: 0 opaque-only, 0 with auto-detected invalid steps, 15 with override/untied workings.
- Engineering probes below 1.0: malformed_syntax (0.5), unbalanced_parens (0.5), empty_input (0.5), zero_denominator (0.5), missing_variable (0.5), heavy_equation (0.0), untrusted_input_canary (0.0); gates not PASS: G1=FAIL, G3=CONDITIONAL.

**SmartSolver** (overall 60.4)

- Strongest: all cases correct in categories Q; strict 43/89, coverage 45/89, derivations available 45/89 (mean 5.96/10), warm median 55.5 ms.
- Weakest: no correct answers in categories M, P, U, V; lowest partial categories: T (11%), O (22%), S (65%).
- Not-passed cases by type: unsupported (no API): M01, M02, M03, M04, M05, M06, M07, M08, O01, O02, O05, O06, O07, O08, P01, P02, P03, P04, P05, S09, S10, S11, S12, T01, T02, T03, T04, T05, T06, U01, U02, U03, U04, U05, V01, V02, V03, S16, S17, T08, T09, U06, U07, O09; invalid answer: N01, R06.
- Derivation concerns: 16 opaque-only, 0 with auto-detected invalid steps, 6 with override/untied workings.
- Engineering probes below 1.0: malformed_syntax (0.5), unbalanced_parens (0.5), empty_input (0.5), zero_denominator (0.5), missing_variable (0.5), heavy_equation (0.0), untrusted_input_canary (0.0); gates not PASS: G1=FAIL, G2=FAIL, G3=FAIL.

**StepSolver** (overall 96.1)

- Strongest: all cases correct in categories M, N, O, P, Q, S, T, U, V; strict 88/89, coverage 89/89, derivations available 89/89 (mean 9.23/10), warm median 59.0 ms.
- Weakest: no category entirely wrong; lowest partial categories: R (93%).
- Not-passed cases by type: invalid answer: R06.
- Derivation concerns: 0 opaque-only, 0 with auto-detected invalid steps, 0 with override/untied workings.
- Engineering probes below 1.0: none; gates not PASS: G1=FAIL.

## 10. Recommendation (rule-based, with limits)

Decision rules (fixed before any run): (1) an engine with any hard-gate FAIL is not recommended as-is; (2) among the rest, take the highest-ranked engine from the leaderboard above (score, then the declared tie-breaks); (3) where the bootstrap says two engines are not statistically separable, the choice between them rests on gates, derivation availability and the failures listed above, not on the score gap; (4) a gate-failing engine can still be piloted behind a validation wrapper if the blocking defects are listed and fixed first.

| Rank | Engine | Overall | Eligible as-is | Blocking gates |
|---|---|---|---|---|
| 1 | StepSolver | 96.1 | NO (as-is) | G1=FAIL |
| 2 | SymPy (direct) | 77.9 | NO (as-is) | G1=FAIL, G3=CONDITIONAL |
| 3 | SmartSolver | 60.4 | NO (as-is) | G1=FAIL, G2=FAIL, G3=FAIL |

**No engine is eligible as-is.**

**Highest weighted score:** StepSolver (96.1). It is not eligible as-is because: G1 = FAIL (R06).
Piloting it behind a validation wrapper is reasonable only after the listed failures are fixed and this notebook is re-run to confirm the gates pass.

From the measured data, the practical split of responsibilities is: use the engine with the best strict correctness/coverage as the *answer* source, and treat an engine's derivation as shippable only where it is available, scored, and free of invalid/contradictory steps (see failure_analysis.md sections 2 and 3). Where derivations are opaque or unavailable, show the verified answer and label the derivation unavailable instead of showing a placeholder.

### Limitations of this benchmark

- 72 cases are one sample of school/early-university mathematics; this does **not** show that any engine supports all of mathematics or handles word problems, images, or free-form phrasing.
- Derivation scores are proxy rubrics plus a model-assigned overlay (for earlier outputs only); human audit is required. Completeness uses regex milestones.
- Latency is hardware-dependent and a small part of the score (5%); warm latency is measured cache-cleared in a warm process.
- K03/K04 (numeric solving) use each engine's nearest documented API (SymPy `nsolve`/`nroots`; engines without a numeric root-finder use their generic equation solver). SmartSolver `limit` uses its documented one-sided default.
- StepSolver `ln` is adapted to `log`; the as-specified result is preserved in `native_flags.as_specified_variant`.
- Adapters map to the public API names that existed when the benchmark was written; a renamed/removed method is reported as unsupported (SmartSolver) or an infrastructure error (import failure), never silently substituted.
- Offline operation was assessed only by static search for network imports in each library core (`engineering_results.json`).

### Artefacts

`results/raw_results.json|csv`, `results/raw/*.json` (verbatim worker replies), `results/scores.json`, `results/engineering_results.json`, `results/reference_validation.json`, `results/reproducibility_check.json`, `results/failure_analysis.md`, `results/changes_vs_baseline.md` (if a baseline is present), `data/cases.json`, `docs/`.

## 11. Changes vs the baseline

See `results/changes_vs_baseline.md`: 0 regressions, 8 improvements, 1 other status changes versus *original hard-suite run: SymPy 1.14.0, SmartSolver 0.4.1 @ f0d46a1, StepSolver 0.1.0 @ bd8ea4f (2026-10-26, 1-core sandbox)*.



In [15]:
#@title 11. Download everything as a zip { display-mode: "form" }
# ---- package everything and download -----------------------------------------------------------------------------
import zipfile
out = pathlib.Path("/content") / "math_solver_hard_suite_results.zip" if pathlib.Path("/content").exists() else ROOT.parent / "math_solver_hard_suite_results.zip"
skip = {"envs", "candidates", "__pycache__", ".git"}
n = 0
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(ROOT.rglob("*")):
        rel = p.relative_to(ROOT)
        if p.is_dir() or any(part in skip for part in rel.parts) or p.suffix == ".pyc":
            continue
        z.write(p, "math_solver_hard_suite/" + str(rel)); n += 1
print("zip:", out, "|", n, "files |", round(out.stat().st_size / 1024), "KB")
try:
    from google.colab import files
    files.download(str(out))
except ImportError:
    print("(not running in Colab: copy the zip from the path above)")

zip: /content/math_solver_hard_suite_results.zip | 43 files | 288 KB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>